# 🧠 Unified Data Assistant — SQL + RAG + Knowledge Graph, one notebook + Global Search

One `ask("...")` over **one** PostgreSQL database, answered by whichever of three engines fits
the question.

| Engine | Answers from | Good at | Example |
|---|---|---|---|
| 🗄️ **SQL** | your structured tables (`bids_old`, `suppliers`, `rfps`…) | counts, totals, rankings, filtered lists | *"Which supplier has the highest total bid amount?"* |
| 📄 **RAG** | document text indexed into `rag_chunks` (pgvector) | a single fact a document states | *"What are Globex's payment terms?"* |
| 🕸️ **Graph** | a knowledge graph over `rag_nodes` / `rag_edges` | relationships, dependencies, consequences, chains across documents | *"What happens if Tidewater's SQF certificate lapses?"* |

```
question
   │
   ├─ router ─── "sql" ──► gate → ambiguity menu → generate SQL
   │                        → validate → EXPLAIN → run → 📊 table
   │
   ├──────────── "rag" ──► embed → hybrid search (vector + keyword, RRF)
   │                        → rerank → grounded answer with [citations]
   │
   └────────── "graph" ──► hybrid search → which ENTITIES did we land on?
                            → walk 1-2 hops → read the neighbours' chunks
                            → rerank → answer, with the subgraph as stated facts
```

**Override the router any time:** `ask("...", mode="sql" | "rag" | "graph")`.
`ask_sql(...)`, `ask_rag(...)` and `ask_graphrag(...)` are all still there and callable directly.

**Everything runs on the Postgres you already have** — no Neo4j, no Spanner, no new service.
The graph is two extra tables and a recursive CTE.

---

### How to run
Run every cell top-to-bottom, in order. Sections **0 → A → B → G → C** are the working
pipeline. **Section D is optional** and entirely commented out, so nothing there runs by
accident.

### What is where
| Section | Contents |
|---|---|
| **0** | Shared: install, config (incl. Cloud SQL connection modes), DB pool, Vertex AI client, embeddings |
| **A** | **SQL engine** — health check, schema, table notes, answerability gate, 5-layer validators, `ask_sql()` |
| **B** | **RAG engine** — pgvector schema, chunking, 16-document seed corpus, ingest, hybrid retrieval, rerank, `ask_rag()` |
| **G** | **Knowledge graph** — nodes/edges from your foreign keys *and* your document metadata, entity linking, traversal, `ask_graphrag()` |
| **C** | **The router** — `ask()` |
| **D** | Optional (commented): guardrail self-test, retrieval eval, LLM-judge, config sweep |
| **E** | Troubleshooting & production notes |

### The one thing you must edit
**Section A.3 (`TABLE_NOTES`)** — your business rules in plain English, e.g. *"suppliers_old is
historical and must be combined with suppliers via UNION ALL for any total."* It is injected
into every SQL prompt and is the single biggest accuracy win available to you.

---
# Section 0 — Shared foundation

Both engines talk to the **same** Postgres database and the **same** Vertex AI project, so
config, the connection pool and the Gemini client are declared once here and reused by both.

## 0.1 — Install

In [ ]:
# google-genai     -> Vertex AI (Gemini + embeddings), used by every engine
# psycopg2-binary  -> PostgreSQL driver
# tiktoken         -> token counting for RAG chunk budgeting
# pandas           -> tabular display of SQL results
%pip install -q google-genai "psycopg2-binary>=2.9" tiktoken pandas

# Only needed if you set CLOUDSQL_MODE = "connector" in Section 0.2 (see the notes there).
# Safe to leave commented out when connecting over private IP.
# %pip install -q "cloud-sql-python-connector[pg8000]" google-cloud-sql-connector

print("Installed. Restart the kernel if pip asked you to.")

## 0.2 — Configuration (the only cell you normally edit)

**Authenticate to Vertex AI first** (pick one):

- your machine: `gcloud auth application-default login`
- Colab: `from google.colab import auth; auth.authenticate_user()`
- GCE / Cloud Run / Vertex Workbench: the attached service account is used automatically
- elsewhere: `GOOGLE_APPLICATION_CREDENTIALS=/path/to/service-account.json`

and make sure the **Vertex AI API is enabled** on the project.

### Connecting to Cloud SQL — three ways, pick by where this notebook runs

| `CLOUDSQL_MODE` | Use when | Needs |
|---|---|---|
| `"direct"` *(default)* | the notebook runs **inside the VPC** — a GCE VM, Vertex AI Workbench, Cloud Run with a VPC connector. This is your current setup: `10.151.179.4` is a private IP. | nothing extra |
| `"connector"` | the notebook runs **outside the VPC** — your laptop, Colab, another project. Handles TLS and IAM for you, no proxy binary to run. | `pip install "cloud-sql-python-connector[pg8000]"`, and the instance's connection name |
| Auth Proxy | you'd rather run a sidecar and keep `"direct"` pointed at `127.0.0.1` | `cloud-sql-proxy PROJECT:REGION:INSTANCE --port 5432` in a terminal |

`10.151.179.4` will simply time out from a laptop — a private IP is not reachable from
outside the VPC. If that is where you are, switch to `"connector"`.

**IAM database authentication** (`CLOUDSQL_IAM_AUTH = True`) is worth knowing about: the
connector fetches a short-lived token instead of sending a password, so there is no
credential in this file at all. It requires `cloudsql.instances.connect` on the principal and
a matching Cloud SQL IAM database user. It's the real fix for the hard-coded password below.

> ⚠️ The password is hard-coded, as you asked. Anyone who receives a copy of this file
> receives the credential. If this notebook leaves your machine, rotate it.

In [ ]:
# =========================== Google Cloud / Vertex AI ===========================
PROJECT_ID   = "div-aais-rfpiq-usc1-uat"   # ⬅️ your GCP project id
LOCATION     = "us-central1"               # Vertex AI region

SQL_MODEL    = "gemini-2.5-pro"     # writes SQL (Section A) - accuracy matters most here
CHAT_MODEL   = "gemini-2.5-flash"   # writes the RAG answer (Section B)
RERANK_MODEL = "gemini-2.5-flash"   # scores search results (Section B)
ROUTER_MODEL = "gemini-2.5-flash"   # picks sql vs rag (Section C) - must be cheap + fast

EMBED_MODEL  = "gemini-embedding-001"  # ONE model, forever: vectors from different models
                                       # are NOT comparable. Changing it means re-ingesting.
EMBED_DIM    = 1536   # model emits 3072 natively; pgvector's hnsw index refuses >2000 dims.
                      # This model is Matryoshka-trained, so 1536 keeps full quality.

# =========================== Cloud SQL connection ===============================
CLOUDSQL_MODE     = "direct"      # "direct" (private IP / proxy) | "connector"
CLOUDSQL_INSTANCE = "div-aais-rfpiq-usc1-uat:us-central1:YOUR-INSTANCE-NAME"  # ⬅️ only for "connector"
CLOUDSQL_IP_TYPE  = "PRIVATE"     # "PRIVATE" | "PUBLIC" — which IP the connector dials
CLOUDSQL_IAM_AUTH = False         # True = short-lived IAM token instead of a password

# =========================== PostgreSQL (shared) ================================
DB_CONFIG = {
    "host":     "10.151.179.4",
    "port":     5432,
    "dbname":   "postgres",
    "user":     "postgres",
    "password": "Rfpiq-usc1",
    "connect_timeout": 10,
    "application_name": "unified_sql_rag",   # shows up in pg_stat_activity
}
DB_SCHEMA = "public"

# =========================== SQL engine (Section A) =============================
MAX_ROWS = 500        # hard cap on rows returned by any generated query

# =========================== RAG engine (Section B) =============================
DOC_TABLE   = "rag_documents"   # raw, un-chunked source text
CHUNK_TABLE = "rag_chunks"      # chunks + vectors (what search actually reads)

# These two are RAG plumbing, not business data - hide them from the SQL engine so it
# never tries to answer a question by querying the vector store.
RAG_INTERNAL_TABLES = {DOC_TABLE, CHUNK_TABLE}

CHUNK_TOKENS         = 350   # max tokens per chunk (~260 words). 250-500 is the sweet spot.
CHUNK_OVERLAP_TOKENS = 60    # ~15% repeated, so a fact on a boundary survives whole somewhere
MAX_EMBED_TOKENS     = 2000  # safety cap per embedding call

VECTOR_CANDIDATES   = 40     # stage 1a: how many chunks the meaning search returns
TEXT_CANDIDATES     = 40     # stage 1b: how many chunks the keyword search returns
RRF_K               = 60     # Reciprocal Rank Fusion constant, weight = 1/(k + rank)
W_VECTOR, W_TEXT    = 1.0, 1.0   # raise W_TEXT if users search by exact IDs/SKUs
TOP_K               = 8      # survivors of the merge when NOT reranking
RERANK_CANDIDATES   = 20     # survivors handed to the reranker
RERANK_TOP_N        = 4      # how many the LLM finally sees. Small on purpose.
USE_RERANKER        = True
MAX_COSINE_DISTANCE = 0.80   # relevance floor - beyond this (and no keyword hit) we refuse

HNSW_M               = 16    # graph connections per node
HNSW_EF_CONSTRUCTION = 64    # effort while BUILDING the index
HNSW_EF_SEARCH       = 100   # effort while SEARCHING - the one you actually tune (free to change)

print(f"Config loaded → {DB_CONFIG['host']}/{DB_CONFIG['dbname']} | "
      f"SQL={SQL_MODEL} | RAG={CHAT_MODEL} | embed={EMBED_MODEL}@{EMBED_DIM}d")

## 0.3 — Database access (one pool, two front doors)

- `db()` — pooled cursor, used by everything in the RAG engine.
- `get_conn()` — a plain connection, used by the SQL engine because it needs its own
  `readonly` session and explicit rollback per query. Same credentials, different lifetime.

In [ ]:
import contextlib, json, re, math
import psycopg2
import psycopg2.extras
import pandas as pd
from psycopg2 import sql as pgsql
from psycopg2.pool import ThreadedConnectionPool

# ============================ how a connection is made ============================
if CLOUDSQL_MODE == "connector":
    # Runs from anywhere: the connector resolves the instance, handles TLS, and (with
    # IAM auth) fetches a short-lived token so no password is sent at all.
    from google.cloud.sql.connector import Connector, IPTypes

    _connector = Connector(
        ip_type=IPTypes.PRIVATE if CLOUDSQL_IP_TYPE.upper() == "PRIVATE" else IPTypes.PUBLIC,
        enable_iam_auth=CLOUDSQL_IAM_AUTH,
    )

    def _new_conn():
        kw = {"user": DB_CONFIG["user"], "db": DB_CONFIG["dbname"]}
        if not CLOUDSQL_IAM_AUTH:
            kw["password"] = DB_CONFIG["password"]
        return _connector.connect(CLOUDSQL_INSTANCE, "psycopg2", **kw)

else:
    # Private IP inside the VPC, or 127.0.0.1 with the Cloud SQL Auth Proxy running.
    def _new_conn():
        return psycopg2.connect(**DB_CONFIG)


class _FactoryPool(ThreadedConnectionPool):
    """psycopg2's pool builds connections from kwargs; we need it to call our factory
    instead, so the same pool serves both connection modes."""
    def __init__(self, minconn, maxconn, factory):
        self._factory = factory
        super().__init__(minconn, maxconn)

    def _connect(self, key=None):
        conn = self._factory()
        if key is not None:
            self._used[key] = conn
            self._rused[id(conn)] = key
        else:
            self._pool.append(conn)
        return conn


# A pool opens a few connections up front and lends them out, instead of a fresh
# TCP + TLS + auth handshake on every query.
POOL = _FactoryPool(1, 8, _new_conn)

@contextlib.contextmanager
def db(dict_rows: bool = False):
    """Borrow a pooled connection and hand back a cursor.

    Leaving the `with conn:` block normally COMMITs; leaving via an exception ROLLBACKs.
    The `finally` guarantees the connection returns to the pool even on a crash — without
    it, 8 crashes would exhaust the pool and hang the notebook.
    """
    conn = POOL.getconn()
    try:
        factory = psycopg2.extras.RealDictCursor if dict_rows else None
        with conn:
            with conn.cursor(cursor_factory=factory) as cur:
                yield cur
    finally:
        POOL.putconn(conn)


def get_conn():
    """A standalone connection for the SQL engine's read-only, always-rolled-back queries."""
    return _new_conn()


def _parse_json_reply(text):
    """Models sometimes wrap JSON in ```json fences. Strip them, then parse defensively."""
    text = (text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        a, b = text.find("{"), text.rfind("}")
        if a != -1 and b != -1:
            return json.loads(text[a:b + 1])
        raise


try:
    with db() as cur:
        cur.execute("SELECT current_database(), version()")
        _dbname, _version = cur.fetchone()
    print(f"Connected to '{_dbname}' via {CLOUDSQL_MODE} — {_version.split(',')[0]}")
except Exception as _e:
    raise SystemExit(
        f"Could not connect: {_e}\n"
        "  • timeout on a 10.x address → you are outside the VPC. Set CLOUDSQL_MODE=\"connector\"\n"
        "    in Section 0.2 (and pip install the connector in 0.1), or run the Cloud SQL Auth Proxy.\n"
        "  • authentication failed → check user/password, or set CLOUDSQL_IAM_AUTH=True\n"
        "  • connector import error → %pip install \"cloud-sql-python-connector[pg8000]\"")

## 0.4 — Vertex AI client + embeddings

One `genai_client` serves both engines. The embedding helpers (batching, retry with
exponential backoff, L2 normalisation, query cache) are RAG-side but live here because the
client they need is shared.

In [ ]:
import random, time
from functools import lru_cache

import tiktoken
from google import genai
from google.genai.types import EmbedContentConfig

genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

# cl100k_base is only a consistent yardstick for budgeting chunk sizes - it does not need
# to match the embedding model's own tokenizer.
_enc = tiktoken.get_encoding("cl100k_base")

def n_tokens(text: str) -> int:
    """How many tokens is this text? Used to keep chunks inside CHUNK_TOKENS."""
    return len(_enc.encode(text))

def _truncate(text: str, max_tokens: int = MAX_EMBED_TOKENS) -> str:
    """Cut text down to max_tokens. One oversized input would otherwise fail the whole batch."""
    toks = _enc.encode(text)                                  # text -> list of token ids
    return text if len(toks) <= max_tokens else _enc.decode(toks[:max_tokens])   # and back again

def _l2_normalize(vec):
    """Scale a vector so its length is exactly 1.0, without changing its direction.

    Why: cosine distance only cares about direction (the angle), not length. Truncating a
    3072-dim embedding to 1536 leaves it slightly shorter than 1.0, so we re-normalise to
    keep <=> numerically well behaved and distances comparable between models.
    """
    norm = math.sqrt(sum(x * x for x in vec))    # Pythagoras in 1536 dimensions
    return [x / norm for x in vec] if norm else vec   # `if norm` guards against divide-by-zero

# Error signatures that mean "the server is busy, try again" rather than "you did it wrong".
_TRANSIENT = ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "500", "INTERNAL", "DEADLINE")

def _with_retry(fn, attempts: int = 5, base: float = 1.0):
    """Call fn(); if it fails with a TEMPORARY error, wait and try again.

    `fn` is a function, not a value - callers pass `lambda: something()` so we can
    re-invoke it. Waits 1s, 2s, 4s, 8s ("exponential backoff"), plus a random fraction
    of a second ("jitter") so that many parallel workers do not all retry in lockstep
    and hammer the server at the same instant.
    """
    for i in range(attempts):
        try:
            return fn()                       # success - return immediately
        except Exception as e:
            # Give up if we are out of attempts, OR if this error will never fix itself.
            # A 403 (no permission) or a typo in the model name is permanent: retrying it
            # 5 times just wastes 15 seconds and hides the real problem.
            if i == attempts - 1 or not any(s in str(e) for s in _TRANSIENT):
                raise
            time.sleep(base * (2 ** i) + random.random())   # 2**i -> 1, 2, 4, 8...

def embed_texts(texts, task_type: str, model: str = EMBED_MODEL,
                dim: int = EMBED_DIM, batch_size: int = 16):
    """Turn a list of strings into a list of vectors.

    task_type must be RETRIEVAL_DOCUMENT for stored chunks and RETRIEVAL_QUERY for
    questions. The model deliberately encodes the two roles differently ("what can be
    found" vs "what is doing the finding"), and mixing them up silently costs accuracy -
    no error, just worse results.
    """
    cfg = EmbedContentConfig(task_type=task_type, output_dimensionality=dim)
    out = []
    # Send 16 texts per API call instead of one call per text: far fewer network round trips.
    # range(0, len, step) walks the list in blocks: 0..15, 16..31, ...
    for i in range(0, len(texts), batch_size):
        batch = [_truncate(t) for t in texts[i:i + batch_size]]
        try:
            resp = _with_retry(lambda: genai_client.models.embed_content(
                model=model, contents=batch, config=cfg))
            out.extend(e.values for e in resp.embeddings)   # .values is the list of floats
        except Exception:
            # Fallback: some model/quota combinations reject batches and accept only one
            # input per call. Retry this block one text at a time rather than failing.
            for t in batch:
                # `lambda t=t:` captures the CURRENT t. Without the `t=t` every lambda would
                # share the loop variable and embed the last text repeatedly - a classic bug.
                resp = _with_retry(lambda t=t: genai_client.models.embed_content(
                    model=model, contents=[t], config=cfg))
                out.append(resp.embeddings[0].values)
    return [_l2_normalize(v) for v in out]

def embed_query(question: str):
    """Embed ONE question. [0] because embed_texts always returns a list."""
    return embed_texts([question], "RETRIEVAL_QUERY")[0]

@lru_cache(maxsize=1024)   # remembers the last 1024 questions: same question -> instant, no API call
def embed_query_cached(question: str):
    """Production path. Saves ~200ms on any repeated question.

    Returns a tuple, not a list, because @lru_cache requires everything it stores to be
    hashable (immutable) - and lists are not. Callers convert back with list(...).

    Section D.2's benchmark deliberately calls the UNCACHED embed_query() instead, so its
    latency numbers reflect a real cold request rather than a cache hit.
    """
    return tuple(embed_query(question))

_probe = embed_query("hello")
assert len(_probe) == EMBED_DIM, f"expected {EMBED_DIM} dims, got {len(_probe)}"
print(f"Vertex OK - {EMBED_MODEL} -> {len(_probe)} dims, "
      f"|v| = {math.sqrt(sum(x * x for x in _probe)):.4f}")

---
# Section A — SQL engine

Natural language → **validated, read-only PostgreSQL** → results.

```
question → answerability gate → split-table / ambiguity menu → Gemini writes SQL
        → keyword+table+column validation → EXPLAIN dry-run → read-only execution → output checks
```

Every layer exists because a layer above it can be fooled. The gate refuses questions your
database cannot answer; the validators refuse SQL that references things that do not exist;
`EXPLAIN` lets Postgres itself veto; execution is `readonly` and always rolled back.

## A.1 — Health check: which tables exist, and do they have rows?

Empty tables are flagged so a *correct but empty* answer never surprises you. The RAG
tables (`rag_documents`, `rag_chunks`) are excluded — they are the other engine's storage,
not business data.

In [ ]:
def health_check():
    with get_conn() as conn, conn.cursor() as cur:
        cur.execute(
            """SELECT table_name FROM information_schema.tables
               WHERE table_schema = %s AND table_type = 'BASE TABLE'
               ORDER BY table_name""",
            (DB_SCHEMA,),
        )
        tables = [r[0] for r in cur.fetchall() if r[0] not in RAG_INTERNAL_TABLES]
        if not tables:
            print(f"⚠️  No business tables found in schema '{DB_SCHEMA}'.")
            print("   → The SQL engine will be disabled; the router will send everything to RAG.")
            return []
        report = []
        for t in tables:
            cur.execute(pgsql.SQL("SELECT COUNT(*) FROM {}.{}").format(
                pgsql.Identifier(DB_SCHEMA), pgsql.Identifier(t)))
            n = cur.fetchone()[0]
            report.append({"table": t, "rows": n,
                           "status": "✅ has data" if n > 0 else "⚠️ EMPTY"})
    display(pd.DataFrame(report))
    empty = [r["table"] for r in report if r["rows"] == 0]
    if empty:
        print(f"⚠️  Empty tables (queries on them WILL return 0 rows): {', '.join(empty)}")
    else:
        print("✅ All tables contain data.")
    return tables

KNOWN_TABLES = health_check()
print(f"\nSQL engine sees {len(KNOWN_TABLES)} business table(s).")

## A.2 — Extract the schema

Builds **`SCHEMA_TEXT`** (human-readable, goes into every prompt) and **`TABLE_COLUMNS`**
(`{table: {columns}}`, what the validator uses to reject invented columns).

In [ ]:
def load_schema():
    lines = []
    table_columns = {}
    with get_conn() as conn, conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        for t in KNOWN_TABLES:
            cur.execute(
                """SELECT column_name, data_type, is_nullable
                   FROM information_schema.columns
                   WHERE table_schema=%s AND table_name=%s ORDER BY ordinal_position""",
                (DB_SCHEMA, t))
            cols = cur.fetchall()
            table_columns[t.lower()] = {c["column_name"].lower() for c in cols}

            cur.execute(
                """SELECT a.attname AS col FROM pg_index i
                   JOIN pg_attribute a ON a.attrelid=i.indrelid AND a.attnum=ANY(i.indkey)
                   WHERE i.indrelid=%s::regclass AND i.indisprimary""",
                (f'{DB_SCHEMA}."{t}"',))
            pks = {r["col"] for r in cur.fetchall()}

            cur.execute(
                """SELECT kcu.column_name AS col, ccu.table_name AS ref_table,
                          ccu.column_name AS ref_col
                   FROM information_schema.table_constraints tc
                   JOIN information_schema.key_column_usage kcu
                     ON tc.constraint_name=kcu.constraint_name AND tc.table_schema=kcu.table_schema
                   JOIN information_schema.constraint_column_usage ccu
                     ON ccu.constraint_name=tc.constraint_name AND ccu.table_schema=tc.table_schema
                   WHERE tc.constraint_type='FOREIGN KEY'
                     AND tc.table_schema=%s AND tc.table_name=%s""",
                (DB_SCHEMA, t))
            fks = cur.fetchall()

            lines.append(f"TABLE {t} (")
            for c in cols:
                pk = " PRIMARY KEY" if c["column_name"] in pks else ""
                nn = "" if c["is_nullable"] == "YES" else " NOT NULL"
                lines.append(f"    {c['column_name']} {c['data_type']}{nn}{pk}")
            for fk in fks:
                lines.append(f"    FOREIGN KEY ({fk['col']}) REFERENCES {fk['ref_table']}({fk['ref_col']})")
            lines.append(")\n")
    return "\n".join(lines), table_columns

SCHEMA_TEXT, TABLE_COLUMNS = load_schema() if KNOWN_TABLES else ("", {})
print(SCHEMA_TEXT if SCHEMA_TEXT else "⚠️ Schema is empty — fix Section A.1 first.")

## A.3 — 📚 Table notes: your business rules in plain English

Column names cannot tell the model that `suppliers_old` holds historical rows that must be
**combined with** `suppliers` via `UNION ALL` for totals. Everything you write here is
injected into *every* SQL-side prompt — the gate, the ambiguity check and generation.

**This is the #1 fix for "it only looked in one table."**

In [ ]:
# ⬅️ EDIT THESE for your real tables. Delete the examples you don't need.
TABLE_NOTES = {
    # "suppliers":     "Current/active suppliers only.",
    # "suppliers_old": "Historical suppliers moved out of the main table. For totals, "
    #                  "'highest', rankings or any aggregate question, COMBINE this table "
    #                  "with suppliers using UNION ALL unless the user says 'current only'.",
    # "orders":        "One row per order. status can be 'completed', 'pending', 'cancelled'.",
}

GENERAL_RULES = [
    # "Amounts are stored in INR.",
    # "Unless the user says otherwise, exclude rows where status = 'cancelled'.",
]

def schema_context():
    """Schema + your notes — this is what every model call sees."""
    ctx = SCHEMA_TEXT
    notes = [f"- {t}: {note}" for t, note in TABLE_NOTES.items() if t.lower() in
             {k.lower() for k in KNOWN_TABLES}]
    unknown_notes = [t for t in TABLE_NOTES if t.lower() not in {k.lower() for k in KNOWN_TABLES}]
    if unknown_notes:
        print(f"⚠️  TABLE_NOTES mentions tables not in your DB (typo?): {', '.join(unknown_notes)}")
    if notes:
        ctx += "\n\nIMPORTANT notes about these tables (follow them strictly):\n" + "\n".join(notes)
    if GENERAL_RULES:
        ctx += "\n\nGeneral rules (follow them strictly):\n" + "\n".join(f"- {r}" for r in GENERAL_RULES)
    return ctx

print("✅ Table notes loaded:", len(TABLE_NOTES), "table note(s),", len(GENERAL_RULES), "general rule(s).")
print("   (Empty is OK to start — but filling this in is the #1 fix for wrong-table answers.)")

## A.4 — The answerability gate + ambiguity handling

1. **Gate** — refuses, with a reason, anything your database cannot answer.
2. **Split-table menu (deterministic)** — for families like `bids` / `bids_old`, *code* builds
   a menu that always has all three options: current / historical / both combined. A Step A.3
   note for those tables decides automatically and suppresses the menu.
3. **Ambiguity check (model fallback)** — other ambiguity (same figure in two unrelated
   tables) is surfaced instead of silently resolved.

In [ ]:
def is_answerable(question):
    """Strict gatekeeper: can this question be answered from THIS schema only?"""
    prompt = f"""You are a strict gatekeeper for a SQL system. Below is the COMPLETE database schema.

{schema_context()}

Question: {question}

Decide if this question can be answered using ONLY the tables and columns above.
Rules:
- Greetings, chit-chat, math, coding help, or general knowledge → NOT answerable.
- Mentions data/entities/tables/columns that do not exist in this schema → NOT answerable,
  and list the missing things.
- Only mark answerable if a correct SELECT query over these exact tables can answer it.
Respond with RAW JSON only:
{{"answerable": true or false, "reason": "<one sentence>", "missing": ["<things not in the schema, if any>"]}}"""
    resp = genai_client.models.generate_content(model=SQL_MODEL, contents=prompt)
    try:
        data = _parse_json_reply(resp.text)
    except Exception:
        # If the gate itself misbehaves, fail CLOSED (refuse), never open.
        return {"answerable": False, "reason": "Gate response was unreadable — refusing to guess.", "missing": []}
    return {"answerable": bool(data.get("answerable")),
            "reason": data.get("reason", ""),
            "missing": data.get("missing", []) or []}

def check_ambiguity(question):
    """Could this question be answered from MORE THAN ONE table/column?"""
    prompt = f"""You are helping disambiguate a database question. Complete schema:

{schema_context()}

Question: {question}

Could this question reasonably be answered in MORE THAN ONE way from this schema — for
example, the same information (an amount, a total, a date, a name) exists in two or more
different tables or columns? Only report REAL alternatives that exist in this schema.
If the same KIND of rows is split across tables (e.g. a current table and an "_old"/archive
table), one interpretation should be "combine both tables with UNION ALL". If a table note
already says how to handle it, follow the note instead of reporting ambiguity.

Respond with RAW JSON only:
{{"ambiguous": true or false,
  "interpretations": [
    {{"summary": "<short plain-English description of this interpretation>",
      "tables": ["<table(s) it would use>"],
      "clarified_question": "<the question rewritten so it is 100% unambiguous>"}}
  ]}}
If NOT ambiguous, return "ambiguous": false with exactly ONE interpretation (the obvious one)."""
    resp = genai_client.models.generate_content(model=SQL_MODEL, contents=prompt)
    try:
        data = _parse_json_reply(resp.text)
    except Exception:
        return {"ambiguous": False, "interpretations": []}
    interps = [i for i in (data.get("interpretations") or []) if isinstance(i, dict)]
    return {"ambiguous": bool(data.get("ambiguous")) and len(interps) > 1,
            "interpretations": interps}

# ---- deterministic split-table handling (bids / bids_old, suppliers / suppliers_old, ...)
SPLIT_SUFFIXES = ("_old", "_history", "_hist", "_archive", "_archived", "_backup", "_prev")

def detect_split_families():
    """Find table families that hold the same kind of rows, e.g. {'bids': ['bids_old']}."""
    lower = {t.lower() for t in KNOWN_TABLES}
    families = {}
    for t in lower:
        for suf in SPLIT_SUFFIXES:
            if t.endswith(suf) and t[: -len(suf)] in lower:
                families.setdefault(t[: -len(suf)], []).append(t)
        if t.startswith("old_") and t[4:] in lower:
            families.setdefault(t[4:], []).append(t)
    return families

def split_table_interpretations(question):
    """If the question touches a split family and no Section A.3 note decides it,
    return the GUARANTEED three options. Built by code — never missing a choice."""
    q = question.lower()
    noted = {k.lower() for k in TABLE_NOTES}
    for base, olds in sorted(detect_split_families().items()):
        if base in noted or any(o in noted for o in olds):
            continue                       # your Section A.3 note decides — no menu needed
        stem = base[:-1] if base.endswith("s") else base
        if stem not in q:
            continue                       # question isn't about this family
        all_tables = [base] + sorted(olds)
        return [
            {"summary": f"Current data only ({base})",
             "tables": [base],
             "clarified_question": f"{question} — use ONLY the {base} table; do not use {', '.join(olds)}"},
            {"summary": f"Historical data only ({', '.join(sorted(olds))})",
             "tables": sorted(olds),
             "clarified_question": f"{question} — use ONLY the {', '.join(sorted(olds))} table(s); do not use {base}"},
            {"summary": f"Current + historical combined ({' + '.join(all_tables)})",
             "tables": all_tables,
             "clarified_question": (f"{question} — combine {' and '.join(all_tables)} "
                                    f"with UNION ALL before aggregating")},
        ]
    return None

fam = detect_split_families()
if fam:
    print("🧩 Split-table families detected:",
          "; ".join(f"{b} + {', '.join(o)}" for b, o in sorted(fam.items())))
    print("   Questions touching these will get a guaranteed 3-option menu")
    print("   (current / historical / both) unless a Section A.3 note decides for you.")
print("✅ Answerability gate + ambiguity checks ready.")

## A.5 — The SQL-generation prompt

Embeds the exact list of allowed tables and forbids guessing.
👉 Once things work, add few-shot examples about *your* tables — the single biggest accuracy win.

In [ ]:
SQL_SYSTEM_PROMPT = """You are an expert PostgreSQL query writer. Convert the user's
natural-language question into ONE correct, read-only PostgreSQL query.

Rules (follow ALL of them):
1. PostgreSQL dialect only.
2. Output ONLY SELECT statements (WITH ... SELECT allowed). NEVER INSERT, UPDATE, DELETE,
   DROP, ALTER, TRUNCATE, CREATE, GRANT or anything that changes data or schema.
3. Use ONLY tables and columns that appear in the schema below. NEVER invent or guess names.
   If the question mentions anything not in the schema, return an empty "sql" and explain
   what is missing. Do NOT substitute a similar-looking table instead.
4. Use explicit JOINs and table aliases; qualify columns when joining.
5. If the SAME KIND of rows is split across tables (e.g. suppliers and suppliers_old),
   combine them with UNION ALL before aggregating for totals/highest/rankings —
   do NOT silently use only one of them, and do NOT JOIN them to each other.
6. Follow every table note and general rule in the schema section strictly.
7. Respond with RAW JSON only (no markdown fences), exactly:
   {"sql": "<query or empty string>", "explanation": "<one short paragraph>"}
"""

FEW_SHOT_EXAMPLES = """
Question: How many orders did each customer place?
{"sql": "SELECT c.id, c.name, COUNT(o.id) AS order_count FROM customers c LEFT JOIN orders o ON o.customer_id = c.id GROUP BY c.id, c.name ORDER BY order_count DESC;", "explanation": "Counts orders per customer, including customers with zero orders."}

Question: Which products have never been ordered?
{"sql": "SELECT p.id, p.name FROM products p WHERE NOT EXISTS (SELECT 1 FROM order_items oi WHERE oi.product_id = p.id);", "explanation": "Anti-join keeps products with no matching order_items row."}

Question: Show me the employee salaries
{"sql": "", "explanation": "The schema has no employees or salaries table, so this cannot be answered."}

Question (schema where supplier rows are split between suppliers and suppliers_old): Which supplier has the highest total amount?
{"sql": "WITH all_suppliers AS (SELECT name, category, amount FROM suppliers UNION ALL SELECT name, category, amount FROM suppliers_old) SELECT name, category, SUM(amount) AS total_amount FROM all_suppliers GROUP BY name, category ORDER BY total_amount DESC LIMIT 1;", "explanation": "Current and old supplier rows are the same kind of data, so they are combined with UNION ALL before aggregating — a JOIN would be wrong here."}
"""

def build_sql_prompt(question, previous_error=None):
    allowed = ", ".join(sorted(KNOWN_TABLES))
    p = (SQL_SYSTEM_PROMPT
         + f"\nALLOWED TABLES (use no others): {allowed}\n"
         + "\nDatabase schema (including notes you MUST follow):\n" + schema_context()
         + "\nExamples:\n" + FEW_SHOT_EXAMPLES
         + f"\nQuestion: {question}\nAnswer with raw JSON only.")
    if previous_error:
        p += (f"\n\nIMPORTANT: your previous attempt failed with this error:\n{previous_error}"
              "\nFix the query and answer with raw JSON only.")
    return p

print("✅ SQL prompt builder ready.")

## A.6 — Validators (five layers between the model and your data)

| Layer | Catches |
|---|---|
| 1. Read-only / single statement | `DELETE`, `DROP`, a hidden `; DROP`, any write |
| 2. Table existence | tables not in your DB (CTE-aware) |
| 3. Column existence | `alias.column` pairs where the column isn't in that table |
| 4. `EXPLAIN` dry-run | everything else — Postgres plans it without running it |
| 5. Read-only execution | `readonly` session, always rolled back, row-capped |

A query must also touch **at least one real table**, so `SELECT 2+2` cannot slip through.

In [ ]:
FORBIDDEN = {"INSERT","UPDATE","DELETE","DROP","ALTER","TRUNCATE","CREATE",
             "GRANT","REVOKE","MERGE","CALL","COPY","VACUUM","REINDEX","COMMENT","SET"}

_SQL_STOPWORDS = {"ON","WHERE","GROUP","ORDER","LEFT","RIGHT","INNER","OUTER","JOIN",
                  "CROSS","FULL","LIMIT","HAVING","UNION","USING","AS","SELECT","WITH",
                  "NATURAL","LATERAL","OFFSET","FETCH"}

def _strip_literals(sql_text):
    s = re.sub(r"/\*.*?\*/", " ", sql_text, flags=re.DOTALL)
    s = re.sub(r"--[^\n]*", " ", s)
    s = re.sub(r"'(?:[^']|'')*'", "''", s)
    # EXTRACT(month FROM x), SUBSTRING(x FROM 1), TRIM(... FROM x) contain a fake FROM —
    # blank their argument lists so the table scanner doesn't trip on them
    s = re.sub(r"\b(EXTRACT|SUBSTRING|TRIM|OVERLAY|POSITION)\s*\([^()]*\)", r"\1()", s, flags=re.I)
    return s

def _scan_tables(cleaned):
    """Return (used_tables, alias_map, cte_names) from FROM/JOIN clauses."""
    cte_names = {m.group(1).lower() for m in
                 re.finditer(r"\b([A-Za-z_]\w*)\s+AS\s*\(", cleaned, re.I)}
    used, alias_map = set(), {}
    for m in re.finditer(
            r"\b(?:FROM|JOIN)\s+([A-Za-z_][\w.]*)(?:\s+(?:AS\s+)?([A-Za-z_]\w*))?",
            cleaned, re.I):
        tbl = m.group(1).split(".")[-1].strip('"').lower()
        used.add(tbl)
        alias = m.group(2)
        if alias and alias.upper() not in _SQL_STOPWORDS:
            alias_map[alias.lower()] = tbl
        alias_map.setdefault(tbl, tbl)   # a table is its own alias
    return used, alias_map, cte_names

def validate_sql(sql_text):
    """Layers 1–3. Returns (ok, reason)."""
    if not sql_text or not sql_text.strip():
        return False, "Empty query."
    cleaned = _strip_literals(sql_text).strip()

    if ";" in cleaned.rstrip(";"):
        return False, "Multiple SQL statements are not allowed."
    first = re.match(r"\s*([A-Za-z]+)", cleaned)
    first_kw = first.group(1).upper() if first else ""
    if first_kw not in ("SELECT", "WITH"):
        return False, f"Only SELECT/WITH queries are allowed (got '{first_kw}')."
    for kw in FORBIDDEN:
        if re.search(rf"\b{kw}\b", cleaned.upper()):
            return False, f"Forbidden keyword found: {kw}."

    known = {t.lower() for t in KNOWN_TABLES}
    used, alias_map, cte_names = _scan_tables(cleaned)

    unknown = used - known - cte_names
    if unknown:
        return False, (f"Query references tables that do NOT exist in your database: "
                       f"{', '.join(sorted(unknown))}. Your tables: {', '.join(sorted(known))}.")
    if not (used & known):
        return False, ("Query does not use any table from your database — refusing "
                       "(questions must be about YOUR data).")

    # Layer 3: every alias.column must exist in that table
    for m in re.finditer(r"\b([A-Za-z_]\w*)\.([A-Za-z_]\w*)\b", cleaned):
        alias, col = m.group(1).lower(), m.group(2).lower()
        tbl = alias_map.get(alias)
        if tbl in TABLE_COLUMNS and col != "*" and col not in TABLE_COLUMNS[tbl]:
            return False, (f"Column '{col}' does not exist in table '{tbl}'. "
                           f"Available columns: {', '.join(sorted(TABLE_COLUMNS[tbl]))}.")
    return True, ""

def tables_in(sql_text):
    used, _, cte = _scan_tables(_strip_literals(sql_text))
    return sorted(used - cte)

def explain_check(sql_text):
    """Layer 4: PostgreSQL plans the query WITHOUT executing it."""
    try:
        with get_conn() as conn:
            conn.set_session(readonly=True)
            with conn.cursor() as cur:
                cur.execute("EXPLAIN " + sql_text)
            conn.rollback()
        return True, ""
    except Exception as e:
        return False, str(e).strip()

def add_limit(sql_text, max_rows=MAX_ROWS):
    if re.search(r"\bLIMIT\s+\d+", sql_text, re.I):
        return sql_text
    return sql_text.rstrip().rstrip(";") + f" LIMIT {max_rows};"

print("✅ Hardened validators ready.")

## A.7 — `ask_sql()` — the SQL pipeline

Gate → ambiguity → generate → validate → `EXPLAIN` → execute → check output.
Any failure is fed back to Gemini as an error message for another attempt (up to 3).

| `ask_sql("...", on_ambiguity=...)` | Behaviour |
|---|---|
| `"ask"` (default) | lists interpretations, asks you to type a number |
| `"both"` | runs every interpretation and labels each result |
| `None` | let the model pick (not recommended) |

In [ ]:
def generate_sql(question, previous_error=None):
    resp = genai_client.models.generate_content(
        model=SQL_MODEL, contents=build_sql_prompt(question, previous_error))
    try:
        data = _parse_json_reply(resp.text)
    except Exception:
        return "", "Model returned unreadable output."
    return (data.get("sql") or "").strip(), (data.get("explanation") or "").strip()

def run_query(sql_text):
    with get_conn() as conn:
        conn.set_session(readonly=True)
        with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            cur.execute(add_limit(sql_text))
            rows = cur.fetchall()
        conn.rollback()
    return pd.DataFrame(rows)

def validate_output(df):
    if df.empty:
        print("⚠️  Query ran fine but returned 0 rows — either the filter matched nothing,")
        print("    or the table is empty (see Section A.1's health report).")
        return
    null_cols = [c for c in df.columns if df[c].isna().all()]
    if null_cols:
        print(f"⚠️  Columns entirely NULL: {', '.join(null_cols)} (often an unmatched LEFT JOIN).")
    if len(df) >= MAX_ROWS:
        print(f"⚠️  Result capped at {MAX_ROWS} rows (raise MAX_ROWS in Section 0.2 if needed).")
    print(f"✅ {len(df)} row(s), {len(df.columns)} column(s).")

def _answer_sql(question, execute=True, max_attempts=3):
    """Generate → validate → EXPLAIN → execute, with self-correction retries."""
    error = None
    for attempt in range(1, max_attempts + 1):
        sql_text, explanation = generate_sql(question, error)
        if not sql_text:
            print(f"🤷 The model could not answer from your schema.\n   Reason: {explanation}")
            return None

        ok, reason = validate_sql(sql_text)
        if not ok:
            print(f"   attempt {attempt}: rejected → {reason}")
            error = reason
            continue

        ok, reason = explain_check(sql_text)
        if not ok:
            print(f"   attempt {attempt}: PostgreSQL rejected the plan → {reason}")
            error = reason
            continue

        print("─" * 60)
        print("📝 SQL:\n" + sql_text)
        print("\n📋 Tables used: " + ", ".join(f"{t} ✅" for t in tables_in(sql_text)))
        print("💡 " + explanation)
        print("─" * 60)
        if not execute:
            return sql_text

        try:
            df = run_query(sql_text)
        except Exception as e:
            error = str(e).strip()
            print(f"   attempt {attempt}: execution failed → {error}")
            continue

        validate_output(df)
        if not df.empty:
            print("\n📊 ANSWER:")
            display(df)          # always show the result table, no matter how ask() was called
        return df

    print(f"❌ Gave up after {max_attempts} attempts. Last error: {error}")
    return None

def ask_sql(question, execute=True, max_attempts=3, on_ambiguity="ask"):
    if not KNOWN_TABLES:
        print("❌ No business tables available — complete Section A.1 first.")
        return None

    # 1) the gate — refuse before generating anything
    gate = is_answerable(question)
    if not gate["answerable"]:
        print("🚫 REFUSED — this question cannot be answered from your database.")
        print(f"   Reason: {gate['reason']}")
        if gate["missing"]:
            print(f"   Not in your schema: {', '.join(str(x) for x in gate['missing'])}")
        print(f"   Tables you DO have: {', '.join(sorted(KNOWN_TABLES))}")
        return None

    # 2) ambiguity — same info in more than one table? Don't silently pick.
    if on_ambiguity in ("ask", "both"):
        # deterministic split-table menu first (always complete: current/historical/both)
        interps = split_table_interpretations(question)
        if interps is None:
            amb = check_ambiguity(question)   # model-based fallback for other ambiguity
            interps = amb["interpretations"] if amb["ambiguous"] else None
        if interps:
            print(f"🔀 Your question could mean {len(interps)} different things:")
            for i, it in enumerate(interps, 1):
                tbls = ", ".join(it.get("tables", [])) or "?"
                print(f"   {i}. {it.get('summary', '(no description)')}   [tables: {tbls}]")

            if on_ambiguity == "both":
                results = {}
                for i, it in enumerate(interps, 1):
                    print("\n" + "=" * 60)
                    print(f"▶ Interpretation {i}: {it.get('summary', '')}")
                    print("=" * 60)
                    q = it.get("clarified_question") or question
                    results[it.get("summary", f"interpretation {i}")] = _answer_sql(q, execute, max_attempts)
                return results

            # default "ask": let the user choose
            choice = input(f"   Which one did you mean? [1-{len(interps)}, Enter = 1]: ").strip()
            idx = int(choice) - 1 if choice.isdigit() and 1 <= int(choice) <= len(interps) else 0
            chosen = interps[idx]
            print(f"   → Using interpretation {idx + 1}: {chosen.get('summary', '')}")
            question = chosen.get("clarified_question") or question

    return _answer_sql(question, execute, max_attempts)

print("✅ SQL engine ready — ask_sql(\"your question\"), or use the router in Section C.")

---
# Section B — RAG engine

Natural language → **retrieved passages** → grounded answer with citations.

An LLM cannot know what is in your private documents, and if you just ask it, it invents
something plausible. So we **find** the relevant text first, **paste it into the prompt**, and
require the model to answer *only* from that. Retrieval does the knowing; the LLM does the writing.

```
documents → chunk (with a contextual header) → embed → store in pgvector
question  → embed → vector search + keyword search → fuse (RRF) → rerank → generate with [citations]
```

**Two design decisions worth knowing:**

1. **Chunks carry a contextual header.** Each chunk is stored twice: `content` (clean text,
   shown to you) and `embed_input` (`Document: … | Supplier Name: … | Category: …` + the text,
   what is actually embedded and keyword-indexed). Without it, a chunk reading
   *"Customer satisfaction: 4.8/5"* has no supplier attached and nobody can tell whose score it is.
2. **Retrieval is hybrid.** Dense vectors are weakest at exactly the tokens people search by —
   `BID-2024-089`, `4.8/5`, `Net-45`, a SKU. The keyword leg catches those; RRF merges the two lists.

## B.1 — pgvector schema

`rag_documents` (source text) and `rag_chunks` (chunks + vectors + a generated `tsvector`
for the keyword leg). Re-runnable: every statement is `IF NOT EXISTS`.

In [ ]:
# An f-string (f""" ... """) substitutes {DOC_TABLE}, {EMBED_DIM} etc. with the Section 0.2 values.
# Because of that, any brace we want to keep LITERAL in the SQL must be doubled: '{{}}' -> '{}'.
DDL = f"""
-- pgvector adds the `vector` column type and the <=> distance operator. Without it the
-- CREATE TABLE below fails with 'type "vector" does not exist'. Safe to re-run.
CREATE EXTENSION IF NOT EXISTS vector;

-- ============ Table 1: the original documents, exactly as you supplied them ============
CREATE TABLE IF NOT EXISTS {DOC_TABLE} (
    doc_id        text PRIMARY KEY,     -- your own stable id, e.g. 'acme-produce-2024'
    title         text NOT NULL,
    content       text NOT NULL,        -- the full, un-chunked text
    metadata      jsonb NOT NULL DEFAULT '{{}}'::jsonb,  -- supplier, category, dates... free-form.
                                        -- jsonb (not json) is the binary form: indexable and faster.
                                        -- Using one jsonb column instead of 10 fixed columns means
                                        -- adding a new field later needs no schema migration.
    content_hash  text NOT NULL,        -- fingerprint of what this document says RIGHT NOW
    indexed_hash  text,                 -- fingerprint of what we last EMBEDDED. NULL = never indexed.
                                        -- content_hash != indexed_hash  =>  needs re-embedding.
                                        -- Comparing these two is the whole trick behind Section B.4.
    created_at    timestamptz NOT NULL DEFAULT now(),
    updated_at    timestamptz NOT NULL DEFAULT now()
);

-- ============ Table 2: the chunks + their vectors (what search actually reads) ==========
CREATE TABLE IF NOT EXISTS {CHUNK_TABLE} (
    chunk_id     bigserial PRIMARY KEY,  -- bigserial = auto-incrementing 64-bit id
    doc_id       text NOT NULL REFERENCES {DOC_TABLE}(doc_id) ON DELETE CASCADE,
                                         -- REFERENCES = this must match a real document.
                                         -- ON DELETE CASCADE = deleting a document automatically
                                         -- deletes its chunks, so you can never leave orphaned
                                         -- vectors behind that still show up in search results.
    chunk_index  int  NOT NULL,          -- 0, 1, 2... position of this chunk within its document
    content      text NOT NULL,          -- CLEAN text: what we show the LLM and the user
    embed_input  text NOT NULL,          -- content + its contextual header: what we actually EMBEDDED
                                         -- and what keyword search reads. Keeping the two separate is
                                         -- what fixes the satisfaction bug without showing the user
                                         -- an ugly metadata header. See Section B.2.
    token_count  int  NOT NULL,          -- stored so you can audit chunk sizes without re-tokenising
    metadata     jsonb NOT NULL DEFAULT '{{}}'::jsonb,  -- copied down from the parent document so we
                                         -- can filter DURING the vector search, not after it
    embedding    vector({EMBED_DIM}) NOT NULL,  -- the {EMBED_DIM} numbers. The size is fixed at table
                                         -- creation: another reason one table = one embedding model.

    -- A GENERATED column is computed by Postgres and kept up to date automatically - you never
    -- write to it, and it can never drift out of sync with the text. to_tsvector() strips
    -- punctuation, lowercases, and reduces words to stems ('deliveries' -> 'deliveri') so that
    -- searching "delivery" also matches "deliveries". coalesce(x,'') guards against NULL.
    tsv tsvector GENERATED ALWAYS AS (to_tsvector('english', coalesce(embed_input, ''))) STORED,

    created_at   timestamptz NOT NULL DEFAULT now(),
    UNIQUE (doc_id, chunk_index)         -- one row per (document, position). Makes re-ingest safe.
);

-- ============ Indexes: how Postgres avoids scanning every row =========================
-- GIN ("generalised inverted index") is the right index type for search-inside-a-value
-- columns. It maps each word -> the rows containing it, like the index at the back of a book.
CREATE INDEX IF NOT EXISTS {CHUNK_TABLE}_tsv_gin  ON {CHUNK_TABLE} USING gin (tsv);
-- jsonb_path_ops is a smaller, faster GIN variant that supports the @> ("contains") operator,
-- which is exactly what our metadata filters use.
CREATE INDEX IF NOT EXISTS {CHUNK_TABLE}_meta_gin ON {CHUNK_TABLE} USING gin (metadata jsonb_path_ops);
-- Plain B-tree index: makes "DELETE ... WHERE doc_id = ..." during re-ingest fast.
CREATE INDEX IF NOT EXISTS {CHUNK_TABLE}_doc_id   ON {CHUNK_TABLE} (doc_id);
"""
# NOTE: the HNSW *vector* index is deliberately NOT created here - see Section B.4. Building it
# on an empty table and then inserting is much slower than loading first, indexing once.

with db() as cur:
    cur.execute(DDL)   # psycopg2 happily runs several statements separated by ';' in one call
print(f"Schema ready: {DOC_TABLE}, {CHUNK_TABLE} (vector({EMBED_DIM}) + tsvector + jsonb).")

## B.2 — Structure-aware chunking + contextual headers

Split on paragraphs, then sentences, then (only if a single sentence exceeds the whole
budget) on tokens. Overlap is carried over as **whole sentences**, so a fact sitting on a
chunk boundary still appears intact somewhere.

In [ ]:
import re   # regular expressions: pattern-matching on text

# A blank line = a paragraph break. \n is a newline, \s* is "any whitespace, possibly none".
# So this matches "newline, optional blank space, newline".
_PARA = re.compile(r"\n\s*\n")

# A sentence boundary. Read it in three parts:
#   (?<=[.!?])      "lookbehind": the character BEFORE this point must be . ! or ?
#   \s+             one or more spaces/newlines (this is the part actually removed)
#   (?=[A-Z0-9("'])  "lookahead": what FOLLOWS must start like a new sentence
# The lookahead is what stops it splitting "3.20 USD" or "Net-30. delivery" incorrectly.
_SENT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9(\"'])")

def _hard_split(unit: str, max_tokens: int):
    """Last resort: chop blindly at token boundaries.

    Only reached when a single 'sentence' is longer than the ENTIRE chunk budget - e.g. a
    giant table row or a wall of text with no punctuation. Ugly, but better than emitting a
    chunk too big to embed.
    """
    toks = _enc.encode(unit)
    return [_enc.decode(toks[i:i + max_tokens]) for i in range(0, len(toks), max_tokens)]

def split_text(text: str, max_tokens: int = CHUNK_TOKENS,
               overlap_tokens: int = CHUNK_OVERLAP_TOKENS):
    """Cut one document into chunks. This is THE chunker - v2 has exactly one.

    Two phases:
      Phase 1 - break the text into the smallest sensible units (sentences).
      Phase 2 - glue those units back together into chunks just under the token budget.

    This is called "recursive" splitting: try the biggest natural boundary first
    (paragraph), fall back to a smaller one (sentence), and only cut mid-sentence when
    there is genuinely no alternative.
    """
    # ---------- Phase 1: text -> list of sentences ----------
    units = []
    for para in _PARA.split(text.strip()):      # first split on blank lines
        para = para.strip()
        if not para:                            # skip empty results
            continue
        for sent in _SENT.split(para):          # then split each paragraph into sentences
            sent = sent.strip()
            if not sent:
                continue
            # If even one sentence blows the budget, chop it; otherwise keep it whole.
            units.extend(_hard_split(sent, max_tokens) if n_tokens(sent) > max_tokens else [sent])

    # ---------- Phase 2: sentences -> chunks ----------
    chunks = []      # finished chunks
    buf = []         # sentences accumulated for the chunk currently being built
    buf_tokens = 0   # running token count of buf, so we do not re-count it every iteration

    for unit in units:
        t = n_tokens(unit)

        # Would adding this sentence overflow the budget? Then finish the current chunk first.
        # `buf and` prevents an empty chunk on the very first iteration.
        if buf and buf_tokens + t > max_tokens:
            chunks.append(" ".join(buf))        # emit the completed chunk

            # ----- overlap: start the NEXT chunk with the last few sentences of this one -----
            # Why: if a fact sits exactly on a chunk boundary ("...satisfaction was" | "4.8/5
            # stars"), neither chunk contains it in full and retrieval loses it. Repeating a
            # little text means the fact appears intact in at least one chunk.
            # We walk BACKWARDS from the end, taking whole sentences until we hit the overlap
            # budget - whole sentences only, so overlap never starts mid-thought.
            keep, kept = [], 0
            for prev in reversed(buf):
                pt = n_tokens(prev)
                if kept + pt > overlap_tokens:  # one more would exceed the overlap budget
                    break
                keep.insert(0, prev)            # insert(0,...) rebuilds the original order
                kept += pt
            buf, buf_tokens = keep, kept        # the new chunk starts with the carried-over text

        buf.append(unit)
        buf_tokens += t

    if buf:                       # whatever is left over becomes the final chunk
        chunks.append(" ".join(buf))
    return chunks

# ======================= Contextual headers: the fix for your bug =======================
# Which metadata fields are worth spending header tokens on - the ones users search BY.
# Adding every field would dilute the embedding; these are the high-value ones.
HEADER_FIELDS = ["supplier_name", "category", "document_type", "bid_date", "status"]

def contextual_header(doc_id: str, title: str, metadata: dict) -> str:
    """Build a one-line 'where did this text come from' banner.

    Produces e.g.:
      Document: Q1 2024 Supplier Performance Review | ID: performance-review-2024 |
      Supplier Name: Riverside USD | Category: Performance Review | Bid Date: 2024-04-01
    """
    bits = [f"Document: {title}", f"ID: {doc_id}"]
    # For each interesting field that is actually present, add "Field Name: value".
    # .replace('_',' ').title() turns 'supplier_name' into 'Supplier Name'.
    # `if metadata.get(f)` skips fields that are missing, None, or empty.
    bits += [f"{f.replace('_', ' ').title()}: {metadata[f]}"
             for f in HEADER_FIELDS if metadata.get(f)]
    return " | ".join(bits)

def build_chunks(doc_id: str, title: str, content: str, metadata: dict):
    """Split a document AND attach its context header to every piece.

    Returns a list of dicts, one per chunk. The key idea is that 'content' and
    'embed_input' differ:
      content     -> clean text, shown to the LLM and the user
      embed_input -> header + clean text, what we embed and keyword-index
    So the header makes the chunk findable without ever appearing in the answer.
    """
    header = contextual_header(doc_id, title, metadata)   # same header for every chunk of this doc
    return [
        {
            "chunk_index": i,                       # enumerate() gives us 0, 1, 2, ...
            "content": piece,                       # shown to the LLM / user
            "embed_input": f"{header}\n\n{piece}",  # embedded + full-text indexed
            "token_count": n_tokens(piece),
        }
        for i, piece in enumerate(split_text(content))
    ]

_demo = build_chunks(
    "performance-review-2024", "Q1 2024 Supplier Performance Review",
    "Acme Foods maintained 98% on-time delivery. Customer satisfaction: 4.8/5 stars.",
    {"supplier_name": "Riverside USD", "category": "Performance Review", "bid_date": "2024-04-01"},
)
print(_demo[0]["embed_input"])

## B.3 — Getting documents in

`upsert_documents()` for documents you build in Python, `import_from_existing_table()` to
pull text straight out of a table you already have. Both are idempotent — the content hash
means re-running with unchanged text costs nothing.

In [ ]:
import hashlib, json
from psycopg2.extras import execute_values

def content_hash(title: str, content: str, metadata: dict) -> str:
    payload = json.dumps({"t": title, "c": content, "m": metadata},
                         sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def upsert_documents(doc_map: dict):
    """doc_map: {doc_id: {"title": str, "content": str, "metadata": dict}}.
    Idempotent - re-running with unchanged text is a no-op that Section B.4 will skip."""
    rows = []
    for doc_id, d in doc_map.items():
        title, content = d["title"], d["content"]
        meta = d.get("metadata", {})
        rows.append((doc_id, title, content, json.dumps(meta), content_hash(title, content, meta)))

    with db() as cur:
        execute_values(cur, f"""
            INSERT INTO {DOC_TABLE} (doc_id, title, content, metadata, content_hash) VALUES %s
            ON CONFLICT (doc_id) DO UPDATE SET
                title        = EXCLUDED.title,
                content      = EXCLUDED.content,
                metadata     = EXCLUDED.metadata,
                content_hash = EXCLUDED.content_hash,
                updated_at   = now()
        """, rows, template="(%s, %s, %s, %s::jsonb, %s)")
    print(f"Upserted {len(rows)} document(s).")

def delete_documents(doc_ids):
    """FK cascade removes the chunks and their vectors - no orphans."""
    doc_ids = list(doc_ids)
    with db() as cur:
        cur.execute(f"DELETE FROM {DOC_TABLE} WHERE doc_id = ANY(%s)", (doc_ids,))
    print(f"Deleted {len(doc_ids)} document(s) and their chunks.")

print("upsert_documents() / delete_documents() ready.")

In [ ]:
def import_from_existing_table(table: str, id_col: str, text_col: str,
                               title_col: str = None,
                               metadata_cols: list = None,
                               where: str = None, limit: int = None):
    """Copy rows from an existing table into rag_documents.

    metadata_cols become JSONB keys, so they are usable as retrieval filters *and*
    appear in the contextual header when they match HEADER_FIELDS.
    """
    metadata_cols = metadata_cols or []
    cols = [id_col, text_col] + ([title_col] if title_col else []) + metadata_cols
    sql = f"SELECT {', '.join(cols)} FROM {table}"
    if where:
        sql += f" WHERE {where}"
    if limit:
        sql += f" LIMIT {int(limit)}"

    with db(dict_rows=True) as cur:
        cur.execute(sql)
        rows = cur.fetchall()

    doc_map = {}
    for r in rows:
        if not r[text_col]:
            continue                       # nothing to chunk
        doc_id = str(r[id_col])
        doc_map[doc_id] = {
            "title": str(r[title_col]) if title_col and r[title_col] else doc_id,
            "content": r[text_col],
            "metadata": {c: (str(r[c]) if r[c] is not None else None) for c in metadata_cols},
        }
    if doc_map:
        upsert_documents(doc_map)
    return len(doc_map)

# Example - point it at your real table and uncomment:
# import_from_existing_table("bids", "bid_id", "description",
#                            title_col="bid_name",
#                            metadata_cols=["supplier_name", "category", "status"],
#                            where="status = 'Active'")
print("import_from_existing_table() ready.")

### B.3b — Seed corpus: 16 procurement documents

A self-contained document set for Riverside Unified School District, built to exercise every
part of the RAG pipeline and to carry real structure for a knowledge graph later.

| | |
|---|---|
| 3 RFPs | produce, dairy, frozen/protein — different evaluation weightings, one with lots |
| 1 addendum | amends the produce RFP: new due date, new delivery window, Q&A |
| 5 bid responses | two competing on produce, one dairy, two competing on frozen/protein |
| 1 award memo | the scoring table and why the *cheaper* produce bid lost |
| 2 contracts | payment terms, SLAs, liquidated damages, certification obligations |
| 1 performance review | Q1 FY2025 scorecard across all five suppliers |
| 1 supplier profile | Tidewater: corporate detail, subsidiary, facilities, risk notes |
| 1 compliance register | every certification, expiry date and contractual consequence |
| 1 email thread | the corrective-action conversation, in four messages |

**Why it's shaped this way.** Facts are deliberately consistent across documents — a price
quoted in a bid reappears in the award memo and again in the contract — so cross-document
questions have exactly one right answer and a wrong retrieval is visible. It is also seeded
with the traps that break naive RAG: two suppliers quote the *same* product at different
prices, per-supplier satisfaction is `4.8/5` while the district-wide index is `8.5/10`, and
exact tokens (`MSC-C-58817`, `Net-45`, `SKU VV-PRD-1180`, `2025-04-30`) are where dense
vectors are weakest and the keyword leg has to earn its place.

**Graph-ready, not graphed.** Every entity has a stable id (`SUP-1042`, `BID-2024-089`,
`CERT-TW-SQF2-STK`) used identically in every document that mentions it. Each document's
metadata carries `entities` (which ids it touches) and `relations` (subject-predicate-object
triples: `SUP-1157 SUBSIDIARY_OF SUP-1156`, `BID-2024-089 RESPONDS_TO RFP-2024-0412`).
Nothing reads them yet — 137 triples across 44 predicate types are simply sitting there for
whenever you build the graph, so you won't have to re-parse the prose.

Run this cell, then **B.4** to chunk and embed.

In [ ]:
# ============================================================================================
# SEED CORPUS — 15 food-service procurement documents for Riverside Unified School District.
#
# These are written to be RAG-realistic AND graph-ready:
#   • Every entity carries a stable ID (SUP-1042, RFP-2024-0412, BID-2024-089, CTR-…, PER-…)
#     and that SAME id is used in every document that mentions it.
#   • Facts are consistent across documents on purpose — a price quoted in a bid reappears in
#     the award memo and the contract, so cross-document questions have one right answer.
#   • metadata["entities"] lists every id the document touches; metadata["relations"] states
#     the relationships as (subject, predicate, object) triples. Nothing reads them yet — they
#     are there so a knowledge graph can be built later without re-parsing the prose.
#   • Exact tokens are sprinkled throughout (SKUs, "Net-45", "4.8/5", "$3.20/case", percentages)
#     because those are precisely where dense vectors are weak and the keyword leg earns its keep.
#
# The world, in one paragraph: Riverside USD (ORG-RUSD) ran three RFPs in spring 2024. Produce
# went to Valle Verde, dairy to Globex, frozen/protein split between Tidewater and Northlake.
# Tidewater owns Harbor Point Seafood, is the weakest performer, and has a certificate expiring.
# Acme Foods bid lower than Valle Verde on produce and still lost — on food safety and delivery.
# ============================================================================================

SOURCE_DOCS = {

# ─────────────────────────────────── RFPs ───────────────────────────────────
"rfp-2024-0412-produce": {
    "title": "RFP-2024-0412 — Fresh Produce Supply, Riverside USD, SY 2024-25",
    "content": (
        "REQUEST FOR PROPOSAL RFP-2024-0412\n"
        "Fresh Produce Supply for School Year 2024-25\n"
        "Issued by: Riverside Unified School District (ORG-RUSD), Nutrition Services Department\n"
        "Issue date: 2024-03-18. Original proposal due date: 2024-04-26, 2:00 PM PT. "
        "Extended to 2024-05-03 by Addendum No. 1.\n\n"

        "1. PURPOSE AND SCOPE\n"
        "Riverside Unified School District solicits proposals from qualified distributors for the "
        "supply and delivery of fresh fruits and vegetables to 42 school sites serving approximately "
        "28,400 students and 3.1 million meals annually. The estimated annual contract value is "
        "1,850,000 USD. The contract term runs 2024-08-01 through 2025-07-31, with two optional "
        "one-year renewals exercisable at the District's sole discretion.\n\n"

        "2. AWARD STRUCTURE\n"
        "The District intends to award a single prime contract for all produce categories. Partial "
        "or lot-based awards will be considered only if no single proposer can demonstrate capacity "
        "for the full scope. Cedar Valley Unified School District (ORG-CVUSD), 19 sites and 11,200 "
        "students, may piggyback on the resulting contract through the Central Valley Purchasing "
        "Cooperative under identical pricing and terms.\n\n"

        "3. EVALUATION CRITERIA\n"
        "Proposals are scored out of 100 points on the following weightings:\n"
        "  Price and total cost of ownership .......... 40%\n"
        "  Quality assurance and food safety .......... 25%\n"
        "  Delivery capability and reliability ........ 20%\n"
        "  Local sourcing commitment .................. 10%\n"
        "  MWBE participation ......................... 5%\n"
        "Price is scored on the extended annual basket total, not on unit price alone. A proposer "
        "may not be awarded on price advantage if it scores below 15 of 25 available points on "
        "quality assurance and food safety.\n\n"

        "4. MANDATORY REQUIREMENTS\n"
        "Proposers must hold a current GlobalG.A.P. or equivalent Good Agricultural Practices "
        "certification, and a current SQF Level 2 or higher facility certification for each "
        "distribution centre servicing this contract. Proposers must carry commercial general "
        "liability insurance of not less than 2,000,000 USD per occurrence and product liability "
        "of not less than 5,000,000 USD aggregate. A HACCP plan covering receiving, cold storage "
        "and transport must be submitted with the proposal.\n\n"

        "5. DELIVERY REQUIREMENTS\n"
        "Deliveries are required a minimum of twice weekly to each site. The original delivery "
        "window of 5:00 AM to 9:00 AM was revised to 4:30 AM to 8:30 AM by Addendum No. 1. "
        "Product must arrive at 34-41 degrees Fahrenheit for temperature-controlled items. Any "
        "delivery arriving outside the window without 24 hours prior notice is recorded as a late "
        "delivery for the purposes of the on-time performance metric.\n\n"

        "6. PERFORMANCE STANDARDS\n"
        "The awarded supplier must maintain a fill rate of at least 97% measured monthly by line "
        "item, and an on-time delivery rate of at least 95% measured monthly by delivery event. "
        "Quality rejection rate must remain below 2.0% of cases delivered.\n\n"

        "7. LOCAL SOURCING\n"
        "Points under the local sourcing criterion are awarded on the percentage of annual spend "
        "sourced from growers located within 250 miles of the District's central kitchen in "
        "Riverside, California. Addendum No. 1 clarified that the 250-mile radius is measured from "
        "the grower's primary production site, not from the distributor's warehouse.\n\n"

        "8. SUBMISSION\n"
        "Direct all questions to Miguel Arredondo (PER-MARREDONDO), Procurement Manager, "
        "purchasing@riversideusd.example.org. Questions closed 2024-04-12. Responses were issued "
        "in Addendum No. 1 on 2024-04-19. Six suppliers were formally solicited: Valle Verde "
        "Produce Co., Acme Foods Distribution, Globex Dairy Cooperative, Tidewater Frozen Holdings, "
        "Northlake Protein Partners and Summit Beverage Distributors."
    ),
    "metadata": {
        "document_type": "rfp", "category": "Produce",
        "rfp_id": "RFP-2024-0412", "buyer_id": "ORG-RUSD",
        "buyer_name": "Riverside Unified School District",
        "issue_date": "2024-03-18", "due_date": "2024-05-03",
        "estimated_value_usd": 1850000, "status": "awarded",
        "entities": ["RFP-2024-0412", "ORG-RUSD", "ORG-CVUSD", "PER-MARREDONDO",
                     "SUP-1042", "SUP-1077", "SUP-1103", "SUP-1156", "SUP-1189", "SUP-1204"],
        "relations": [
            ["ORG-RUSD", "ISSUED", "RFP-2024-0412"],
            ["RFP-2024-0412", "SOLICITS", "SUP-1042"],
            ["RFP-2024-0412", "SOLICITS", "SUP-1077"],
            ["RFP-2024-0412", "COVERS_CATEGORY", "Produce"],
            ["RFP-2024-0412", "AMENDED_BY", "ADD-2024-0412-001"],
            ["ORG-CVUSD", "MAY_PIGGYBACK_ON", "RFP-2024-0412"],
            ["PER-MARREDONDO", "CONTACT_FOR", "RFP-2024-0412"],
            ["PER-MARREDONDO", "WORKS_FOR", "ORG-RUSD"],
        ],
    },
},

"rfp-2024-0455-dairy": {
    "title": "RFP-2024-0455 — Dairy and Refrigerated Products, Riverside USD, SY 2024-25",
    "content": (
        "REQUEST FOR PROPOSAL RFP-2024-0455\n"
        "Dairy and Refrigerated Products for School Year 2024-25\n"
        "Issued by: Riverside Unified School District (ORG-RUSD)\n"
        "Issue date: 2024-04-02. Proposal due date: 2024-05-10, 2:00 PM PT.\n\n"

        "1. SCOPE\n"
        "The District seeks a single supplier for fluid milk, cultured dairy, cheese and "
        "refrigerated ready-to-eat items across 42 sites. Estimated annual value is 2,400,000 USD. "
        "Fluid milk alone represents approximately 4.6 million half-pint units annually. The "
        "contract term is 2024-08-01 through 2025-07-31 with two optional one-year renewals.\n\n"

        "2. EVALUATION CRITERIA\n"
        "  Price ........................................ 45%\n"
        "  Cold chain integrity and food safety ......... 25%\n"
        "  Delivery capability .......................... 20%\n"
        "  Sustainability and packaging ................. 10%\n\n"

        "3. COLD CHAIN REQUIREMENTS\n"
        "This is the District's controlling technical requirement. All fluid dairy must be "
        "maintained at 34 to 38 degrees Fahrenheit continuously from the processing plant to the "
        "point of delivery. Proposers must describe their telemetry capability: the District "
        "requires continuous temperature logging with data retained for a minimum of 24 months and "
        "made available to the District on request within two business days. Any load arriving "
        "above 41 degrees Fahrenheit is rejected in full at the supplier's cost and is recorded as "
        "a quality rejection event.\n\n"

        "4. MANDATORY REQUIREMENTS\n"
        "Proposers must hold a current SQF Level 2 or higher certification and a valid California "
        "Grade A dairy permit. Proposers must demonstrate the ability to deliver to all 42 sites "
        "within a four-hour window and to provide emergency replacement delivery within six hours "
        "of a rejected load.\n\n"

        "5. PACKAGING AND SUSTAINABILITY\n"
        "The District's Board resolution 2023-14 commits to reducing single-use plastic in "
        "nutrition services by 30% by 2027. Proposals are scored on paper-based carton use, "
        "recyclability, and any crate return or reuse programme offered at no additional charge.\n\n"

        "6. PRICE ADJUSTMENT\n"
        "Because fluid milk pricing is tied to federal milk marketing order Class I mover pricing, "
        "the District will accept monthly price adjustment on fluid milk line items only, indexed "
        "to the announced Class I mover, provided the proposer discloses its fixed markup over the "
        "mover at proposal time and that markup remains fixed for the contract term. All non-fluid "
        "line items are firm-fixed for the contract year, adjustable at renewal only, capped at "
        "CPI-U food-at-home or 4%, whichever is lower.\n\n"

        "7. PERFORMANCE STANDARDS\n"
        "Fill rate of at least 98% by line item measured monthly, given that a missed milk delivery "
        "cannot be absorbed by menu substitution. On-time delivery of at least 96%. Quality "
        "rejection rate below 1.0%.\n\n"

        "8. SUBMISSION\n"
        "Questions to Miguel Arredondo (PER-MARREDONDO), Procurement Manager. Three suppliers were "
        "solicited: Globex Dairy Cooperative (SUP-1103), Acme Foods Distribution (SUP-1077) and "
        "Tidewater Frozen Holdings (SUP-1156). One responsive proposal was received."
    ),
    "metadata": {
        "document_type": "rfp", "category": "Dairy",
        "rfp_id": "RFP-2024-0455", "buyer_id": "ORG-RUSD",
        "buyer_name": "Riverside Unified School District",
        "issue_date": "2024-04-02", "due_date": "2024-05-10",
        "estimated_value_usd": 2400000, "status": "awarded",
        "entities": ["RFP-2024-0455", "ORG-RUSD", "PER-MARREDONDO",
                     "SUP-1103", "SUP-1077", "SUP-1156"],
        "relations": [
            ["ORG-RUSD", "ISSUED", "RFP-2024-0455"],
            ["RFP-2024-0455", "SOLICITS", "SUP-1103"],
            ["RFP-2024-0455", "SOLICITS", "SUP-1077"],
            ["RFP-2024-0455", "SOLICITS", "SUP-1156"],
            ["RFP-2024-0455", "COVERS_CATEGORY", "Dairy"],
        ],
    },
},

"rfp-2024-0489-frozen-protein": {
    "title": "RFP-2024-0489 — Frozen Foods, Protein and Seafood, Riverside USD, SY 2024-25",
    "content": (
        "REQUEST FOR PROPOSAL RFP-2024-0489\n"
        "Frozen Foods, Protein and Seafood for School Year 2024-25\n"
        "Issued by: Riverside Unified School District (ORG-RUSD)\n"
        "Issue date: 2024-04-22. Proposal due date: 2024-05-31, 2:00 PM PT.\n\n"

        "1. SCOPE AND LOT STRUCTURE\n"
        "Estimated annual value 3,200,000 USD, the District's largest single solicitation for "
        "SY 2024-25. Unlike RFP-2024-0412 and RFP-2024-0455, this solicitation is structured in "
        "three lots and proposers may bid on any or all:\n"
        "  Lot 1 (LOT-0489-1) — Frozen prepared foods, frozen vegetables, frozen bakery. Est. 1,450,000 USD.\n"
        "  Lot 2 (LOT-0489-2) — Frozen and further-processed protein, including seafood. Est. 1,180,000 USD.\n"
        "  Lot 3 (LOT-0489-3) — Fresh and frozen poultry. Est. 570,000 USD.\n"
        "The District reserves the right to award each lot to a different supplier where doing so "
        "produces the best value.\n\n"

        "2. EVALUATION CRITERIA\n"
        "  Price ........................................ 35%\n"
        "  Food safety and traceability ................. 30%\n"
        "  Delivery capability .......................... 20%\n"
        "  Product range and menu fit ................... 15%\n"
        "Food safety carries a heavier weight here than in the District's other solicitations "
        "because of the protein and seafood content.\n\n"

        "3. FOOD SAFETY AND TRACEABILITY\n"
        "Proposers must hold SQF Level 2 or higher for every facility touching product under this "
        "contract, and must operate under a documented HACCP plan. Seafood items require Marine "
        "Stewardship Council Chain of Custody certification or an equivalent recognised by the "
        "District's Nutrition Services Director. Proposers must be able to trace any delivered lot "
        "to its production facility and production date within four hours of a District request. "
        "Where a proposer intends to fulfil any portion of Lot 2 through a subsidiary or affiliated "
        "entity, that entity must be named in the proposal and must independently satisfy every "
        "food safety requirement in this section.\n\n"

        "4. USDA FOODS PROCESSING\n"
        "The District diverts USDA Foods entitlement poultry and beef to further processing. "
        "Proposers must describe their capability to accept diverted commodity, track entitlement "
        "value, and reflect the commodity offset on invoices as a separate line.\n\n"

        "5. DELIVERY REQUIREMENTS\n"
        "Frozen product must arrive at 0 degrees Fahrenheit or below. Fresh poultry must arrive at "
        "28 to 34 degrees Fahrenheit. Deliveries are required weekly for frozen and twice weekly "
        "for fresh poultry. The District's central kitchen has 11 dock positions and will schedule "
        "arrival windows in 45-minute slots.\n\n"

        "6. PERFORMANCE STANDARDS\n"
        "Fill rate of at least 96% by line item, on-time delivery of at least 95%, quality "
        "rejection rate below 1.5%. A supplier falling below 92% on-time in any two consecutive "
        "months is subject to a mandatory corrective action plan.\n\n"

        "7. SUBMISSION\n"
        "Questions to Miguel Arredondo (PER-MARREDONDO). Four suppliers were solicited: Tidewater "
        "Frozen Holdings (SUP-1156), Northlake Protein Partners (SUP-1189), Acme Foods Distribution "
        "(SUP-1077) and Harbor Point Seafood (SUP-1157). Harbor Point Seafood did not submit an "
        "independent proposal and was instead named as the seafood fulfilment affiliate within the "
        "Tidewater Frozen Holdings proposal."
    ),
    "metadata": {
        "document_type": "rfp", "category": "Frozen Foods",
        "rfp_id": "RFP-2024-0489", "buyer_id": "ORG-RUSD",
        "buyer_name": "Riverside Unified School District",
        "issue_date": "2024-04-22", "due_date": "2024-05-31",
        "estimated_value_usd": 3200000, "status": "awarded",
        "entities": ["RFP-2024-0489", "ORG-RUSD", "PER-MARREDONDO",
                     "SUP-1156", "SUP-1189", "SUP-1077", "SUP-1157"],
        "relations": [
            ["ORG-RUSD", "ISSUED", "RFP-2024-0489"],
            ["RFP-2024-0489", "SOLICITS", "SUP-1156"],
            ["RFP-2024-0489", "SOLICITS", "SUP-1189"],
            ["RFP-2024-0489", "SOLICITS", "SUP-1077"],
            ["RFP-2024-0489", "SOLICITS", "SUP-1157"],
            ["RFP-2024-0489", "HAS_LOT", "LOT-0489-1"],
            ["RFP-2024-0489", "HAS_LOT", "LOT-0489-2"],
            ["RFP-2024-0489", "HAS_LOT", "LOT-0489-3"],
            ["SUP-1157", "SUBSIDIARY_OF", "SUP-1156"],
        ],
    },
},

"addendum-rfp-0412-001": {
    "title": "Addendum No. 1 to RFP-2024-0412 — Fresh Produce Supply",
    "content": (
        "ADDENDUM NO. 1 TO RFP-2024-0412 (document reference ADD-2024-0412-001)\n"
        "Issued: 2024-04-19 by Riverside Unified School District (ORG-RUSD)\n"
        "Acknowledgement of this addendum is mandatory. A proposal that fails to acknowledge "
        "Addendum No. 1 will be deemed non-responsive.\n\n"

        "PART A — CHANGES TO THE SOLICITATION\n\n"
        "A.1 Proposal due date. The due date is extended from 2024-04-26, 2:00 PM PT to "
        "2024-05-03, 2:00 PM PT. No further extension will be granted.\n\n"
        "A.2 Delivery window. Section 5 is amended. The delivery window is changed from 5:00 AM "
        "to 9:00 AM, to 4:30 AM to 8:30 AM. This change was made at the request of two site "
        "kitchen managers whose meal service begins at 7:15 AM and who reported insufficient "
        "receiving time under the original window.\n\n"
        "A.3 Local sourcing definition. Section 7 is clarified. The 250-mile radius is measured "
        "from the grower's primary production site to the District's central kitchen in Riverside, "
        "California. It is not measured from the distributor's warehouse. Proposers claiming local "
        "sourcing percentages must be able to substantiate them with grower addresses on request.\n\n"
        "A.4 Insurance. Section 4 is amended to clarify that the 5,000,000 USD product liability "
        "requirement may be met through a combination of primary and umbrella coverage.\n\n"

        "PART B — QUESTIONS AND ANSWERS\n"
        "Questions closed 2024-04-12. Eleven questions were received from three suppliers.\n\n"
        "Q1 (Valle Verde Produce Co., SUP-1042): May a proposer offer a volume discount tied to "
        "order value rather than to annual spend?\n"
        "A1: Yes. Volume discount structures will be evaluated as part of the extended annual "
        "basket total. State the threshold and the discount percentage clearly.\n\n"
        "Q2 (Acme Foods Distribution, SUP-1077): Will the District consider a proposer whose SQF "
        "certification for one of two distribution centres is pending renewal at proposal time?\n"
        "A2: No. Section 4 is a mandatory requirement. Every distribution centre servicing this "
        "contract must hold a current certificate on the proposal due date. A certificate in "
        "renewal is not a current certificate for the purposes of this solicitation.\n\n"
        "Q3 (Valle Verde Produce Co., SUP-1042): Is the 97% fill rate measured by case or by line "
        "item?\n"
        "A3: By line item, measured monthly. A line item short-shipped in any quantity counts as "
        "unfilled for that delivery event.\n\n"
        "Q4 (Acme Foods Distribution, SUP-1077): Can the twice-weekly delivery requirement be met "
        "with a single larger delivery for smaller sites?\n"
        "A4: No. Fresh produce shelf life at site level does not support consolidation. Twice "
        "weekly is a minimum at every site regardless of site size.\n\n"
        "Q5 (Tidewater Frozen Holdings, SUP-1156): Does the District intend to award produce and "
        "frozen categories to a single supplier?\n"
        "A5: No. RFP-2024-0412 covers produce only. Frozen, protein and seafood are solicited "
        "separately under RFP-2024-0489.\n\n"
        "Q6-Q11 concerned invoice formatting, EDI capability, the District's site list, crate "
        "return, substitution approval workflow and payment timing. The District confirmed that "
        "standard payment terms are Net-30 from receipt of a correct invoice, that early payment "
        "discount offers are welcome and will be evaluated within the price criterion, and that "
        "all substitutions require written approval from the Nutrition Services Director "
        "(PER-DWHITFIELD) before delivery.\n\n"
        "All other terms and conditions of RFP-2024-0412 remain unchanged."
    ),
    "metadata": {
        "document_type": "addendum", "category": "Produce",
        "addendum_id": "ADD-2024-0412-001", "rfp_id": "RFP-2024-0412",
        "buyer_id": "ORG-RUSD", "issue_date": "2024-04-19", "status": "issued",
        "entities": ["ADD-2024-0412-001", "RFP-2024-0412", "ORG-RUSD",
                     "SUP-1042", "SUP-1077", "SUP-1156", "PER-DWHITFIELD"],
        "relations": [
            ["ADD-2024-0412-001", "AMENDS", "RFP-2024-0412"],
            ["ORG-RUSD", "ISSUED", "ADD-2024-0412-001"],
            ["SUP-1042", "ASKED_QUESTION_IN", "ADD-2024-0412-001"],
            ["SUP-1077", "ASKED_QUESTION_IN", "ADD-2024-0412-001"],
            ["SUP-1156", "ASKED_QUESTION_IN", "ADD-2024-0412-001"],
            ["PER-DWHITFIELD", "APPROVES", "substitutions"],
        ],
    },
},

# ─────────────────────────────────── BIDS ───────────────────────────────────
"bid-2024-089-valleverde": {
    "title": "BID-2024-089 — Valle Verde Produce Co. response to RFP-2024-0412",
    "content": (
        "PROPOSAL BID-2024-089\n"
        "Submitted by: Valle Verde Produce Co. (SUP-1042)\n"
        "In response to: RFP-2024-0412, Fresh Produce Supply, Riverside Unified School District\n"
        "Submitted: 2024-05-01, 10:42 AM PT. Addendum No. 1 acknowledged.\n"
        "Total extended annual basket: 1,742,880 USD\n"
        "Primary contact: Elena Marquez (PER-EMARQUEZ), VP Sales, "
        "e.marquez@valleverdeproduce.example.com\n\n"

        "1. COMPANY\n"
        "Valle Verde Produce Co. was established in 1998 and is headquartered in Salinas, "
        "California. We employ 240 people and operate a single 96,000 square foot distribution "
        "centre in Fresno, California, certified SQF Level 2 (certificate expires 2025-09-14). "
        "We hold GlobalG.A.P. certification through 2025-06-30 and a USDA Organic handler "
        "certificate through 2025-11-01. We currently serve 38 K-12 districts across central and "
        "southern California.\n\n"

        "2. PRICING\n"
        "All prices are per case, delivered, firm-fixed for the contract year:\n"
        "  Romaine lettuce, 24 ct .............. 3.20 USD/case   SKU VV-PRD-1180\n"
        "  Spinach, baby, 4x2.5 lb ............. 4.10 USD/case   SKU VV-PRD-1204\n"
        "  Carrots, whole, 50 lb ............... 2.05 USD/case   SKU VV-PRD-1311\n"
        "  Brussels sprouts, 25 lb ............. 2.85 USD/case   SKU VV-PRD-1422\n"
        "  Bell peppers, mixed colour, 25 lb ... 3.50 USD/case   SKU VV-PRD-1508\n"
        "  Roma tomatoes, 25 lb ................ 3.75 USD/case   SKU VV-PRD-1560\n"
        "  Broccoli crowns, 20 lb .............. 4.40 USD/case   SKU VV-PRD-1614\n"
        "A further 61 line items are priced in Attachment C.\n\n"

        "3. COMMERCIAL TERMS\n"
        "Payment terms: Net-30 from receipt of a correct invoice. We do not offer an early payment "
        "discount. Minimum order value: 100 USD per site per delivery. Volume discount: 5% applied "
        "at the invoice level on any single site delivery exceeding 500 USD. Price escalation is "
        "not sought during the initial contract year. At renewal we request adjustment capped at "
        "CPI-U food-at-home or 4%, whichever is lower.\n\n"

        "4. DELIVERY\n"
        "We commit to twice-weekly delivery to all 42 sites on Tuesdays and Thursdays within the "
        "revised 4:30 AM to 8:30 AM window established by Addendum No. 1. We operate 34 refrigerated "
        "vehicles and will dedicate 6 to this contract. Emergency replacement delivery is available "
        "within four hours during the school week.\n\n"

        "5. LOCAL SOURCING\n"
        "We commit that 64% of annual spend under this contract will be sourced from growers whose "
        "primary production site lies within 250 miles of the District's central kitchen, as "
        "clarified by Addendum No. 1. Named growers include Sandoval Family Farms (Oxnard, 118 "
        "miles), Mesa Grande Growers (Coachella, 74 miles) and Ridgeline Organics (Bakersfield, "
        "162 miles). We will report actual local spend quarterly.\n\n"

        "6. QUALITY ASSURANCE\n"
        "Every inbound lot is inspected against USDA grade standards at our Fresno facility. We "
        "operate a documented HACCP plan covering receiving, cold storage and transport, last "
        "audited 2024-02-08 with zero non-conformances. Product is held at 34 to 38 degrees "
        "Fahrenheit and we log trailer temperature continuously with data retained 36 months.\n\n"

        "7. MWBE\n"
        "Valle Verde Produce Co. is a certified Women's Business Enterprise, California "
        "certification WBE-CA-22841, and 31% of our contract spend flows to MWBE-certified growers "
        "and service providers."
    ),
    "metadata": {
        "document_type": "bid_response", "category": "Produce",
        "bid_id": "BID-2024-089", "rfp_id": "RFP-2024-0412",
        "supplier_id": "SUP-1042", "supplier_name": "Valle Verde Produce Co.",
        "buyer_id": "ORG-RUSD", "bid_date": "2024-05-01",
        "total_bid_amount_usd": 1742880, "payment_terms": "Net-30",
        "status": "awarded", "contract_id": "CTR-2024-0412-VV",
        "entities": ["BID-2024-089", "RFP-2024-0412", "SUP-1042", "ORG-RUSD",
                     "PER-EMARQUEZ", "CTR-2024-0412-VV",
                     "VV-PRD-1180", "VV-PRD-1204", "VV-PRD-1311", "VV-PRD-1422",
                     "VV-PRD-1508", "VV-PRD-1560", "VV-PRD-1614"],
        "relations": [
            ["SUP-1042", "SUBMITTED", "BID-2024-089"],
            ["BID-2024-089", "RESPONDS_TO", "RFP-2024-0412"],
            ["BID-2024-089", "RESULTED_IN", "CTR-2024-0412-VV"],
            ["BID-2024-089", "QUOTES_PRODUCT", "VV-PRD-1180"],
            ["BID-2024-089", "QUOTES_PRODUCT", "VV-PRD-1204"],
            ["PER-EMARQUEZ", "WORKS_FOR", "SUP-1042"],
            ["PER-EMARQUEZ", "CONTACT_FOR", "BID-2024-089"],
            ["SUP-1042", "HOLDS_CERTIFICATION", "CERT-VV-SQF2"],
            ["SUP-1042", "HOLDS_CERTIFICATION", "CERT-VV-GAP"],
            ["SUP-1042", "OPERATES_FACILITY", "FAC-VV-FRESNO"],
        ],
    },
},

"bid-2024-091-acme": {
    "title": "BID-2024-091 — Acme Foods Distribution response to RFP-2024-0412",
    "content": (
        "PROPOSAL BID-2024-091\n"
        "Submitted by: Acme Foods Distribution (SUP-1077)\n"
        "In response to: RFP-2024-0412, Fresh Produce Supply, Riverside Unified School District\n"
        "Submitted: 2024-05-02, 4:55 PM PT. Addendum No. 1 acknowledged.\n"
        "Total extended annual basket: 1,689,450 USD\n"
        "Primary contact: Ray Okafor (PER-ROKAFOR), Director of K-12 Sales, "
        "r.okafor@acmefoods.example.com\n\n"

        "1. COMPANY\n"
        "Acme Foods Distribution is a broadline distributor headquartered in Sacramento, "
        "California, founded 1974, with 610 employees. We operate two distribution centres: "
        "Sacramento (210,000 square feet, SQF Level 2, certificate expires 2025-07-22) and "
        "Riverside (88,000 square feet, SQF Level 2 certificate submitted for renewal on "
        "2024-04-10, decision pending as of the proposal date). We hold GlobalG.A.P. certification "
        "through 2026-03-15. We serve 71 K-12 districts.\n\n"

        "2. PRICING\n"
        "All prices per case, delivered, firm-fixed for the contract year:\n"
        "  Romaine lettuce, 24 ct .............. 3.05 USD/case   SKU AF-P-4410\n"
        "  Spinach, baby, 4x2.5 lb ............. 3.95 USD/case   SKU AF-P-4418\n"
        "  Carrots, whole, 50 lb ............... 1.98 USD/case   SKU AF-P-4433\n"
        "  Brussels sprouts, 25 lb ............. 2.90 USD/case   SKU AF-P-4460\n"
        "  Bell peppers, mixed colour, 25 lb ... 3.42 USD/case   SKU AF-P-4471\n"
        "  Roma tomatoes, 25 lb ................ 3.60 USD/case   SKU AF-P-4488\n"
        "  Broccoli crowns, 20 lb .............. 4.55 USD/case   SKU AF-P-4501\n"
        "Our extended basket total of 1,689,450 USD is 53,430 USD below the next lowest proposal, "
        "a 3.1% saving to the District.\n\n"

        "3. COMMERCIAL TERMS\n"
        "Payment terms: Net-30. Early payment discount: 1% if paid within 15 days. Minimum order "
        "value: 150 USD per site per delivery. Volume discount: 4% on any single site delivery "
        "exceeding 600 USD.\n\n"

        "4. DELIVERY\n"
        "We commit to twice-weekly delivery on Mondays and Wednesdays. We note that our Riverside "
        "distribution centre currently services 71 districts from 22 vehicles and that the revised "
        "4:30 AM window in Addendum No. 1 requires a shift start change we are working to "
        "implement. We propose a 60-day transition period during which deliveries to 9 outlying "
        "sites would arrive between 8:30 AM and 9:30 AM, outside the specified window.\n\n"

        "5. LOCAL SOURCING\n"
        "We commit that 41% of annual spend will be sourced within 250 miles of the District's "
        "central kitchen. We source through 14 grower partners and can provide addresses on "
        "request.\n\n"

        "6. QUALITY ASSURANCE\n"
        "Inbound lots are inspected against USDA grade standards. Our HACCP plan was last audited "
        "2023-11-14. That audit recorded three minor non-conformances relating to cold storage "
        "door seals and receiving log completeness at the Riverside facility; two were closed "
        "2024-01-30 and one remains open pending a capital repair scheduled for July 2024. Trailer "
        "temperature is logged at 15-minute intervals with data retained 18 months.\n\n"

        "7. MWBE\n"
        "Acme Foods Distribution is not an MWBE-certified entity. 12% of contract spend is directed "
        "to MWBE-certified suppliers and service providers."
    ),
    "metadata": {
        "document_type": "bid_response", "category": "Produce",
        "bid_id": "BID-2024-091", "rfp_id": "RFP-2024-0412",
        "supplier_id": "SUP-1077", "supplier_name": "Acme Foods Distribution",
        "buyer_id": "ORG-RUSD", "bid_date": "2024-05-02",
        "total_bid_amount_usd": 1689450, "payment_terms": "Net-30",
        "status": "not_awarded",
        "entities": ["BID-2024-091", "RFP-2024-0412", "SUP-1077", "ORG-RUSD",
                     "PER-ROKAFOR", "AF-P-4410", "AF-P-4418", "AF-P-4433",
                     "FAC-AF-SAC", "FAC-AF-RIV"],
        "relations": [
            ["SUP-1077", "SUBMITTED", "BID-2024-091"],
            ["BID-2024-091", "RESPONDS_TO", "RFP-2024-0412"],
            ["BID-2024-091", "COMPETES_WITH", "BID-2024-089"],
            ["BID-2024-091", "QUOTES_PRODUCT", "AF-P-4410"],
            ["PER-ROKAFOR", "WORKS_FOR", "SUP-1077"],
            ["SUP-1077", "OPERATES_FACILITY", "FAC-AF-SAC"],
            ["SUP-1077", "OPERATES_FACILITY", "FAC-AF-RIV"],
        ],
    },
},

"bid-2024-104-globex": {
    "title": "BID-2024-104 — Globex Dairy Cooperative response to RFP-2024-0455",
    "content": (
        "PROPOSAL BID-2024-104\n"
        "Submitted by: Globex Dairy Cooperative (SUP-1103)\n"
        "In response to: RFP-2024-0455, Dairy and Refrigerated Products, Riverside USD\n"
        "Submitted: 2024-05-08, 11:20 AM PT\n"
        "Total extended annual basket: 2,311,600 USD\n"
        "Primary contact: Priya Raghunathan (PER-PRAGHUNATHAN), Director of Institutional Sales, "
        "p.raghunathan@globexdairy.example.coop\n\n"

        "1. COMPANY\n"
        "Globex Dairy Cooperative is a member-owned cooperative of 61 family dairy farms across "
        "the San Joaquin Valley, formed in 1961. We operate one processing plant in Visalia, "
        "California, certified SQF Level 3 (certificate expires 2026-02-28), and hold California "
        "Grade A dairy permit CA-DP-4471. We supply 44 school districts and 12 hospital systems.\n\n"

        "2. PRICING\n"
        "  Milk, 1% white, half-pint carton ......... 0.2450 USD/unit   SKU GX-DRY-0101\n"
        "  Milk, fat-free chocolate, half-pint ...... 0.2675 USD/unit   SKU GX-DRY-0104\n"
        "  Milk, whole, half-pint ................... 0.2610 USD/unit   SKU GX-DRY-0108\n"
        "  String cheese, part-skim, 1 oz ........... 0.1890 USD/unit   SKU GX-DRY-0240\n"
        "  Yogurt, low-fat strawberry, 4 oz cup ..... 0.3120 USD/unit   SKU GX-DRY-0312\n"
        "  Mozzarella, shredded, 5 lb bag ........... 11.40 USD/bag     SKU GX-DRY-0455\n"
        "  Cottage cheese, 5 lb tub ................. 9.75 USD/tub      SKU GX-DRY-0470\n"
        "Fluid milk line items are quoted as the announced Class I mover plus a fixed markup of "
        "0.0385 USD per half-pint. That markup is fixed for the contract term as required by "
        "Section 6 of the solicitation. All non-fluid items are firm-fixed for the contract year.\n\n"

        "3. COMMERCIAL TERMS\n"
        "Payment terms: Net-45 from receipt of a correct invoice. Early payment discount: 2% if "
        "paid within 10 days. Minimum order value: 200 USD per site per delivery. No volume "
        "discount is offered; cooperative pricing is already at member cost plus operating margin.\n\n"

        "4. COLD CHAIN\n"
        "This is the core of our proposal. Product is held at 34 to 38 degrees Fahrenheit "
        "continuously from the Visalia plant to the point of delivery. Every trailer carries two "
        "independent telemetry units logging at 5-minute intervals, with data transmitted in "
        "real time and retained for 36 months, exceeding the 24-month requirement. The District "
        "will be issued read-only portal credentials so temperature history for any delivery can "
        "be pulled without contacting us. We guarantee cold chain integrity: any load arriving "
        "above 41 degrees Fahrenheit is replaced in full at our cost within six hours, and we do "
        "not invoice the rejected load.\n\n"

        "5. DELIVERY\n"
        "Delivery three times weekly to all 42 sites on Mondays, Wednesdays and Fridays, within a "
        "four-hour window. We operate 28 refrigerated vehicles from Visalia and will dedicate 9 to "
        "this contract. Emergency replacement within six hours as required.\n\n"

        "6. SUSTAINABILITY AND PACKAGING\n"
        "All half-pint fluid milk is supplied in paper-based gable-top cartons, recyclable in "
        "Riverside County's programme. We operate a crate return programme at no charge: crates "
        "are collected on the following scheduled delivery, which removes approximately 51 tons of "
        "plastic from the District's waste stream annually. This directly supports Board "
        "resolution 2023-14.\n\n"

        "7. QUALITY\n"
        "Our HACCP plan covers receiving, pasteurisation, packaging, cold storage and transport, "
        "last audited 2024-03-05 with zero non-conformances. We conduct finished-product microbial "
        "testing on every production lot and retain samples for 21 days past code date."
    ),
    "metadata": {
        "document_type": "bid_response", "category": "Dairy",
        "bid_id": "BID-2024-104", "rfp_id": "RFP-2024-0455",
        "supplier_id": "SUP-1103", "supplier_name": "Globex Dairy Cooperative",
        "buyer_id": "ORG-RUSD", "bid_date": "2024-05-08",
        "total_bid_amount_usd": 2311600, "payment_terms": "Net-45",
        "early_payment_discount": "2% net 10", "status": "awarded",
        "entities": ["BID-2024-104", "RFP-2024-0455", "SUP-1103", "ORG-RUSD",
                     "PER-PRAGHUNATHAN", "GX-DRY-0101", "GX-DRY-0104", "GX-DRY-0240",
                     "GX-DRY-0312", "FAC-GX-VISALIA", "CERT-GX-SQF3"],
        "relations": [
            ["SUP-1103", "SUBMITTED", "BID-2024-104"],
            ["BID-2024-104", "RESPONDS_TO", "RFP-2024-0455"],
            ["BID-2024-104", "QUOTES_PRODUCT", "GX-DRY-0101"],
            ["BID-2024-104", "QUOTES_PRODUCT", "GX-DRY-0312"],
            ["PER-PRAGHUNATHAN", "WORKS_FOR", "SUP-1103"],
            ["SUP-1103", "OPERATES_FACILITY", "FAC-GX-VISALIA"],
            ["SUP-1103", "HOLDS_CERTIFICATION", "CERT-GX-SQF3"],
        ],
    },
},

"bid-2024-117-tidewater": {
    "title": "BID-2024-117 — Tidewater Frozen Holdings response to RFP-2024-0489 (Lots 1 and 2)",
    "content": (
        "PROPOSAL BID-2024-117\n"
        "Submitted by: Tidewater Frozen Holdings (SUP-1156)\n"
        "In response to: RFP-2024-0489, Frozen Foods, Protein and Seafood, Riverside USD\n"
        "Lots bid: Lot 1 (frozen prepared, vegetables, bakery) and Lot 2 (frozen protein and "
        "seafood). We do not bid Lot 3.\n"
        "Submitted: 2024-05-29, 1:05 PM PT\n"
        "Total extended annual basket, Lots 1 and 2 combined: 2,984,220 USD\n"
        "Primary contact: Gregory Halloran (PER-GHALLORAN), SVP National Accounts, "
        "g.halloran@tidewaterfrozen.example.com\n\n"

        "1. COMPANY\n"
        "Tidewater Frozen Holdings was founded in 1987 and is headquartered in Stockton, "
        "California. We recorded 412 million USD in revenue in FY2024 and employ 1,180 people. "
        "Our categories are Frozen Foods, Dry Groceries and Meat & Poultry. We operate two "
        "distribution centres: Stockton (285,000 square feet) and Bakersfield (140,000 square "
        "feet), and a fleet of 96 refrigerated vehicles. We serve 214 K-12 districts.\n\n"

        "2. SEAFOOD FULFILMENT AFFILIATE\n"
        "As required by Section 3 of the solicitation, we name Harbor Point Seafood (SUP-1157) as "
        "the affiliate fulfilling all seafood line items under Lot 2. Harbor Point Seafood has "
        "been a wholly owned subsidiary of Tidewater Frozen Holdings since our acquisition of the "
        "business on 2021-06-30 for 34 million USD. Harbor Point holds Marine Stewardship Council "
        "Chain of Custody certification MSC-C-58817, valid through 2026-01-31, and operates under "
        "an independent seafood HACCP plan.\n\n"

        "3. PRICING, SELECTED LINE ITEMS\n"
        "  Beef patty, 100% beef, 2.0 oz, CN ....... 0.4180 USD/unit   SKU TW-PRO-2210\n"
        "  Chicken nuggets, WG breaded, 5 ct ....... 0.3925 USD/unit   SKU TW-PRO-2244\n"
        "  Pollock fillet, MSC, 3.6 oz ............. 0.8140 USD/unit   SKU HP-SEA-7702\n"
        "  Cheese pizza, WG crust, 4x6 ............. 0.6350 USD/unit   SKU TW-FRZ-3105\n"
        "  Green beans, IQF, 20 lb ................. 18.90 USD/case    SKU TW-FRZ-3320\n"
        "  Sweet corn, IQF, 20 lb .................. 16.75 USD/case    SKU TW-FRZ-3324\n"
        "  Dinner roll, WG, 1 oz ................... 0.1140 USD/unit   SKU TW-FRZ-3488\n"
        "A further 214 line items are priced in Attachment D.\n\n"

        "4. COMMERCIAL TERMS\n"
        "Payment terms: Net-45. Early payment discount: 1.5% if paid within 15 days. Minimum "
        "order value: 400 USD per delivery. Volume discount: 3% on monthly spend exceeding "
        "200,000 USD, applied as a rebate credited quarterly.\n\n"

        "5. USDA FOODS PROCESSING\n"
        "We accept diverted USDA Foods entitlement beef and poultry at both our Stockton facility "
        "and through our processor network. Entitlement value is tracked per district and appears "
        "as a separate commodity offset line on every invoice. In SY 2023-24 we processed "
        "1.9 million pounds of diverted commodity across our K-12 accounts.\n\n"

        "6. FOOD SAFETY AND TRACEABILITY\n"
        "Both distribution centres are certified SQF Level 2. The Stockton certificate expires "
        "2025-04-30 and renewal audit is scheduled for 2025-03-11. The Bakersfield certificate "
        "expires 2025-10-08. We additionally hold BRCGS AA grade at Stockton through 2025-08-19. "
        "Lot traceability is available within two hours through our warehouse management system, "
        "exceeding the four-hour requirement.\n"
        "In the interest of full disclosure, our Stockton facility's September 2024 third-party "
        "audit recorded two minor non-conformances, relating to a freezer door seal and a gap in "
        "manual temperature logging. Both were closed on 2024-10-11 and verified by the auditor.\n\n"

        "7. DELIVERY\n"
        "Weekly delivery for frozen product to all 42 sites, arriving at 0 degrees Fahrenheit or "
        "below. We will accept 45-minute scheduled dock slots at the central kitchen. We note that "
        "our current on-time performance across our K-12 book is 93.4% and we are investing in "
        "route optimisation to raise this above 95% during SY 2024-25."
    ),
    "metadata": {
        "document_type": "bid_response", "category": "Frozen Foods",
        "bid_id": "BID-2024-117", "rfp_id": "RFP-2024-0489",
        "supplier_id": "SUP-1156", "supplier_name": "Tidewater Frozen Holdings",
        "buyer_id": "ORG-RUSD", "bid_date": "2024-05-29",
        "total_bid_amount_usd": 2984220, "payment_terms": "Net-45",
        "early_payment_discount": "1.5% net 15", "status": "awarded",
        "contract_id": "CTR-2024-0489-TFH", "lots": ["LOT-0489-1", "LOT-0489-2"],
        "entities": ["BID-2024-117", "RFP-2024-0489", "SUP-1156", "SUP-1157", "ORG-RUSD",
                     "PER-GHALLORAN", "CTR-2024-0489-TFH", "TW-PRO-2210", "TW-PRO-2244",
                     "HP-SEA-7702", "TW-FRZ-3105", "FAC-TW-STOCKTON", "FAC-TW-BAKERSFIELD",
                     "CERT-TW-SQF2-STK", "CERT-HP-MSC"],
        "relations": [
            ["SUP-1156", "SUBMITTED", "BID-2024-117"],
            ["BID-2024-117", "RESPONDS_TO", "RFP-2024-0489"],
            ["BID-2024-117", "BIDS_LOT", "LOT-0489-1"],
            ["BID-2024-117", "BIDS_LOT", "LOT-0489-2"],
            ["BID-2024-117", "RESULTED_IN", "CTR-2024-0489-TFH"],
            ["SUP-1157", "SUBSIDIARY_OF", "SUP-1156"],
            ["SUP-1157", "FULFILS_CATEGORY_FOR", "BID-2024-117"],
            ["SUP-1157", "HOLDS_CERTIFICATION", "CERT-HP-MSC"],
            ["SUP-1156", "OPERATES_FACILITY", "FAC-TW-STOCKTON"],
            ["SUP-1156", "OPERATES_FACILITY", "FAC-TW-BAKERSFIELD"],
            ["PER-GHALLORAN", "WORKS_FOR", "SUP-1156"],
        ],
    },
},

"bid-2024-118-northlake": {
    "title": "BID-2024-118 — Northlake Protein Partners response to RFP-2024-0489 (Lots 2 and 3)",
    "content": (
        "PROPOSAL BID-2024-118\n"
        "Submitted by: Northlake Protein Partners (SUP-1189)\n"
        "In response to: RFP-2024-0489, Frozen Foods, Protein and Seafood, Riverside USD\n"
        "Lots bid: Lot 2 (frozen protein and seafood) and Lot 3 (fresh and frozen poultry). "
        "We do not bid Lot 1.\n"
        "Submitted: 2024-05-30, 9:15 AM PT\n"
        "Total extended annual basket, Lots 2 and 3 combined: 3,102,700 USD\n"
        "Primary contact: Sandra Kelley (PER-SKELLEY), Director of Education Sales, "
        "s.kelley@northlakeprotein.example.com\n\n"

        "1. COMPANY\n"
        "Northlake Protein Partners is a protein specialist founded in 2004, headquartered in "
        "Modesto, California, with 430 employees. We operate USDA establishment EST. 18442 in "
        "Modesto, a further-processing facility certified SQF Level 2 through 2025-12-05, and a "
        "cold storage facility in Ontario, California. We do not operate a broadline frozen "
        "business, which is why we bid Lots 2 and 3 only.\n\n"

        "2. PRICING, SELECTED LINE ITEMS\n"
        "  Chicken drumstick, fresh, bulk .......... 1.2450 USD/lb     SKU NL-PLT-5010\n"
        "  Chicken thigh, boneless, fresh .......... 2.1850 USD/lb     SKU NL-PLT-5024\n"
        "  Whole grain breaded chicken patty, 3 oz . 0.4310 USD/unit   SKU NL-PLT-5150\n"
        "  Turkey taco filling, cooked, 5 lb ....... 14.20 USD/bag     SKU NL-PRO-5402\n"
        "  Beef crumble, cooked, 5 lb .............. 17.85 USD/bag     SKU NL-PRO-5418\n"
        "  Pollock fillet, 3.6 oz, non-MSC ......... 0.7480 USD/unit   SKU NL-SEA-5900\n"
        "Our poultry pricing under Lot 3 is 6.4% below the District's SY 2023-24 contracted "
        "average and reflects direct grower relationships rather than brokered supply.\n\n"

        "3. COMMERCIAL TERMS\n"
        "Payment terms: Net-30. Early payment discount: 1% if paid within 10 days. Minimum order "
        "value: 350 USD per delivery. No volume discount offered.\n\n"

        "4. FOOD SAFETY AND TRACEABILITY\n"
        "Our Modesto facility operates under continuous USDA FSIS inspection as EST. 18442. Our "
        "HACCP plan covers receiving, further processing, freezing, cold storage and transport, "
        "last audited 2024-04-18 with one minor non-conformance concerning label verification, "
        "closed 2024-05-02. Lot traceability is available within 90 minutes.\n"
        "We note that our pollock offering under Lot 2 is not Marine Stewardship Council "
        "certified. We are able to source an MSC-certified equivalent at 0.8320 USD per unit if "
        "the District requires MSC Chain of Custody as a mandatory condition of award for seafood "
        "line items.\n\n"

        "5. DELIVERY\n"
        "Twice-weekly delivery of fresh poultry to all 42 sites at 28 to 34 degrees Fahrenheit, "
        "and weekly delivery of frozen protein at 0 degrees Fahrenheit or below. We operate 41 "
        "refrigerated vehicles and will dedicate 11 to this contract. Our current on-time "
        "performance across our education book is 95.8%.\n\n"

        "6. USDA FOODS PROCESSING\n"
        "We are an approved USDA Foods processor for beef and poultry and hold current end-product "
        "data schedules for 34 items. Entitlement drawdown is reported monthly and reflected as a "
        "commodity offset line on each invoice.\n\n"

        "7. PRODUCT RANGE\n"
        "We offer 96 line items across Lots 2 and 3. We acknowledge that this is narrower than a "
        "broadline competitor and we do not offer frozen bakery, frozen vegetables or frozen "
        "prepared entrées, which is the basis of our decision not to bid Lot 1."
    ),
    "metadata": {
        "document_type": "bid_response", "category": "Meat & Poultry",
        "bid_id": "BID-2024-118", "rfp_id": "RFP-2024-0489",
        "supplier_id": "SUP-1189", "supplier_name": "Northlake Protein Partners",
        "buyer_id": "ORG-RUSD", "bid_date": "2024-05-30",
        "total_bid_amount_usd": 3102700, "payment_terms": "Net-30",
        "status": "partially_awarded", "awarded_lots": ["LOT-0489-3"],
        "entities": ["BID-2024-118", "RFP-2024-0489", "SUP-1189", "ORG-RUSD",
                     "PER-SKELLEY", "NL-PLT-5010", "NL-PLT-5150", "NL-SEA-5900",
                     "FAC-NL-MODESTO", "CERT-NL-SQF2"],
        "relations": [
            ["SUP-1189", "SUBMITTED", "BID-2024-118"],
            ["BID-2024-118", "RESPONDS_TO", "RFP-2024-0489"],
            ["BID-2024-118", "BIDS_LOT", "LOT-0489-2"],
            ["BID-2024-118", "BIDS_LOT", "LOT-0489-3"],
            ["BID-2024-118", "COMPETES_WITH", "BID-2024-117"],
            ["BID-2024-118", "QUOTES_PRODUCT", "NL-PLT-5010"],
            ["PER-SKELLEY", "WORKS_FOR", "SUP-1189"],
            ["SUP-1189", "OPERATES_FACILITY", "FAC-NL-MODESTO"],
            ["SUP-1189", "HOLDS_CERTIFICATION", "CERT-NL-SQF2"],
        ],
    },
},

# ─────────────────────────── AWARD, CONTRACTS ───────────────────────────
"award-memo-rfp-0412": {
    "title": "Award Recommendation Memorandum — RFP-2024-0412 Fresh Produce Supply",
    "content": (
        "MEMORANDUM\n"
        "To: Board of Education, Riverside Unified School District\n"
        "From: Dana Whitfield (PER-DWHITFIELD), Director of Nutrition Services\n"
        "Prepared by: Miguel Arredondo (PER-MARREDONDO), Procurement Manager\n"
        "Date: 2024-05-20\n"
        "Subject: Award recommendation, RFP-2024-0412, Fresh Produce Supply, SY 2024-25\n\n"

        "1. RECOMMENDATION\n"
        "Staff recommend award of RFP-2024-0412 to Valle Verde Produce Co. (SUP-1042) under "
        "proposal BID-2024-089, in the amount of 1,742,880 USD for the initial contract year "
        "beginning 2024-08-01, with two optional one-year renewals. The resulting agreement is "
        "designated CTR-2024-0412-VV.\n\n"

        "2. PROPOSALS RECEIVED\n"
        "Two responsive proposals were received by the extended deadline of 2024-05-03:\n"
        "  BID-2024-089, Valle Verde Produce Co. (SUP-1042), 1,742,880 USD\n"
        "  BID-2024-091, Acme Foods Distribution (SUP-1077), 1,689,450 USD\n"
        "Four additional suppliers were solicited and did not respond.\n\n"

        "3. EVALUATION SCORES\n"
        "The evaluation committee comprised the Nutrition Services Director, the Procurement "
        "Manager, two site kitchen managers and a District food safety specialist. Scores are out "
        "of 100 on the weightings published in Section 3 of the solicitation.\n\n"
        "  Criterion                        Weight   Valle Verde   Acme Foods\n"
        "  Price                             40        36.8          40.0\n"
        "  Quality assurance & food safety   25        23.5          14.2\n"
        "  Delivery capability               20        18.6          13.5\n"
        "  Local sourcing                    10         8.5           5.4\n"
        "  MWBE participation                 5         4.0           1.6\n"
        "  TOTAL                            100        91.4          74.7\n\n"

        "4. BASIS OF RECOMMENDATION\n"
        "Acme Foods submitted the lower extended basket total, 53,430 USD below Valle Verde, and "
        "correctly received the full 40 price points. The recommendation nevertheless favours "
        "Valle Verde on three grounds.\n\n"
        "First, food safety. Section 4 of the solicitation requires a current SQF Level 2 "
        "certificate for every distribution centre servicing the contract on the proposal due "
        "date. Acme Foods' Riverside distribution centre certificate was in renewal and not "
        "current on 2024-05-03. Addendum No. 1, answer A2, stated explicitly that a certificate in "
        "renewal is not a current certificate for the purposes of this solicitation. Acme Foods' "
        "November 2023 audit also left one non-conformance open pending a capital repair. Acme "
        "scored 14.2 of 25 available points on this criterion, below the 15-point floor set in "
        "Section 3, which independently bars award on price advantage.\n\n"
        "Second, delivery. Acme Foods proposed a 60-day transition during which 9 outlying sites "
        "would receive deliveries between 8:30 AM and 9:30 AM, outside the window established by "
        "Addendum No. 1. Two committee members who manage site kitchens rated this unworkable "
        "against a 7:15 AM meal service.\n\n"
        "Third, local sourcing. Valle Verde committed 64% of annual spend within the 250-mile "
        "radius against Acme Foods' 41%, with named growers and quarterly reporting.\n\n"

        "5. FINANCIAL IMPACT\n"
        "The recommended award is 107,120 USD below the 1,850,000 USD estimated annual value in "
        "the solicitation. Funding is from the Cafeteria Special Revenue Fund. Cedar Valley "
        "Unified School District (ORG-CVUSD) has indicated intent to piggyback under the Central "
        "Valley Purchasing Cooperative at identical pricing, which does not alter the District's "
        "financial exposure.\n\n"

        "6. PROTEST\n"
        "No protest was filed within the five business day window closing 2024-05-28."
    ),
    "metadata": {
        "document_type": "award_memo", "category": "Produce",
        "rfp_id": "RFP-2024-0412", "buyer_id": "ORG-RUSD",
        "awarded_bid_id": "BID-2024-089", "awarded_supplier_id": "SUP-1042",
        "contract_id": "CTR-2024-0412-VV", "issue_date": "2024-05-20",
        "award_amount_usd": 1742880, "status": "approved",
        "entities": ["RFP-2024-0412", "BID-2024-089", "BID-2024-091", "SUP-1042", "SUP-1077",
                     "ORG-RUSD", "ORG-CVUSD", "CTR-2024-0412-VV",
                     "PER-DWHITFIELD", "PER-MARREDONDO"],
        "relations": [
            ["ORG-RUSD", "AWARDED", "BID-2024-089"],
            ["BID-2024-089", "SCORED", "91.4"],
            ["BID-2024-091", "SCORED", "74.7"],
            ["BID-2024-091", "LOST_TO", "BID-2024-089"],
            ["SUP-1042", "PARTY_TO", "CTR-2024-0412-VV"],
            ["ORG-RUSD", "PARTY_TO", "CTR-2024-0412-VV"],
            ["PER-DWHITFIELD", "WORKS_FOR", "ORG-RUSD"],
            ["PER-DWHITFIELD", "RECOMMENDED", "BID-2024-089"],
        ],
    },
},

"contract-ctr-2024-0412-vv": {
    "title": "Contract CTR-2024-0412-VV — Riverside USD and Valle Verde Produce Co.",
    "content": (
        "AGREEMENT FOR THE SUPPLY OF FRESH PRODUCE\n"
        "Contract number: CTR-2024-0412-VV\n"
        "Between: Riverside Unified School District (ORG-RUSD), 'the District'\n"
        "And: Valle Verde Produce Co. (SUP-1042), 'the Supplier'\n"
        "Arising from: RFP-2024-0412 and proposal BID-2024-089\n"
        "Effective: 2024-08-01. Initial term ends 2025-07-31. Two optional one-year renewals.\n"
        "Initial term value: 1,742,880 USD\n\n"

        "ARTICLE 1 — SCOPE\n"
        "The Supplier shall furnish all fresh fruits and vegetables listed in Attachment C to the "
        "42 school sites listed in Attachment A, at the unit prices set out in BID-2024-089, which "
        "are incorporated by reference and are firm-fixed for the initial term.\n\n"

        "ARTICLE 2 — PAYMENT\n"
        "Payment terms are Net-30 from the District's receipt of a correct invoice. The Supplier "
        "offers no early payment discount. The volume discount of 5% applies at the invoice level "
        "to any single site delivery exceeding 500 USD and shall be shown as a discount line on "
        "the invoice, not applied silently to unit prices. Minimum order value is 100 USD per site "
        "per delivery.\n\n"

        "ARTICLE 3 — DELIVERY\n"
        "Delivery shall occur twice weekly to every site, on Tuesdays and Thursdays, within the "
        "window 4:30 AM to 8:30 AM. Temperature-controlled product shall arrive between 34 and 41 "
        "degrees Fahrenheit. A delivery arriving outside the window without 24 hours prior written "
        "notice is recorded as a late delivery.\n\n"
        "ARTICLE 4 — PERFORMANCE STANDARDS AND LIQUIDATED DAMAGES\n"
        "4.1 The Supplier shall maintain a fill rate of not less than 97%, measured monthly by "
        "line item.\n"
        "4.2 The Supplier shall maintain an on-time delivery rate of not less than 95%, measured "
        "monthly by delivery event.\n"
        "4.3 Quality rejection rate shall remain below 2.0% of cases delivered, measured monthly.\n"
        "4.4 Where monthly on-time delivery falls below 92%, the District may assess liquidated "
        "damages of 2% of that month's invoiced value for each full percentage point below 92%, to "
        "a maximum of 10% of the monthly invoiced value. The parties agree this is a reasonable "
        "estimate of the District's costs of menu substitution and emergency procurement and is "
        "not a penalty.\n"
        "4.5 Two consecutive months below 92% on-time obliges the Supplier to submit a written "
        "corrective action plan within 10 business days.\n\n"

        "ARTICLE 5 — PRICE ADJUSTMENT\n"
        "No price adjustment is permitted during the initial term. At each renewal the Supplier "
        "may request adjustment once, capped at the lower of the twelve-month change in CPI-U "
        "food-at-home or 4%. Requests must be submitted no later than 60 days before the renewal "
        "date with supporting documentation.\n\n"

        "ARTICLE 6 — SUBSTITUTIONS\n"
        "No substitution may be delivered without prior written approval from the District's "
        "Nutrition Services Director (PER-DWHITFIELD). An unapproved substitution may be rejected "
        "at the Supplier's cost and is recorded as an unfilled line item.\n\n"

        "ARTICLE 7 — CERTIFICATIONS\n"
        "The Supplier shall maintain, for the full term, current GlobalG.A.P. certification and "
        "SQF Level 2 or higher certification for every facility servicing this contract, and shall "
        "notify the District in writing within 5 business days of any lapse, suspension or adverse "
        "audit finding. Failure to maintain certification is a material breach.\n\n"

        "ARTICLE 8 — LOCAL SOURCING\n"
        "The Supplier shall source not less than 64% of annual contract spend from growers whose "
        "primary production site lies within 250 miles of the District's central kitchen, and "
        "shall report actual local spend quarterly with grower addresses.\n\n"

        "ARTICLE 9 — TERMINATION\n"
        "The District may terminate for convenience on 60 days written notice, and for cause on "
        "30 days written notice where a material breach is not cured within that period. Loss of "
        "required certification, or three consecutive months below 92% on-time delivery, each "
        "constitute cause.\n\n"

        "ARTICLE 10 — COOPERATIVE PURCHASING\n"
        "Cedar Valley Unified School District (ORG-CVUSD) may purchase under this agreement "
        "through the Central Valley Purchasing Cooperative at identical unit prices and terms. Any "
        "such purchase is a separate obligation between the Supplier and ORG-CVUSD, and Riverside "
        "Unified School District bears no liability for it."
    ),
    "metadata": {
        "document_type": "contract", "category": "Produce",
        "contract_id": "CTR-2024-0412-VV", "rfp_id": "RFP-2024-0412",
        "bid_id": "BID-2024-089", "supplier_id": "SUP-1042",
        "supplier_name": "Valle Verde Produce Co.", "buyer_id": "ORG-RUSD",
        "effective_date": "2024-08-01", "expiry_date": "2025-07-31",
        "contract_value_usd": 1742880, "payment_terms": "Net-30", "status": "active",
        "entities": ["CTR-2024-0412-VV", "RFP-2024-0412", "BID-2024-089", "SUP-1042",
                     "ORG-RUSD", "ORG-CVUSD", "PER-DWHITFIELD"],
        "relations": [
            ["CTR-2024-0412-VV", "ARISES_FROM", "BID-2024-089"],
            ["CTR-2024-0412-VV", "ARISES_FROM", "RFP-2024-0412"],
            ["SUP-1042", "PARTY_TO", "CTR-2024-0412-VV"],
            ["ORG-RUSD", "PARTY_TO", "CTR-2024-0412-VV"],
            ["ORG-CVUSD", "MAY_PURCHASE_UNDER", "CTR-2024-0412-VV"],
            ["CTR-2024-0412-VV", "REQUIRES_CERTIFICATION", "CERT-VV-GAP"],
            ["CTR-2024-0412-VV", "REQUIRES_CERTIFICATION", "CERT-VV-SQF2"],
        ],
    },
},

"contract-ctr-2024-0489-tfh": {
    "title": "Contract CTR-2024-0489-TFH — Riverside USD and Tidewater Frozen Holdings",
    "content": (
        "AGREEMENT FOR THE SUPPLY OF FROZEN FOODS AND PROTEIN\n"
        "Contract number: CTR-2024-0489-TFH\n"
        "Between: Riverside Unified School District (ORG-RUSD), 'the District'\n"
        "And: Tidewater Frozen Holdings (SUP-1156), 'the Supplier'\n"
        "Arising from: RFP-2024-0489 and proposal BID-2024-117, Lots 1 and 2\n"
        "Effective: 2024-08-15. Initial term ends 2026-08-14, a 24-month term.\n"
        "Annual value: 2,984,220 USD\n\n"

        "ARTICLE 1 — SCOPE AND LOTS\n"
        "This agreement covers Lot 1, frozen prepared foods, frozen vegetables and frozen bakery, "
        "and Lot 2, frozen and further-processed protein including seafood. Lot 3, fresh and "
        "frozen poultry, was awarded separately to Northlake Protein Partners (SUP-1189) under "
        "contract CTR-2024-0489-NPP and is not covered here.\n\n"

        "ARTICLE 2 — AFFILIATE PERFORMANCE\n"
        "2.1 The Supplier has named Harbor Point Seafood (SUP-1157), its wholly owned subsidiary, "
        "as the fulfilment affiliate for all seafood line items under Lot 2.\n"
        "2.2 The Supplier remains fully and solely liable to the District for the performance of "
        "Harbor Point Seafood. The District will not accept a defence based on the acts or "
        "omissions of the affiliate.\n"
        "2.3 Harbor Point Seafood shall maintain Marine Stewardship Council Chain of Custody "
        "certification MSC-C-58817 for the full term. Lapse of that certification suspends the "
        "Supplier's right to deliver seafood line items until it is restored.\n\n"

        "ARTICLE 3 — PAYMENT\n"
        "Payment terms are Net-45 from receipt of a correct invoice. The District may take an "
        "early payment discount of 1.5% where payment issues within 15 days. The volume rebate of "
        "3% on monthly spend exceeding 200,000 USD is credited quarterly in arrears and shall be "
        "reconciled against actual invoiced spend. Minimum order value is 400 USD per delivery.\n\n"

        "ARTICLE 4 — DELIVERY AND TEMPERATURE\n"
        "Weekly delivery of frozen product to all 42 sites, arriving at 0 degrees Fahrenheit or "
        "below. Deliveries to the central kitchen shall use scheduled 45-minute dock slots. A load "
        "arriving above 10 degrees Fahrenheit is rejected in full at the Supplier's cost.\n\n"

        "ARTICLE 5 — PERFORMANCE STANDARDS\n"
        "5.1 Fill rate not less than 96%, measured monthly by line item.\n"
        "5.2 On-time delivery not less than 95%, measured monthly by delivery event.\n"
        "5.3 Quality rejection rate below 1.5% of cases delivered.\n"
        "5.4 A supplier falling below 92% on-time in any two consecutive months shall submit a "
        "corrective action plan within 10 business days. Failure below 90% in any single month "
        "entitles the District to withhold the quarterly volume rebate for that quarter.\n\n"

        "ARTICLE 6 — USDA FOODS\n"
        "The Supplier shall accept diverted USDA Foods entitlement beef and poultry, track "
        "entitlement value by district, and present the commodity offset as a separate line on "
        "every invoice. Unused entitlement value shall be reported to the District no later than "
        "45 days before the end of each entitlement year.\n\n"

        "ARTICLE 7 — TRACEABILITY\n"
        "The Supplier shall trace any delivered lot to its production facility and production date "
        "within four hours of a District request, and within two hours where the request arises "
        "from a suspected food safety incident or a public health enquiry.\n\n"

        "ARTICLE 8 — CERTIFICATIONS\n"
        "The Supplier shall maintain SQF Level 2 or higher for every facility touching product "
        "under this contract. The District notes that the Stockton facility certificate expires "
        "2025-04-30, within the contract term, and requires written evidence of renewal no later "
        "than 2025-05-14. Failure to provide evidence within that period suspends new orders until "
        "cured.\n\n"

        "ARTICLE 9 — PRICE ADJUSTMENT\n"
        "Prices are firm-fixed for the first 12 months. One adjustment is permitted effective "
        "2025-08-15, capped at the lower of the twelve-month change in the Producer Price Index "
        "for processed foods and feeds or 5%.\n\n"

        "ARTICLE 10 — TERMINATION\n"
        "Termination for convenience on 90 days written notice, reflecting the longer term and the "
        "Supplier's commodity processing commitments. Termination for cause on 30 days notice "
        "where a material breach is not cured. Loss of SQF certification at any facility servicing "
        "this contract is a material breach."
    ),
    "metadata": {
        "document_type": "contract", "category": "Frozen Foods",
        "contract_id": "CTR-2024-0489-TFH", "rfp_id": "RFP-2024-0489",
        "bid_id": "BID-2024-117", "supplier_id": "SUP-1156",
        "supplier_name": "Tidewater Frozen Holdings", "buyer_id": "ORG-RUSD",
        "effective_date": "2024-08-15", "expiry_date": "2026-08-14",
        "contract_value_usd": 2984220, "payment_terms": "Net-45", "status": "active",
        "entities": ["CTR-2024-0489-TFH", "RFP-2024-0489", "BID-2024-117", "SUP-1156",
                     "SUP-1157", "SUP-1189", "ORG-RUSD", "CTR-2024-0489-NPP",
                     "CERT-HP-MSC", "CERT-TW-SQF2-STK", "FAC-TW-STOCKTON"],
        "relations": [
            ["CTR-2024-0489-TFH", "ARISES_FROM", "BID-2024-117"],
            ["SUP-1156", "PARTY_TO", "CTR-2024-0489-TFH"],
            ["ORG-RUSD", "PARTY_TO", "CTR-2024-0489-TFH"],
            ["SUP-1157", "FULFILS_CATEGORY_UNDER", "CTR-2024-0489-TFH"],
            ["SUP-1156", "LIABLE_FOR", "SUP-1157"],
            ["SUP-1189", "PARTY_TO", "CTR-2024-0489-NPP"],
            ["CTR-2024-0489-TFH", "REQUIRES_CERTIFICATION", "CERT-HP-MSC"],
            ["CTR-2024-0489-TFH", "REQUIRES_CERTIFICATION", "CERT-TW-SQF2-STK"],
        ],
    },
},

# ─────────────────────── PERFORMANCE, PROFILE, COMPLIANCE ───────────────────────
"perf-review-q1-fy2025": {
    "title": "Q1 FY2025 Supplier Performance Review — Riverside USD Nutrition Services",
    "content": (
        "QUARTERLY SUPPLIER PERFORMANCE REVIEW\n"
        "Period: Q1 FY2025, 2024-09-01 through 2024-11-30\n"
        "Prepared by: Dana Whitfield (PER-DWHITFIELD), Director of Nutrition Services, "
        "Riverside Unified School District (ORG-RUSD)\n"
        "Issued: 2024-12-05\n\n"

        "1. SUMMARY\n"
        "Overall district satisfaction with nutrition services suppliers was 8.5 out of 10 for the "
        "quarter, measured by the District's site manager survey across 42 sites, up from 8.1 in "
        "Q4 FY2024. This is an aggregate district-level index and should not be confused with the "
        "individual supplier satisfaction ratings in Section 3, which are scored out of 5.\n\n"

        "2. SCORECARD\n"
        "  Supplier                       On-time   Fill rate   Quality rej.   Satisfaction\n"
        "  Valle Verde Produce Co.          96.2%      97.8%         0.9%          4.8/5\n"
        "  Globex Dairy Cooperative         98.4%      99.1%         0.4%          4.6/5\n"
        "  Summit Beverage Distributors     97.1%      96.9%         0.7%          4.5/5\n"
        "  Northlake Protein Partners       94.7%      96.0%         1.1%          4.3/5\n"
        "  Tidewater Frozen Holdings        92.0%      95.3%         1.7%          4.1/5\n\n"

        "3. SUPPLIER COMMENTARY\n\n"
        "3.1 Valle Verde Produce Co. (SUP-1042), contract CTR-2024-0412-VV. Customer satisfaction "
        "4.8 out of 5 stars, the highest of any supplier this quarter. On-time delivery 96.2% "
        "against a 95% contractual minimum. Fill rate 97.8% against a 97% minimum. Quality "
        "rejections 0.9% against a 2.0% ceiling. Credit memos totalled 4,120 USD, almost entirely "
        "from a single incident on 2024-10-17 when a romaine lettuce lot (SKU VV-PRD-1180) failed "
        "site-level inspection at nine sites. Valle Verde replaced the product within four hours "
        "and issued credit without dispute. Local sourcing reported at 66.1% against a 64% "
        "contractual commitment. No contractual remedies were triggered.\n\n"
        "3.2 Globex Dairy Cooperative (SUP-1103). The strongest operational performer of the "
        "quarter: on-time 98.4%, fill rate 99.1%, quality rejections 0.4%. Satisfaction 4.6 out of "
        "5. Zero loads rejected on temperature. The crate return programme removed an estimated "
        "12.4 tons of plastic during the quarter. Two site managers noted that the Friday delivery "
        "occasionally arrives at the end of the four-hour window, which compresses receiving.\n\n"
        "3.3 Summit Beverage Distributors (SUP-1204). On-time 97.1%, fill rate 96.9%, satisfaction "
        "4.5 out of 5. Performance is steady. No issues escalated.\n\n"
        "3.4 Northlake Protein Partners (SUP-1189), contract CTR-2024-0489-NPP, Lot 3 poultry. "
        "On-time 94.7%, marginally below the 95% contractual minimum, driven by three late "
        "deliveries in the week of 2024-10-28 attributed to a vehicle breakdown. Fill rate 96.0%. "
        "Satisfaction 4.3 out of 5. A written notice of the on-time shortfall was issued "
        "2024-12-05. No corrective action plan is required as the threshold for that remedy is two "
        "consecutive months below 92%.\n\n"
        "3.5 Tidewater Frozen Holdings (SUP-1156), contract CTR-2024-0489-TFH, Lots 1 and 2. The "
        "weakest performer this quarter and the only supplier requiring formal action. On-time "
        "delivery 92.0%, the lowest of any District supplier, against a 95% contractual minimum. "
        "Fill rate 95.3% against a 96% minimum. Quality rejections 1.7% against a 1.5% ceiling. "
        "Satisfaction 4.1 out of 5, the lowest recorded. October and November both fell below 92% "
        "on-time (91.6% and 91.8% respectively when measured monthly), which triggers Article 5.4 "
        "of CTR-2024-0489-TFH. A mandatory corrective action plan was requested on 2024-12-05 with "
        "a response due within 10 business days. Seafood line items fulfilled by Harbor Point "
        "Seafood (SUP-1157) performed better than the Tidewater average at 96.8% on-time, "
        "indicating the shortfall sits in the frozen prepared and vegetable categories rather than "
        "in the affiliate's seafood operation.\n\n"

        "4. ACTIONS\n"
        "4.1 Corrective action plan requested from Tidewater Frozen Holdings, due 2024-12-19.\n"
        "4.2 Written notice of on-time shortfall issued to Northlake Protein Partners.\n"
        "4.3 Globex Dairy Cooperative to be asked to review Friday route sequencing.\n"
        "4.4 The District will not assess liquidated damages this quarter. Article 4.4 of "
        "CTR-2024-0412-VV and Article 5.4 of CTR-2024-0489-TFH remain available if performance "
        "does not improve in Q2 FY2025."
    ),
    "metadata": {
        "document_type": "performance_review", "category": "Performance Review",
        "buyer_id": "ORG-RUSD", "period": "Q1 FY2025",
        "period_start": "2024-09-01", "period_end": "2024-11-30",
        "issue_date": "2024-12-05", "status": "issued",
        "district_satisfaction_index": "8.5/10",
        "entities": ["ORG-RUSD", "PER-DWHITFIELD", "SUP-1042", "SUP-1103", "SUP-1204",
                     "SUP-1189", "SUP-1156", "SUP-1157",
                     "CTR-2024-0412-VV", "CTR-2024-0489-TFH", "CTR-2024-0489-NPP",
                     "VV-PRD-1180"],
        "relations": [
            ["ORG-RUSD", "EVALUATED", "SUP-1042"],
            ["ORG-RUSD", "EVALUATED", "SUP-1103"],
            ["ORG-RUSD", "EVALUATED", "SUP-1156"],
            ["ORG-RUSD", "EVALUATED", "SUP-1189"],
            ["ORG-RUSD", "EVALUATED", "SUP-1204"],
            ["SUP-1042", "SATISFACTION_RATING", "4.8/5"],
            ["SUP-1103", "SATISFACTION_RATING", "4.6/5"],
            ["SUP-1204", "SATISFACTION_RATING", "4.5/5"],
            ["SUP-1189", "SATISFACTION_RATING", "4.3/5"],
            ["SUP-1156", "SATISFACTION_RATING", "4.1/5"],
            ["ORG-RUSD", "REQUESTED_CORRECTIVE_ACTION_FROM", "SUP-1156"],
            ["PER-DWHITFIELD", "AUTHORED", "perf-review-q1-fy2025"],
        ],
    },
},

"supplier-profile-tidewater": {
    "title": "Supplier Qualification Profile — Tidewater Frozen Holdings (SUP-1156)",
    "content": (
        "SUPPLIER QUALIFICATION PROFILE\n"
        "Supplier: Tidewater Frozen Holdings (SUP-1156)\n"
        "Maintained by: Riverside Unified School District Procurement\n"
        "Last updated: 2024-11-20\n\n"

        "1. CORPORATE\n"
        "Legal name: Tidewater Frozen Holdings, Inc. Founded 1987. Headquarters: 4400 Newton Road, "
        "Stockton, California. Ownership: privately held; majority stake acquired by Cordova "
        "Capital Partners in 2019. FY2024 revenue 412 million USD, up 6.2% year on year. "
        "Employees: 1,180. DUNS 08-441-9027.\n"
        "Categories served: Frozen Foods, Dry Groceries, Meat & Poultry. Customer base: 214 K-12 "
        "school districts, 46 higher education accounts, 31 healthcare accounts across California, "
        "Nevada and Arizona.\n\n"

        "2. SUBSIDIARY\n"
        "Harbor Point Seafood (SUP-1157) has been a wholly owned subsidiary since 2021-06-30, "
        "acquired for 34 million USD. Harbor Point operates independently from a 62,000 square "
        "foot facility in Oxnard, California, employs 145 people, and holds Marine Stewardship "
        "Council Chain of Custody certification MSC-C-58817 through 2026-01-31. Harbor Point is "
        "the named seafood fulfilment affiliate under BID-2024-117 and contract CTR-2024-0489-TFH. "
        "Under Article 2.2 of that contract, Tidewater remains solely liable to the District for "
        "Harbor Point's performance.\n\n"

        "3. FACILITIES\n"
        "  FAC-TW-STOCKTON      Stockton, CA      285,000 sq ft   SQF Level 2, exp. 2025-04-30\n"
        "                                                          BRCGS grade AA, exp. 2025-08-19\n"
        "  FAC-TW-BAKERSFIELD   Bakersfield, CA   140,000 sq ft   SQF Level 2, exp. 2025-10-08\n"
        "  FAC-HP-OXNARD        Oxnard, CA         62,000 sq ft   MSC CoC, exp. 2026-01-31\n"
        "Fleet: 96 refrigerated vehicles, average age 4.2 years, all fitted with continuous "
        "temperature telemetry.\n\n"

        "4. CONTRACT HISTORY WITH THE DISTRICT\n"
        "  CTR-2024-0489-TFH   Frozen Foods and Protein, Lots 1 and 2. Effective 2024-08-15 "
        "through 2026-08-14. Annual value 2,984,220 USD. Status: active.\n"
        "Tidewater was also solicited under RFP-2024-0412 (produce) and RFP-2024-0455 (dairy) and "
        "did not submit a proposal for either, citing category fit. Tidewater submitted question "
        "Q5 during the RFP-2024-0412 question period asking whether produce and frozen would be "
        "awarded together; the District confirmed they would not.\n\n"

        "5. PERFORMANCE HISTORY\n"
        "Q1 FY2025 was the supplier's first full quarter under CTR-2024-0489-TFH and the results "
        "were the weakest of any District supplier: on-time delivery 92.0%, fill rate 95.3%, "
        "quality rejections 1.7%, customer satisfaction 4.1 out of 5. Both October and November "
        "fell below the 92% monthly on-time threshold, triggering the mandatory corrective action "
        "plan under Article 5.4. The plan was requested 2024-12-05 with a response due 2024-12-19.\n"
        "Notably, seafood line items fulfilled by Harbor Point Seafood ran at 96.8% on-time, well "
        "above the Tidewater average, which locates the problem in frozen prepared foods and "
        "frozen vegetables rather than in the affiliate.\n\n"

        "6. RISK NOTES\n"
        "6.1 Certification timing. The Stockton SQF Level 2 certificate expires 2025-04-30, inside "
        "the contract term. Article 8 of CTR-2024-0489-TFH requires written evidence of renewal by "
        "2025-05-14 or new orders are suspended. The renewal audit is scheduled 2025-03-11. This "
        "is the District's most time-sensitive supplier compliance item.\n"
        "6.2 Audit findings. The September 2024 third-party audit at Stockton recorded two minor "
        "non-conformances, a freezer door seal and a gap in manual temperature logging. Both "
        "closed 2024-10-11 with auditor verification.\n"
        "6.3 Concentration. Tidewater represents 43% of the District's total SY 2024-25 food "
        "spend across all categories, the largest single-supplier concentration in the "
        "District's portfolio. Loss of this supplier mid-year would require emergency "
        "re-solicitation of two lots.\n\n"

        "7. REFERENCES\n"
        "Three K-12 references were checked in May 2024: Kern County Consolidated (positive, noted "
        "delivery timing issues in the first six months), Sierra Foothills USD (positive), and "
        "Antelope Valley Schools (positive, noted strong USDA Foods processing support)."
    ),
    "metadata": {
        "document_type": "supplier_profile", "category": "Frozen Foods",
        "supplier_id": "SUP-1156", "supplier_name": "Tidewater Frozen Holdings",
        "buyer_id": "ORG-RUSD", "issue_date": "2024-11-20", "status": "current",
        "entities": ["SUP-1156", "SUP-1157", "ORG-RUSD", "CTR-2024-0489-TFH",
                     "BID-2024-117", "RFP-2024-0412", "RFP-2024-0455", "RFP-2024-0489",
                     "FAC-TW-STOCKTON", "FAC-TW-BAKERSFIELD", "FAC-HP-OXNARD",
                     "CERT-TW-SQF2-STK", "CERT-TW-BRCGS", "CERT-HP-MSC", "PER-GHALLORAN"],
        "relations": [
            ["SUP-1157", "SUBSIDIARY_OF", "SUP-1156"],
            ["SUP-1156", "ACQUIRED", "SUP-1157"],
            ["SUP-1156", "OPERATES_FACILITY", "FAC-TW-STOCKTON"],
            ["SUP-1156", "OPERATES_FACILITY", "FAC-TW-BAKERSFIELD"],
            ["SUP-1157", "OPERATES_FACILITY", "FAC-HP-OXNARD"],
            ["FAC-TW-STOCKTON", "CERTIFIED_BY", "CERT-TW-SQF2-STK"],
            ["FAC-TW-STOCKTON", "CERTIFIED_BY", "CERT-TW-BRCGS"],
            ["FAC-HP-OXNARD", "CERTIFIED_BY", "CERT-HP-MSC"],
            ["SUP-1156", "PARTY_TO", "CTR-2024-0489-TFH"],
            ["SUP-1156", "DECLINED_TO_BID", "RFP-2024-0412"],
            ["SUP-1156", "DECLINED_TO_BID", "RFP-2024-0455"],
        ],
    },
},

"cert-compliance-register-2024": {
    "title": "Food Safety Certification and Compliance Register — Riverside USD, SY 2024-25",
    "content": (
        "FOOD SAFETY CERTIFICATION AND COMPLIANCE REGISTER\n"
        "Riverside Unified School District (ORG-RUSD), Nutrition Services\n"
        "Position as at 2024-12-01. Maintained by Miguel Arredondo (PER-MARREDONDO).\n\n"

        "1. PURPOSE\n"
        "This register tracks every certification the District's contracts require a supplier to "
        "hold, its expiry date, and the contractual consequence of lapse. It is reviewed monthly. "
        "Every District supply contract makes loss of a required certification a material breach.\n\n"

        "2. REGISTER\n\n"
        "CERT-VV-GAP — GlobalG.A.P., Valle Verde Produce Co. (SUP-1042)\n"
        "  Expires 2025-06-30. Required by Article 7, CTR-2024-0412-VV. Status: current.\n\n"
        "CERT-VV-SQF2 — SQF Level 2, Valle Verde Fresno facility (FAC-VV-FRESNO)\n"
        "  Expires 2025-09-14. Required by Article 7, CTR-2024-0412-VV. Status: current.\n"
        "  Last audit 2024-02-08, zero non-conformances.\n\n"
        "CERT-VV-ORG — USDA Organic handler, Valle Verde Produce Co. (SUP-1042)\n"
        "  Expires 2025-11-01. Not contractually required; supports organic line items. Current.\n\n"
        "CERT-GX-SQF3 — SQF Level 3, Globex Visalia plant (FAC-GX-VISALIA)\n"
        "  Expires 2026-02-28. Exceeds the SQF Level 2 minimum in RFP-2024-0455. Current.\n"
        "  Last audit 2024-03-05, zero non-conformances.\n\n"
        "CERT-GX-GRADEA — California Grade A dairy permit CA-DP-4471, Globex (SUP-1103)\n"
        "  Renewed annually, current through 2025-06-30. Mandatory for fluid milk supply.\n\n"
        "CERT-TW-SQF2-STK — SQF Level 2, Tidewater Stockton facility (FAC-TW-STOCKTON)\n"
        "  Expires 2025-04-30. ** EARLIEST EXPIRY IN THE REGISTER. ** Required by Article 8, "
        "CTR-2024-0489-TFH, which requires written evidence of renewal no later than 2025-05-14 or "
        "new orders are suspended until cured. Renewal audit scheduled 2025-03-11. Flagged for "
        "monthly follow-up from January 2025.\n"
        "  September 2024 audit: two minor non-conformances (freezer door seal; gap in manual "
        "temperature logging). Both closed 2024-10-11, auditor verified.\n\n"
        "CERT-TW-SQF2-BKF — SQF Level 2, Tidewater Bakersfield facility (FAC-TW-BAKERSFIELD)\n"
        "  Expires 2025-10-08. Required by Article 8, CTR-2024-0489-TFH. Status: current.\n\n"
        "CERT-TW-BRCGS — BRCGS grade AA, Tidewater Stockton facility (FAC-TW-STOCKTON)\n"
        "  Expires 2025-08-19. Not contractually required; supplementary assurance. Current.\n\n"
        "CERT-HP-MSC — Marine Stewardship Council Chain of Custody MSC-C-58817, "
        "Harbor Point Seafood (SUP-1157), Oxnard facility (FAC-HP-OXNARD)\n"
        "  Expires 2026-01-31. Required by Article 2.3, CTR-2024-0489-TFH. Lapse suspends the "
        "Supplier's right to deliver seafood line items until restored. Status: current.\n\n"
        "CERT-NL-SQF2 — SQF Level 2, Northlake Modesto facility (FAC-NL-MODESTO)\n"
        "  Expires 2025-12-05. Required by CTR-2024-0489-NPP. Status: current.\n"
        "  Last audit 2024-04-18, one minor non-conformance on label verification, closed "
        "2024-05-02.\n\n"
        "CERT-NL-USDA — USDA FSIS establishment EST. 18442, Northlake (SUP-1189)\n"
        "  Continuous inspection, no expiry. Mandatory for further-processed protein.\n\n"

        "3. SUPPLIERS NOT UNDER CONTRACT\n"
        "Acme Foods Distribution (SUP-1077) is not currently under District contract. Its Riverside "
        "facility (FAC-AF-RIV) SQF Level 2 certificate was in renewal and not current on "
        "2024-05-03, which was a material factor in the RFP-2024-0412 award decision. Its "
        "Sacramento facility (FAC-AF-SAC) certificate expires 2025-07-22. Should Acme Foods bid a "
        "future solicitation, both certificates must be current on the proposal due date.\n\n"

        "4. EXPIRY CALENDAR, NEXT 12 MONTHS\n"
        "  2025-04-30   CERT-TW-SQF2-STK    Tidewater Stockton      ** action required **\n"
        "  2025-06-30   CERT-VV-GAP         Valle Verde\n"
        "  2025-06-30   CERT-GX-GRADEA      Globex\n"
        "  2025-08-19   CERT-TW-BRCGS       Tidewater Stockton\n"
        "  2025-09-14   CERT-VV-SQF2        Valle Verde Fresno\n"
        "  2025-10-08   CERT-TW-SQF2-BKF    Tidewater Bakersfield\n"
        "  2025-11-01   CERT-VV-ORG         Valle Verde\n"
        "  2025-12-05   CERT-NL-SQF2        Northlake Modesto\n\n"

        "5. OPEN COMPLIANCE ITEMS\n"
        "5.1 Tidewater Frozen Holdings corrective action plan, requested 2024-12-05 following two "
        "consecutive months below 92% on-time delivery. Response due 2024-12-19. Performance, not "
        "certification, but tracked here because Article 10 of CTR-2024-0489-TFH makes sustained "
        "failure a termination-for-cause ground.\n"
        "5.2 No other open compliance items across the District's supplier portfolio."
    ),
    "metadata": {
        "document_type": "compliance_register", "category": "Compliance",
        "buyer_id": "ORG-RUSD", "issue_date": "2024-12-01", "status": "current",
        "entities": ["ORG-RUSD", "PER-MARREDONDO",
                     "SUP-1042", "SUP-1077", "SUP-1103", "SUP-1156", "SUP-1157", "SUP-1189",
                     "CERT-VV-GAP", "CERT-VV-SQF2", "CERT-VV-ORG", "CERT-GX-SQF3",
                     "CERT-GX-GRADEA", "CERT-TW-SQF2-STK", "CERT-TW-SQF2-BKF",
                     "CERT-TW-BRCGS", "CERT-HP-MSC", "CERT-NL-SQF2", "CERT-NL-USDA",
                     "FAC-VV-FRESNO", "FAC-GX-VISALIA", "FAC-TW-STOCKTON",
                     "FAC-TW-BAKERSFIELD", "FAC-HP-OXNARD", "FAC-NL-MODESTO",
                     "FAC-AF-RIV", "FAC-AF-SAC",
                     "CTR-2024-0412-VV", "CTR-2024-0489-TFH", "CTR-2024-0489-NPP"],
        "relations": [
            ["CERT-VV-GAP", "HELD_BY", "SUP-1042"],
            ["CERT-VV-SQF2", "COVERS_FACILITY", "FAC-VV-FRESNO"],
            ["CERT-GX-SQF3", "COVERS_FACILITY", "FAC-GX-VISALIA"],
            ["CERT-TW-SQF2-STK", "COVERS_FACILITY", "FAC-TW-STOCKTON"],
            ["CERT-TW-SQF2-BKF", "COVERS_FACILITY", "FAC-TW-BAKERSFIELD"],
            ["CERT-HP-MSC", "COVERS_FACILITY", "FAC-HP-OXNARD"],
            ["CERT-NL-SQF2", "COVERS_FACILITY", "FAC-NL-MODESTO"],
            ["CERT-TW-SQF2-STK", "REQUIRED_BY", "CTR-2024-0489-TFH"],
            ["CERT-HP-MSC", "REQUIRED_BY", "CTR-2024-0489-TFH"],
            ["CERT-VV-GAP", "REQUIRED_BY", "CTR-2024-0412-VV"],
            ["CERT-VV-SQF2", "REQUIRED_BY", "CTR-2024-0412-VV"],
            ["CERT-NL-SQF2", "REQUIRED_BY", "CTR-2024-0489-NPP"],
            ["PER-MARREDONDO", "MAINTAINS", "cert-compliance-register-2024"],
        ],
    },
},

"email-thread-tidewater-cap": {
    "title": "Email thread — Tidewater corrective action plan, December 2024",
    "content": (
        "EMAIL THREAD — District reference THR-2024-0912\n"
        "Subject: CTR-2024-0489-TFH — Corrective Action Plan request, Q1 FY2025\n"
        "Participants: Dana Whitfield (PER-DWHITFIELD, Riverside USD), Miguel Arredondo "
        "(PER-MARREDONDO, Riverside USD), Gregory Halloran (PER-GHALLORAN, Tidewater Frozen "
        "Holdings)\n\n"

        "----- Message 1 -----\n"
        "From: Dana Whitfield\n"
        "To: Gregory Halloran\n"
        "Cc: Miguel Arredondo\n"
        "Date: 2024-12-05 09:14 PT\n\n"
        "Gregory,\n"
        "Attached is the Q1 FY2025 performance review. Tidewater finished the quarter at 92.0% "
        "on-time delivery against the 95% minimum in Article 5.2 of CTR-2024-0489-TFH, with "
        "October at 91.6% and November at 91.8%. Two consecutive months below 92% triggers "
        "Article 5.4, so I am formally requesting a written corrective action plan. Please respond "
        "within 10 business days, by 2024-12-19.\n"
        "For context, fill rate was 95.3% against a 96% minimum and quality rejections were 1.7% "
        "against a 1.5% ceiling. Satisfaction came in at 4.1 out of 5, the lowest in our portfolio "
        "this quarter. I want to be clear that we are not assessing liquidated damages for Q1 and "
        "we are not contemplating termination. We do need a plan.\n"
        "Dana\n\n"

        "----- Message 2 -----\n"
        "From: Gregory Halloran\n"
        "To: Dana Whitfield\n"
        "Cc: Miguel Arredondo\n"
        "Date: 2024-12-09 16:41 PT\n\n"
        "Dana,\n"
        "Understood, and the numbers are not in dispute. Our own read matches yours. Three points "
        "while we prepare the formal plan.\n"
        "First, the shortfall is concentrated in Lot 1, frozen prepared and frozen vegetables, "
        "running out of Stockton. Our seafood line items through Harbor Point ran 96.8% on-time in "
        "the same period, so this is a Stockton routing problem rather than a network problem.\n"
        "Second, root cause. We added 19 new K-12 accounts in August and September and did not "
        "re-sequence the Stockton routes until late October. Riverside's 45-minute dock slots at "
        "the central kitchen sat at the end of a route that grew by roughly 70 minutes.\n"
        "Third, what we have already done. Route re-sequencing went live 2024-11-18 and November "
        "closed at 91.8% against October's 91.6%. December to date is tracking at 95.1%. We are "
        "adding two vehicles to the Stockton fleet in January.\n"
        "The written plan will be with you by 2024-12-17, ahead of your deadline.\n"
        "Gregory\n\n"

        "----- Message 3 -----\n"
        "From: Miguel Arredondo\n"
        "To: Gregory Halloran\n"
        "Cc: Dana Whitfield\n"
        "Date: 2024-12-10 08:02 PT\n\n"
        "Gregory,\n"
        "Thank you. Two additions for the plan, please.\n"
        "One, the quality rejection rate of 1.7% against the 1.5% ceiling needs addressing "
        "separately from delivery timing. Site managers logged most rejections against frozen "
        "vegetable cases with freezer burn, which reads as a cold chain or holding issue rather "
        "than a routing one.\n"
        "Two, unrelated to performance but time-sensitive: the SQF Level 2 certificate for "
        "Stockton (CERT-TW-SQF2-STK) expires 2025-04-30. Article 8 requires written evidence of "
        "renewal by 2025-05-14 or we suspend new orders. You have told us the renewal audit is "
        "booked for 2025-03-11. Please confirm that date in writing and send the certificate the "
        "day you receive it. This is on our register as the District's most time-sensitive "
        "compliance item.\n"
        "Miguel\n\n"

        "----- Message 4 -----\n"
        "From: Gregory Halloran\n"
        "To: Miguel Arredondo\n"
        "Cc: Dana Whitfield\n"
        "Date: 2024-12-10 11:27 PT\n\n"
        "Miguel,\n"
        "Confirmed on both. The Stockton SQF renewal audit is booked for 2025-03-11 with NSF as "
        "certification body; I will send the certificate the day it issues and will not wait for "
        "the 2025-05-14 deadline. On freezer burn, we pulled the affected lots and believe it "
        "traces to holding time in the Stockton freezer rather than transport, since trailer "
        "telemetry for those deliveries was clean. That analysis will be in the plan.\n"
        "Gregory"
    ),
    "metadata": {
        "document_type": "email_thread", "category": "Frozen Foods",
        "thread_id": "THR-2024-0912", "supplier_id": "SUP-1156",
        "supplier_name": "Tidewater Frozen Holdings", "buyer_id": "ORG-RUSD",
        "contract_id": "CTR-2024-0489-TFH",
        "issue_date": "2024-12-05", "status": "open",
        "entities": ["THR-2024-0912", "SUP-1156", "SUP-1157", "ORG-RUSD",
                     "PER-DWHITFIELD", "PER-MARREDONDO", "PER-GHALLORAN",
                     "CTR-2024-0489-TFH", "CERT-TW-SQF2-STK", "FAC-TW-STOCKTON"],
        "relations": [
            ["PER-DWHITFIELD", "SENT_MESSAGE_IN", "THR-2024-0912"],
            ["PER-MARREDONDO", "SENT_MESSAGE_IN", "THR-2024-0912"],
            ["PER-GHALLORAN", "SENT_MESSAGE_IN", "THR-2024-0912"],
            ["THR-2024-0912", "CONCERNS", "CTR-2024-0489-TFH"],
            ["THR-2024-0912", "CONCERNS", "CERT-TW-SQF2-STK"],
            ["ORG-RUSD", "REQUESTED_CORRECTIVE_ACTION_FROM", "SUP-1156"],
        ],
    },
},

}

upsert_documents(SOURCE_DOCS)

_words = sum(len(d["content"].split()) for d in SOURCE_DOCS.values())
_rels  = sum(len(d["metadata"].get("relations", [])) for d in SOURCE_DOCS.values())
print(f"   {len(SOURCE_DOCS)} documents, ~{_words:,} words, {_rels} relation triples.")
print(f"   → Now run Section B.4 to chunk and embed them into {CHUNK_TABLE}.")

## B.4 — Incremental ingest

Only documents whose `content_hash` differs from their `indexed_hash` are re-chunked and
re-embedded, so re-running this is cheap. `force=True` re-embeds everything — **required**
after changing `EMBED_MODEL`, `EMBED_DIM` or the chunk settings, because the stored vectors
were produced under different rules.

The HNSW index is built *after* the load, not before: indexing an empty table then inserting
is much slower.

In [ ]:
def ensure_vector_index():
    """Build the HNSW index once, after the bulk load.

    m / ef_construction trade build time and memory against recall. m=16, ef_construction=64
    is the standard starting point; raise ef_construction toward 128 when recall matters more
    than build time. Requires EMBED_DIM <= 2000.
    """
    with db() as cur:
        cur.execute(f"""
            CREATE INDEX IF NOT EXISTS {CHUNK_TABLE}_embedding_hnsw
            ON {CHUNK_TABLE} USING hnsw (embedding vector_cosine_ops)
            WITH (m = {HNSW_M}, ef_construction = {HNSW_EF_CONSTRUCTION})
        """)
        cur.execute(f"ANALYZE {CHUNK_TABLE}")

def ingest(force: bool = False, only: list = None) -> dict:
    """Chunk + embed + store every document whose content changed since it was last indexed.

    force=True -> re-embed everything regardless. You MUST do this after changing
                  EMBED_MODEL, EMBED_DIM, or the chunking settings, because the vectors
                  already stored were produced by different rules and are now invalid.
    only=[ids] -> restrict to specific documents.
    """
    # Build the WHERE clause. "IS DISTINCT FROM" is like != but treats NULL sensibly:
    # a plain "indexed_hash != content_hash" would be NULL (not true!) for never-indexed
    # documents, silently skipping every new document you add. A classic SQL trap.
    clause = "TRUE" if force else "indexed_hash IS DISTINCT FROM content_hash"
    params = []
    if only:
        clause += " AND doc_id = ANY(%s)"    # ANY(array) is the Postgres way to say "IN (list)"
        params.append(only)

    with db(dict_rows=True) as cur:
        cur.execute(f"SELECT doc_id, title, content, metadata, content_hash "
                    f"FROM {DOC_TABLE} WHERE {clause} ORDER BY doc_id", params)
        pending = cur.fetchall()             # only the documents that actually need work

    if not pending:
        print("Nothing to ingest - every document is already up to date.")
        ensure_vector_index()                # still make sure the index exists
        return {"documents": 0, "chunks": 0}

    total_chunks = 0
    for doc in pending:
        # 1. Cut the document into chunks, each carrying its contextual header (Section B.2).
        chunks = build_chunks(doc["doc_id"], doc["title"], doc["content"], doc["metadata"])

        # 2. Embed the HEADER+TEXT version, not the clean text. This is the fix for your bug:
        #    the supplier/category/date become part of what the vector represents.
        vectors = embed_texts([c["embed_input"] for c in chunks], "RETRIEVAL_DOCUMENT")

        # 3. Pair each chunk with its vector. zip() walks both lists together.
        #    str(v) turns [0.1, 0.2, ...] into "[0.1, 0.2, ...]", which ::vector parses.
        rows = [
            (doc["doc_id"], c["chunk_index"], c["content"], c["embed_input"],
             c["token_count"], json.dumps(doc["metadata"]), str(v))
            for c, v in zip(chunks, vectors)
        ]

        # 4. Replace this document's chunks. All three statements are in ONE transaction
        #    (one `with db()` block), so either all of it lands or none of it does - a crash
        #    here can never leave a document with its old chunks deleted and no new ones.
        with db() as cur:
            # Delete-then-insert rather than update: the new chunk COUNT may differ from the
            # old one, so there is no row-by-row correspondence to update.
            cur.execute(f"DELETE FROM {CHUNK_TABLE} WHERE doc_id = %s", (doc["doc_id"],))
            # execute_values sends all rows in one round trip instead of one INSERT per chunk.
            # `template` tells it how to render each row, including the ::jsonb / ::vector casts.
            execute_values(cur, f"""
                INSERT INTO {CHUNK_TABLE}
                    (doc_id, chunk_index, content, embed_input, token_count, metadata, embedding)
                VALUES %s
            """, rows, template="(%s, %s, %s, %s, %s, %s::jsonb, %s::vector)", page_size=200)
            # Advance the watermark: "the vectors on disk now reflect this version of the text".
            # Because this is inside the same transaction, it can only be recorded if the
            # chunks actually landed - so the two can never disagree.
            cur.execute(f"UPDATE {DOC_TABLE} SET indexed_hash = %s WHERE doc_id = %s",
                        (doc["content_hash"], doc["doc_id"]))

        total_chunks += len(rows)
        avg_tokens = sum(c["token_count"] for c in chunks) // max(1, len(chunks))
        print(f"  {doc['doc_id']:<28} {len(rows):>3} chunks (avg {avg_tokens} tokens)")

    ensure_vector_index()
    print(f"\nIngested {len(pending)} document(s), {total_chunks} chunks. HNSW index ready.")
    return {"documents": len(pending), "chunks": total_chunks}

stats = ingest()

with db(dict_rows=True) as cur:
    cur.execute(f"""
        SELECT count(*) AS chunks, count(DISTINCT doc_id) AS docs,
               round(avg(token_count)) AS avg_tokens, max(token_count) AS max_tokens,
               pg_size_pretty(pg_total_relation_size('{CHUNK_TABLE}')) AS on_disk
        FROM {CHUNK_TABLE}
    """)
    print("\nIndex state:", dict(cur.fetchone()))

## B.5 — Hybrid retrieval (vector + full-text, fused with RRF)

One SQL statement runs both searches and merges them. `mode="vector"` / `"text"` exist so you
can *prove* hybrid earns its complexity rather than assert it.

In [ ]:
# =====================================================================================
# One SQL statement runs BOTH searches and merges them. Reading guide:
#   WITH name AS (...)  = a "CTE", a named temporary result you can refer to below.
#                         Think of it as a variable holding a table.
#   %(name)s            = a placeholder. psycopg2 substitutes the value safely -
#                         this is what prevents SQL injection. NEVER build SQL with
#                         f-strings around user input; table names only, as here.
# The three CTEs are: vec (meaning search) -> txt (keyword search) -> fused (merge).
# =====================================================================================
HYBRID_SQL = f"""
-- ============ CTE 1: the MEANING search (dense / vector / ANN) ============
WITH vec AS (
    -- Why the nested SELECT: a window function like ROW_NUMBER() at the SAME level as
    -- "ORDER BY embedding <=> ..." forces Postgres to rank EVERY row in the table,
    -- which throws away the HNSW index and turns this into a full scan. So the inner
    -- query does the indexed nearest-neighbour lookup and takes the top N; the outer
    -- query then numbers only those N surviving rows. This one detail is the
    -- difference between a millisecond and a full table scan.
    SELECT chunk_id, ROW_NUMBER() OVER (ORDER BY distance) AS rank
    FROM (
        SELECT chunk_id, embedding <=> %(qv)s::vector AS distance
        FROM {CHUNK_TABLE}
        -- The metadata filter. If no filter was passed we send NULL, and
        -- "NULL IS NULL" is true, so the whole condition passes and nothing is excluded.
        -- @> means "contains": metadata @> '{{"category":"Dairy"}}' matches rows whose
        -- metadata includes that key/value. Backed by the GIN index from Section B.1.
        WHERE (%(filter)s::jsonb IS NULL OR metadata @> %(filter)s::jsonb)
        ORDER BY embedding <=> %(qv)s::vector   -- <=> is cosine distance; ASC = nearest first
        LIMIT %(vec_limit)s
    ) v
),
-- ============ CTE 2: the KEYWORD search (lexical / full-text) ============
txt AS (
    SELECT chunk_id, ROW_NUMBER() OVER (ORDER BY score DESC) AS rank
    FROM (
        -- websearch_to_tsquery parses a normal user query the way a search engine would
        -- (it understands quotes and OR), so you can pass raw user input straight in.
        -- The comma is an implicit CROSS JOIN: it just names that parsed query as `q`.
        -- ts_rank_cd scores how well a row matches, factoring in how CLOSE the matched
        -- words are to each other ("cover density") - not merely how often they appear.
        SELECT c.chunk_id, ts_rank_cd(c.tsv, q) AS score
        FROM {CHUNK_TABLE} c, websearch_to_tsquery('english', %(question)s) q
        WHERE c.tsv @@ q        -- @@ means "this text matches this query"
          AND (%(filter)s::jsonb IS NULL OR c.metadata @> %(filter)s::jsonb)
        ORDER BY score DESC     -- keyword scores are "higher is better", unlike distance
        LIMIT %(txt_limit)s
    ) t
),
-- ============ CTE 3: merge the two ranked lists (Reciprocal Rank Fusion) ============
fused AS (
    SELECT
        -- FULL OUTER JOIN keeps chunks found by EITHER search, so a chunk that only the
        -- keyword search found is not discarded. But that means v.chunk_id is NULL for
        -- those rows, hence COALESCE (= "first non-NULL value") to get a usable id.
        COALESCE(v.chunk_id, t.chunk_id) AS chunk_id,

        -- The RRF formula: each search contributes weight / (k + its_rank).
        -- rank 1 -> 1/61 = 0.0164,  rank 2 -> 1/62 = 0.0161,  rank 40 -> 1/100 = 0.0100.
        -- A chunk both searches ranked highly beats a chunk only one of them loved.
        -- COALESCE(..., 0) means "found by only one search" scores 0 from the other,
        -- rather than making the whole sum NULL.
        COALESCE(%(w_vec)s / (%(rrf_k)s + v.rank), 0)
      + COALESCE(%(w_txt)s / (%(rrf_k)s + t.rank), 0) AS rrf_score,

        v.rank AS vec_rank,   -- kept for debugging: WHICH search found this chunk, and where?
        t.rank AS txt_rank    -- NULL here = the keyword search did not find it at all
    FROM vec v FULL OUTER JOIN txt t ON v.chunk_id = t.chunk_id
)
-- ============ Final: attach the actual text and document title ============
-- The CTEs only carried ids and scores around (cheap). Now we join back to fetch the
-- content for the handful of rows that actually won.
SELECT c.chunk_id, c.doc_id, c.chunk_index, c.content, c.metadata,
       d.title, f.rrf_score, f.vec_rank, f.txt_rank,
       (c.embedding <=> %(qv)s::vector) AS cosine_distance   -- reported so we can apply a relevance floor
FROM fused f
JOIN {CHUNK_TABLE} c ON c.chunk_id = f.chunk_id
JOIN {DOC_TABLE}   d ON d.doc_id   = c.doc_id
ORDER BY f.rrf_score DESC, cosine_distance ASC   -- best fused score first; distance breaks ties
LIMIT %(limit)s
"""

def retrieve(question: str, qv=None, mode: str = "hybrid", top_k: int = TOP_K,
             filters: dict = None):
    """First-stage retrieval: find candidate chunks.

    question - the user's question, used by the keyword leg verbatim
    qv       - a pre-computed query embedding. Passing it in lets Section D.2 time the
               embedding and the search separately instead of lumping them together
               (the mistake that made v1.1's benchmark meaningless).
    mode     - "hybrid" (both searches) | "vector" (meaning only) | "text" (keywords only).
               The single-leg modes exist so Section D.2 can PROVE hybrid earns its complexity
               rather than asserting it.
    filters  - metadata restriction, e.g. {"category": "Dairy"}.
    """
    if qv is None:
        qv = list(embed_query_cached(question))   # cached tuple -> list

    # One switchboard drives all three modes: disabling a search leg is just setting its
    # LIMIT to 0 and its weight to 0. Fewer code paths = fewer places for a bug to hide.
    params = {
        "qv": str(list(qv)),   # pgvector accepts the Python list's string form: "[0.1, 0.2, ...]"
        "question": question,
        # json.dumps turns {"category":"Dairy"} into the string '{"category":"Dairy"}';
        # None becomes SQL NULL, which the "IS NULL OR ..." check treats as "no filter".
        "filter": json.dumps(filters) if filters else None,
        "vec_limit": VECTOR_CANDIDATES if mode in ("hybrid", "vector") else 0,
        "txt_limit": TEXT_CANDIDATES   if mode in ("hybrid", "text")   else 0,
        "w_vec":     W_VECTOR if mode in ("hybrid", "vector") else 0.0,
        "w_txt":     W_TEXT   if mode in ("hybrid", "text")   else 0.0,
        "rrf_k": RRF_K,
        "limit": top_k,
    }
    with db(dict_rows=True) as cur:
        # ef_search = how hard HNSW looks before settling. Higher = better recall, slower.
        # SET LOCAL confines it to THIS transaction, so one expensive query cannot slow
        # everyone else down. Postgres does not allow placeholders in SET, so the value is
        # interpolated - int() makes that safe by guaranteeing it is a number, not text.
        cur.execute(f"SET LOCAL hnsw.ef_search = {int(HNSW_EF_SEARCH)}")
        cur.execute(HYBRID_SQL, params)
        return [dict(r) for r in cur.fetchall()]   # RealDictRow -> plain dict


print("✅ Hybrid retrieval ready.")

# ── OPTIONAL retrieval-mode comparison (needs an ingested corpus) ──
# # Compare the three modes on one question:
# q = "What was the Customer Satisfaction level for supplier performance review?"
# for mode in ("vector", "text", "hybrid"):
#     print(f"\n--- {mode} ---")
#     for i, h in enumerate(retrieve(q, mode=mode, top_k=3), 1):
#         print(f"{i}. {h['doc_id']:<26} vec={h['vec_rank']} txt={h['txt_rank']} "
#               f"dist={h['cosine_distance']:.3f}\n   {h['content'][:110]}...")

## B.6 — Reranking

RRF gives a good *recall* list; the reranker gives *precision*. It scores each candidate
0-10 against the question and drops anything below 4 — padding the prompt with irrelevant
passages measurably degrades the answer.

In [ ]:
from google.genai.types import GenerateContentConfig

RERANK_PROMPT = """You are a search relevance rater for a procurement document system.

Rate how well each passage answers the QUESTION, on a 0-10 scale:
  10 = contains the exact answer
   7 = directly about the question's subject, partial answer
   4 = same topic, does not answer the question
   0 = irrelevant

Judge each passage independently. Return ONLY a JSON array of objects with keys
"id" (the passage number) and "score" (integer 0-10). No prose.

QUESTION: {question}

PASSAGES:
{passages}"""

def rerank(question: str, hits: list, top_n: int = RERANK_TOP_N):
    """Second-stage precision pass. Returns at most top_n hits, each with 'rerank_score'.

    Production swap - Vertex AI Ranking API:
        from google.cloud import discoveryengine_v1 as de
        client.rank(ranking_config=..., model="semantic-ranker-default@latest", records=[...])
    """
    if not hits:
        return hits
    passages = "\n\n".join(
        f"[{i}] ({h['title']}) {h['content'][:1200]}" for i, h in enumerate(hits)
    )
    try:
        resp = _with_retry(lambda: genai_client.models.generate_content(
            model=RERANK_MODEL,
            contents=RERANK_PROMPT.format(question=question, passages=passages),
            config=GenerateContentConfig(temperature=0, response_mime_type="application/json"),
        ))
        scores = {int(r["id"]): float(r["score"]) for r in json.loads(resp.text)}
    except Exception as e:
        # Degrade to first-stage order rather than failing the whole query.
        print(f"  [rerank unavailable, keeping RRF order: {type(e).__name__}]")
        return hits[:top_n]

    for i, h in enumerate(hits):
        h["rerank_score"] = scores.get(i, 0.0)
    ranked = sorted(hits, key=lambda h: h["rerank_score"], reverse=True)
    # Drop candidates the reranker judged off-topic even when there is room for them:
    # padding the context with irrelevant passages measurably degrades the answer.
    return [h for h in ranked if h["rerank_score"] >= 4][:top_n] or ranked[:1]


print("✅ Reranker ready.")


## B.7 — `ask_rag()` — grounded generation with citations

Four timed stages: embed → search → rerank → generate. Refuses **before** paying for a
generation call when nothing was semantically close *and* no keyword matched.

In [ ]:
REFUSAL = "I don't have that in the indexed documents."

SYSTEM_RULES = f"""You are a procurement analyst assistant. Answer using ONLY the numbered SOURCES below.

Rules:
1. Cite every fact with its source number in square brackets, e.g. [2]. No citation, no claim.
2. If several entities or several values legitimately match the question, list ALL of them with
   attribution. Never collapse distinct figures into one number, and never present an aggregate as
   though it were an individual entity's value (or the reverse).
3. If the question names a specific entity (supplier, bid, category), answer for THAT entity. If the
   sources only cover a different entity, say so explicitly instead of substituting it.
4. Quote figures exactly as written in the sources - same units, scale, currency and precision.
5. If the sources do not contain the answer, reply with exactly: "{REFUSAL}"
   Do not guess, and do not use knowledge from outside the sources.
6. Be concise. Lead with the answer, then the supporting detail."""

def format_sources(hits) -> str:
    """Turn retrieved chunks into the numbered [1] [2] [3] block the prompt refers to.

    Each block carries the document title AND its metadata, so the model can attribute a
    bare number like "4.8/5" to the right supplier. Without this the model sees floating
    facts with no owner - which is precisely how v1.1 produced the wrong answer.
    """
    blocks = []
    for i, h in enumerate(hits, 1):        # enumerate(..., 1) numbers from 1, not 0
        m = h.get("metadata") or {}        # `or {}` guards against metadata being None
        facets = " | ".join(f"{k}: {m[k]}" for k in HEADER_FIELDS if m.get(k))
        head = f"[{i}] {h['title']} (doc_id: {h['doc_id']}" + (f" | {facets}" if facets else "") + ")"
        blocks.append(f"{head}\n{h['content']}")
    return "\n\n".join(blocks)

def ask_rag(question: str, filters: dict = None, mode: str = "hybrid",
        use_reranker: bool = USE_RERANKER, verbose: bool = False) -> dict:
    """The full RAG request. Four stages, each timed separately:

        1. embed    - turn the question into a vector
        2. search   - find candidate chunks (Section B.5)
        3. rerank   - keep only the best few (Section B.6)
        4. generate - Gemini writes the answer from those chunks only

    Returns a dict with the answer, the sources behind it, and per-stage timings - so
    every answer is auditable and every slow request is diagnosable.

    verbose=True prints the assembled prompt. Do this once: seeing exactly what the model
    was given is the fastest way to understand any wrong answer.
    """
    timings = {}

    # ---- Stage 1: embed the question ----
    t0 = time.perf_counter()               # perf_counter is a high-resolution stopwatch
    qv = list(embed_query_cached(question))
    timings["embed_ms"] = (time.perf_counter() - t0) * 1000   # seconds -> milliseconds

    # ---- Stage 2: search ----
    # Retrieve MORE candidates when reranking, because the reranker's job is to sift a wide
    # net. Without one, take only what will fit in the prompt.
    first_stage_k = RERANK_CANDIDATES if use_reranker else TOP_K
    t0 = time.perf_counter()
    hits = retrieve(question, qv=qv, mode=mode, top_k=first_stage_k, filters=filters)
    timings["search_ms"] = (time.perf_counter() - t0) * 1000

    # ---- Guardrail: refuse BEFORE paying for a generation call ----
    # Two conditions must BOTH hold to refuse: nothing was semantically close, AND no
    # keyword matched either (txt_rank is None means the keyword search never found it).
    # Requiring both avoids refusing on a valid exact-ID lookup, where cosine distance can
    # look poor but the keyword hit is decisive.
    # v1.1 had no such check: it always returned its top 3, however irrelevant, which is
    # how RAG systems end up confidently answering from unrelated documents.
    if not hits or (min(h["cosine_distance"] for h in hits) > MAX_COSINE_DISTANCE
                    and all(h["txt_rank"] is None for h in hits)):
        return {"answer": REFUSAL, "sources": [], "timings": timings, "refused": True}

    # ---- Stage 3: rerank ----
    t0 = time.perf_counter()
    hits = rerank(question, hits) if use_reranker else hits[:RERANK_TOP_N]
    timings["rerank_ms"] = (time.perf_counter() - t0) * 1000

    # ---- Stage 4: build the prompt and generate ----
    # Structure: rules first, then the evidence, then the question. The model is instructed
    # to use ONLY what appears between them.
    prompt = f"{SYSTEM_RULES}\n\nSOURCES:\n{format_sources(hits)}\n\nQUESTION: {question}"
    if verbose:
        print(prompt[:2000], "\n...\n")

    t0 = time.perf_counter()
    resp = _with_retry(lambda: genai_client.models.generate_content(
        model=CHAT_MODEL, contents=prompt,
        # temperature=0 = be as deterministic as possible. Creativity is exactly what you
        # do NOT want here; you want faithful reporting of the sources.
        config=GenerateContentConfig(temperature=0, max_output_tokens=1024),
    ))
    timings["generate_ms"] = (time.perf_counter() - t0) * 1000

    answer = (resp.text or "").strip()
    return {
        "answer": answer,
        "sources": [{"n": i, "doc_id": h["doc_id"], "title": h["title"],
                     "content": h["content"], "metadata": h.get("metadata") or {},
                     "cosine_distance": round(h["cosine_distance"], 4),
                     "rerank_score": h.get("rerank_score")}
                    for i, h in enumerate(hits, 1)],
        "timings": timings,
        "refused": answer.startswith(REFUSAL),
    }

def show(result: dict):
    print(result["answer"])
    print("\nSources:")
    for s in result["sources"]:
        print(f"  [{s['n']}] {s['doc_id']} (dist={s['cosine_distance']}, "
              f"rerank={s['rerank_score']})")
    print("  timings:", {k: round(v) for k, v in result["timings"].items()}, "ms")

print("✅ RAG engine ready — ask_rag(\"your question\"), or use the router in Section C.")


---
# Section G — GraphRAG

Hybrid retrieval finds passages that *look like the question*. That fails on **multi-hop**
questions, where the pieces of the answer live in documents that don't resemble each other:

> *"When does Tidewater's Stockton SQF certificate expire and what happens if it lapses?"*

The expiry is in the compliance register. The consequence is in Article 8 of the contract.
The renewal audit date is in an email. Nothing about those three passages is textually
similar — but they are all one hop from the node `CERT-TW-SQF2-STK`.

GraphRAG adds one stage between retrieval and generation:

```
question → hybrid search (Section B) → seed chunks
         → which ENTITIES do those chunks mention?        ← rag_node_mentions
         → walk 1-2 hops from those entities              ← rag_edges
         → pull the chunks attached to the neighbours
         → rerank the union → generate, with the SUBGRAPH in the prompt
```

### Where the graph comes from — two sources, no LLM extraction

| Source | Gives you | Cost |
|---|---|---|
| **Your foreign keys** (G.3) | `bids_old.supplierid → suppliers_old.id` is an edge. `rfp_solicited_suppliers` is an edge table. Thousands of relationships already stated exactly. | free, deterministic |
| **Document metadata** (G.4) | the `entities` / `relations` triples carried by each document | free, deterministic |

Microsoft-style GraphRAG pays an LLM per chunk to *guess* entities and relations. You don't
need to: your schema already declares them, and your documents already carry them. Extraction
is the expensive, error-prone part of GraphRAG and you get to skip it entirely.

**What this section does NOT do:** no community detection, no LLM-written community summaries,
no global "what are the themes" search. Those earn their cost at thousands of documents; at
your size they cost money and tell you less than reading the corpus. Add them later if the
corpus grows — the node/edge tables built here are exactly what they'd sit on.

## G.1 — Graph configuration

`GRAPH_MAX_ROWS_PER_TABLE` is the important one. Tables above it are skipped, because turning
17,000 line items into 17,000 nodes buries the entities you actually reason about. Raise it
once you know you want them.

In [ ]:
# ============================ what goes into the graph ============================
NODE_TABLE     = "rag_nodes"
EDGE_TABLE     = "rag_edges"
MENTION_TABLE  = "rag_node_mentions"

# Graph plumbing is not business data — hide it from the SQL engine too.
RAG_INTERNAL_TABLES = RAG_INTERNAL_TABLES | {NODE_TABLE, EDGE_TABLE, MENTION_TABLE}

# Tables with more rows than this become edges but not nodes. Line-item tables are the
# usual casualty and that is deliberate: 17,000 line-item nodes drown the suppliers,
# bids and RFPs you actually ask questions about.
GRAPH_MAX_ROWS_PER_TABLE = 5000

# Restrict to specific tables, or None for "every business table under the row cap".
GRAPH_INCLUDE_TABLES = None      # e.g. ["suppliers", "suppliers_old", "bids_old", "rfps_old"]
GRAPH_EXCLUDE_TABLES = {"poll_state", "document_vectors", "portal_templates"}

# Column names checked, in order, when picking a human-readable label for a row.
LABEL_COLUMN_CANDIDATES = [
    "organizationname", "organization_name", "supplier_name", "suppliername",
    "name", "title", "subject", "label", "description", "code", "number",
    "rfp_number", "bid_number", "email",
]

# ============================ traversal behaviour ============================
GRAPH_HOPS          = 2     # how far to walk from the entities the question landed on.
                            # 1 = direct neighbours. 2 = neighbours of neighbours — the
                            # sweet spot. 3+ pulls in half the graph and hurts precision.
GRAPH_HUB_DEGREE_MAX = 60   # THE guard rail. A node connected to everything (your own
                            # organisation, a status value) would make 2 hops reach the
                            # entire graph. Nodes above this degree can be an ANSWER but
                            # are never walked THROUGH. Lower it if expansion feels noisy.
GRAPH_MAX_NODES     = 60    # cap on expanded nodes handed to the prompt
GRAPH_MAX_EXTRA_CHUNKS = 12 # cap on chunks pulled in purely because the graph said so
GRAPH_MAX_CHUNKS_PER_DOC = 2  # DIVERSITY GUARD. Without it, one long document that mentions
                            # many expanded nodes eats the whole budget and the short document
                            # holding the other half of the answer never gets read.
GRAPH_SEED_CHUNKS   = 6     # how many hybrid-search hits are used to find starting entities
GRAPH_MAX_FACTS     = 40    # triples rendered into the prompt

print(f"Graph config: {GRAPH_HOPS} hops, hub guard at degree {GRAPH_HUB_DEGREE_MAX}, "
      f"node tables capped at {GRAPH_MAX_ROWS_PER_TABLE:,} rows.")

## G.2 — Graph schema

Three tables. `rag_nodes` and `rag_edges` are the graph; `rag_node_mentions` is the bridge
between the graph and the text — *this node is talked about in that chunk*. That bridge is
what makes GraphRAG possible: without it you have a graph and a vector store that never meet.

In [ ]:
GRAPH_DDL = f"""
CREATE TABLE IF NOT EXISTS {NODE_TABLE} (
    node_id     text PRIMARY KEY,          -- 'SUP-1042', or 'suppliers_old:412' for a table row
    node_type   text NOT NULL,             -- 'supplier', 'bid', 'certification', 'suppliers_old'...
    label       text NOT NULL,             -- human-readable: 'Tidewater Frozen Holdings'
    properties  jsonb NOT NULL DEFAULT '{{}}'::jsonb,
    source      text NOT NULL,             -- 'relational' (from a foreign key) | 'metadata'
    created_at  timestamptz NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS {EDGE_TABLE} (
    edge_id    bigserial PRIMARY KEY,
    src_id     text NOT NULL REFERENCES {NODE_TABLE}(node_id) ON DELETE CASCADE,
    dst_id     text NOT NULL REFERENCES {NODE_TABLE}(node_id) ON DELETE CASCADE,
    predicate  text NOT NULL,              -- 'SUBMITTED', 'SUBSIDIARY_OF', 'supplierid'...
    properties jsonb NOT NULL DEFAULT '{{}}'::jsonb,
    source     text NOT NULL,
    -- One row per distinct relationship. Re-running the builders is then idempotent
    -- instead of doubling every edge.
    UNIQUE (src_id, predicate, dst_id)
);

-- The bridge: which chunk of which document talks about which node.
CREATE TABLE IF NOT EXISTS {MENTION_TABLE} (
    node_id   text   NOT NULL REFERENCES {NODE_TABLE}(node_id) ON DELETE CASCADE,
    doc_id    text   NOT NULL REFERENCES {DOC_TABLE}(doc_id)   ON DELETE CASCADE,
    chunk_id  bigint NOT NULL REFERENCES {CHUNK_TABLE}(chunk_id) ON DELETE CASCADE,
    method    text   NOT NULL,             -- 'metadata' (declared) | 'label' | 'id_token'
    PRIMARY KEY (node_id, chunk_id)
);

CREATE INDEX IF NOT EXISTS {EDGE_TABLE}_src   ON {EDGE_TABLE} (src_id);
CREATE INDEX IF NOT EXISTS {EDGE_TABLE}_dst   ON {EDGE_TABLE} (dst_id);
CREATE INDEX IF NOT EXISTS {EDGE_TABLE}_pred  ON {EDGE_TABLE} (predicate);
CREATE INDEX IF NOT EXISTS {NODE_TABLE}_type  ON {NODE_TABLE} (node_type);
CREATE INDEX IF NOT EXISTS {NODE_TABLE}_label ON {NODE_TABLE} (lower(label));
CREATE INDEX IF NOT EXISTS {MENTION_TABLE}_chunk ON {MENTION_TABLE} (chunk_id);
CREATE INDEX IF NOT EXISTS {MENTION_TABLE}_doc   ON {MENTION_TABLE} (doc_id);
"""

with db() as cur:
    cur.execute(GRAPH_DDL)
print(f"Graph schema ready: {NODE_TABLE}, {EDGE_TABLE}, {MENTION_TABLE}.")

## G.3 — Build the graph from your foreign keys

**Run this cell to see what your schema would become before building anything.**
`inspect_schema_graph()` reads `information_schema`, reports every foreign key, flags tables
that look like pure join tables (an edge table wearing a table's clothes, e.g.
`rfp_solicited_suppliers`), and shows which tables clear the row cap.

Then `build_relational_graph()` does the work: one node per row of each qualifying table, one
edge per foreign key. Nothing is invented — every edge is a relationship your database already
enforces.

In [ ]:
def _pk_column(table):
    """The single-column primary key of a table, or None (composite PKs are skipped)."""
    with db(dict_rows=True) as cur:
        cur.execute("""
            SELECT a.attname AS col
            FROM pg_index i
            JOIN pg_attribute a ON a.attrelid = i.indrelid AND a.attnum = ANY(i.indkey)
            WHERE i.indrelid = %s::regclass AND i.indisprimary
        """, (f'{DB_SCHEMA}."{table}"',))
        cols = [r["col"] for r in cur.fetchall()]
    return cols[0] if len(cols) == 1 else None


def _foreign_keys():
    """Every single-column FK in the schema: (table, column) -> (ref_table, ref_column)."""
    with db(dict_rows=True) as cur:
        cur.execute("""
            SELECT tc.table_name AS tbl, kcu.column_name AS col,
                   ccu.table_name AS ref_tbl, ccu.column_name AS ref_col
            FROM information_schema.table_constraints tc
            JOIN information_schema.key_column_usage kcu
              ON tc.constraint_name = kcu.constraint_name
             AND tc.table_schema = kcu.table_schema
            JOIN information_schema.constraint_column_usage ccu
              ON ccu.constraint_name = tc.constraint_name
             AND ccu.table_schema = tc.table_schema
            WHERE tc.constraint_type = 'FOREIGN KEY' AND tc.table_schema = %s
        """, (DB_SCHEMA,))
        # Skip the RAG/graph plumbing's own foreign keys — they are not business relationships.
        return [dict(r) for r in cur.fetchall()
                if r["tbl"] not in RAG_INTERNAL_TABLES and r["ref_tbl"] not in RAG_INTERNAL_TABLES]


def _row_counts(tables):
    counts = {}
    with db() as cur:
        for t in tables:
            cur.execute(pgsql.SQL("SELECT count(*) FROM {}.{}").format(
                pgsql.Identifier(DB_SCHEMA), pgsql.Identifier(t)))
            counts[t] = cur.fetchone()[0]
    return counts


def _graph_tables():
    """Business tables eligible to become nodes."""
    cand = [t for t in KNOWN_TABLES if t not in GRAPH_EXCLUDE_TABLES]
    if GRAPH_INCLUDE_TABLES:
        cand = [t for t in cand if t in set(GRAPH_INCLUDE_TABLES)]
    return cand


def _label_column(table):
    """Best human-readable column for this table, or None -> fall back to the id."""
    with db(dict_rows=True) as cur:
        cur.execute("""SELECT column_name, data_type FROM information_schema.columns
                       WHERE table_schema=%s AND table_name=%s""", (DB_SCHEMA, table))
        cols = {r["column_name"].lower(): r["data_type"] for r in cur.fetchall()}
    for c in LABEL_COLUMN_CANDIDATES:
        if c in cols and cols[c] in ("text", "character varying", "character"):
            return c
    # otherwise the first text column that is not obviously a blob of prose
    for c, dt in cols.items():
        if dt in ("text", "character varying") and not c.endswith(("_json", "_xml")):
            return c
    return None


def inspect_schema_graph():
    """Report what a graph built from THIS schema would look like. Changes nothing."""
    tables = _graph_tables()
    counts = _row_counts(tables)
    fks = _foreign_keys()

    fk_by_table = {}
    for fk in fks:
        fk_by_table.setdefault(fk["tbl"], []).append(fk)

    eligible = {t: n for t, n in counts.items() if n and n <= GRAPH_MAX_ROWS_PER_TABLE}
    too_big  = {t: n for t, n in counts.items() if n > GRAPH_MAX_ROWS_PER_TABLE}
    empty    = [t for t, n in counts.items() if n == 0]

    print(f"FOREIGN KEYS FOUND: {len(fks)}")
    if fks:
        for fk in sorted(fks, key=lambda f: (f["tbl"], f["col"]))[:40]:
            print(f"   {fk['tbl']}.{fk['col']}  →  {fk['ref_tbl']}.{fk['ref_col']}")
        if len(fks) > 40:
            print(f"   ... and {len(fks) - 40} more")
    else:
        print("   None. Your tables have no declared foreign keys, so no edges can be")
        print("   derived automatically. Two options: add the FK constraints, or declare")
        print("   the relationships by hand in MANUAL_EDGES below.")

    # A join table is one whose non-PK columns are (almost) all foreign keys.
    print("\nLIKELY EDGE TABLES (a relationship wearing a table's clothes):")
    found_join = False
    for t in tables:
        tf = fk_by_table.get(t, [])
        if len(tf) >= 2:
            print(f"   {t}: {' + '.join(f['ref_tbl'] for f in tf)}  ({counts.get(t, 0):,} rows)")
            found_join = True
    if not found_join:
        print("   none detected")

    print(f"\nTABLES THAT WILL BECOME NODES ({len(eligible)}):")
    for t, n in sorted(eligible.items(), key=lambda x: -x[1]):
        print(f"   {t:<28} {n:>7,} rows   label: {_label_column(t) or '(id only)'}")
    if too_big:
        print(f"\nSKIPPED — over the {GRAPH_MAX_ROWS_PER_TABLE:,}-row cap "
              f"(their FKs still become edges between other nodes):")
        for t, n in sorted(too_big.items(), key=lambda x: -x[1]):
            print(f"   {t:<28} {n:>7,} rows")
    if empty:
        print(f"\nSKIPPED — empty: {', '.join(empty)}")
    print(f"\nEstimated nodes: ~{sum(eligible.values()):,}")
    return {"eligible": eligible, "too_big": too_big, "fks": fks}


# Relationships your schema does not declare as foreign keys. Add them here as
# (child_table, child_column, parent_table, parent_column, PREDICATE) and they are built
# exactly like real FKs. This is the escape hatch when the DB has no FK constraints.
MANUAL_EDGES = [
    # ("bids_old", "supplierid", "suppliers_old", "id", "SUBMITTED_BY"),
    # ("bids_old", "rfpid",      "rfps_old",      "id", "RESPONDS_TO"),
]

schema_graph_report = inspect_schema_graph()

### G.3b — Actually build it

One node per row, one edge per foreign key. Idempotent: `ON CONFLICT DO NOTHING` means
re-running never doubles anything.

In [ ]:
def build_relational_graph(verbose=True):
    """Turn tables into nodes and foreign keys into edges."""
    tables  = _graph_tables()
    counts  = _row_counts(tables)
    node_tables = [t for t, n in counts.items() if 0 < n <= GRAPH_MAX_ROWS_PER_TABLE]

    n_nodes = 0
    pk_of, label_of = {}, {}
    for t in node_tables:
        pk = _pk_column(t)
        if not pk:
            if verbose:
                print(f"   skip {t}: no single-column primary key")
            continue
        pk_of[t] = pk
        lbl = _label_column(t)
        label_of[t] = lbl

        cols = [pk] + ([lbl] if lbl and lbl != pk else [])
        with db(dict_rows=True) as cur:
            cur.execute(pgsql.SQL("SELECT {} FROM {}.{}").format(
                pgsql.SQL(", ").join(map(pgsql.Identifier, cols)),
                pgsql.Identifier(DB_SCHEMA), pgsql.Identifier(t)))
            rows = cur.fetchall()

        payload = []
        for r in rows:
            key = r[pk]
            if key is None:
                continue
            label = str(r[lbl]) if lbl and r.get(lbl) else f"{t} #{key}"
            payload.append((f"{t}:{key}", t, label[:400],
                            json.dumps({"table": t, "pk": str(key)}), "relational"))
        if payload:
            with db() as cur:
                execute_values(cur, f"""
                    INSERT INTO {NODE_TABLE} (node_id, node_type, label, properties, source)
                    VALUES %s ON CONFLICT (node_id) DO NOTHING
                """, payload, template="(%s, %s, %s, %s::jsonb, %s)", page_size=500)
            n_nodes += len(payload)
            if verbose:
                print(f"   {t:<28} {len(payload):>6,} nodes")

    # ---- edges from foreign keys (plus anything declared in MANUAL_EDGES) ----
    fk_specs = [(fk["tbl"], fk["col"], fk["ref_tbl"], fk["ref_col"], fk["col"].upper())
                for fk in _foreign_keys()]
    fk_specs += [(a, b, c, d, e) for a, b, c, d, e in MANUAL_EDGES]

    n_edges = 0
    for tbl, col, ref_tbl, ref_col, predicate in fk_specs:
        # An edge needs BOTH endpoints to exist as nodes.
        if tbl not in pk_of or ref_tbl not in pk_of:
            continue
        src_pk = pk_of[tbl]
        with db(dict_rows=True) as cur:
            cur.execute(pgsql.SQL("SELECT {} AS s, {} AS d FROM {}.{} WHERE {} IS NOT NULL").format(
                pgsql.Identifier(src_pk), pgsql.Identifier(col),
                pgsql.Identifier(DB_SCHEMA), pgsql.Identifier(tbl), pgsql.Identifier(col)))
            pairs = cur.fetchall()
        payload = [(f"{tbl}:{r['s']}", f"{ref_tbl}:{r['d']}", predicate, "relational")
                   for r in pairs if r["s"] is not None and r["d"] is not None]
        if not payload:
            continue
        with db() as cur:
            # Endpoints that were skipped (over the row cap) would violate the FK on
            # rag_edges, so filter to pairs whose nodes actually exist.
            execute_values(cur, f"""
                INSERT INTO {EDGE_TABLE} (src_id, dst_id, predicate, source)
                SELECT v.src, v.dst, v.pred, v.source
                FROM (VALUES %s) AS v(src, dst, pred, source)
                WHERE EXISTS (SELECT 1 FROM {NODE_TABLE} n WHERE n.node_id = v.src)
                  AND EXISTS (SELECT 1 FROM {NODE_TABLE} n WHERE n.node_id = v.dst)
                ON CONFLICT (src_id, predicate, dst_id) DO NOTHING
            """, payload, page_size=1000)
            n_edges += cur.rowcount if cur.rowcount and cur.rowcount > 0 else 0
        if verbose:
            print(f"   {tbl}.{col} → {ref_tbl}: {len(payload):,} candidate edges")

    print(f"\nRelational graph: {n_nodes:,} nodes inserted, {n_edges:,} edges inserted.")
    return {"nodes": n_nodes, "edges": n_edges}


build_relational_graph()

## G.4 — Build the graph from document metadata

The seed corpus in B.3b carries `entities` and `relations` in each document's JSONB. This
reads them directly. If your own documents arrive without metadata, this cell simply finds
nothing and the relational graph from G.3 still stands on its own.

In [ ]:
# Node type is inferred from the id prefix. Extend this as your id conventions grow.
ID_PREFIX_TYPES = {
    "SUP": "supplier", "ORG": "organization", "PER": "person", "RFP": "rfp",
    "BID": "bid", "CTR": "contract", "CERT": "certification", "FAC": "facility",
    "ADD": "addendum", "LOT": "lot", "THR": "email_thread",
}

def _infer_type(node_id: str) -> str:
    head = node_id.split("-")[0].upper()
    if head in ID_PREFIX_TYPES:
        return ID_PREFIX_TYPES[head]
    if ":" in node_id:                       # 'suppliers_old:412' -> the table name
        return node_id.split(":")[0]
    # SKU-shaped ids: VV-PRD-1180, GX-DRY-0101, HP-SEA-7702
    if _re_sku.match(node_id):
        return "product"
    # A relation object that is a value rather than a thing — "4.8/5", "Produce", "91.4".
    # Kept as a node on purpose: it makes "which suppliers scored above 4.5" traversable.
    if len(node_id) < 24 and not node_id.isupper():
        return "value"
    return "entity"


import re as _re_mod
_re_sku = _re_mod.compile(r"^[A-Z]{2}-[A-Z]{3}-\d{3,}$")


def build_metadata_graph(verbose=True):
    """Read entities/relations out of rag_documents.metadata."""
    with db(dict_rows=True) as cur:
        cur.execute(f"SELECT doc_id, title, metadata FROM {DOC_TABLE}")
        docs = cur.fetchall()

    nodes, edges = {}, set()
    labels = {}
    for d in docs:
        meta = d["metadata"] or {}
        # A supplier id and its name usually travel together in the same document —
        # harvest that pairing so nodes get real labels instead of bare ids.
        if meta.get("supplier_id") and meta.get("supplier_name"):
            labels[meta["supplier_id"]] = meta["supplier_name"]
        if meta.get("buyer_id") and meta.get("buyer_name"):
            labels[meta["buyer_id"]] = meta["buyer_name"]

        for e in meta.get("entities", []) or []:
            nodes.setdefault(str(e), {})
        for triple in meta.get("relations", []) or []:
            if not (isinstance(triple, (list, tuple)) and len(triple) == 3):
                continue
            s, pred, o = (str(x) for x in triple)
            nodes.setdefault(s, {}); nodes.setdefault(o, {})
            edges.add((s, pred, o))

    if not nodes:
        print("No entities/relations found in document metadata — nothing to build here.")
        print("(That is fine: the relational graph from G.3 works on its own.)")
        return {"nodes": 0, "edges": 0}

    node_rows = [(nid, _infer_type(nid), labels.get(nid, nid),
                  json.dumps({}), "metadata") for nid in nodes]
    with db() as cur:
        execute_values(cur, f"""
            INSERT INTO {NODE_TABLE} (node_id, node_type, label, properties, source)
            VALUES %s
            ON CONFLICT (node_id) DO UPDATE
              SET label = CASE WHEN {NODE_TABLE}.label = {NODE_TABLE}.node_id
                               THEN EXCLUDED.label ELSE {NODE_TABLE}.label END
        """, node_rows, template="(%s, %s, %s, %s::jsonb, %s)", page_size=500)

        edge_rows = [(s, o, p, "metadata") for s, p, o in edges]
        execute_values(cur, f"""
            INSERT INTO {EDGE_TABLE} (src_id, dst_id, predicate, source)
            SELECT v.src, v.dst, v.pred, v.source
            FROM (VALUES %s) AS v(src, dst, pred, source)
            WHERE EXISTS (SELECT 1 FROM {NODE_TABLE} n WHERE n.node_id = v.src)
              AND EXISTS (SELECT 1 FROM {NODE_TABLE} n WHERE n.node_id = v.dst)
            ON CONFLICT (src_id, predicate, dst_id) DO NOTHING
        """, edge_rows, page_size=1000)

    if verbose:
        types = {}
        for nid in nodes:
            types[_infer_type(nid)] = types.get(_infer_type(nid), 0) + 1
        print("   node types:", ", ".join(f"{k}={v}" for k, v in sorted(types.items())))
    print(f"Metadata graph: {len(nodes)} nodes, {len(edges)} distinct relationships.")
    return {"nodes": len(nodes), "edges": len(edges)}


build_metadata_graph()

## G.5 — Link the graph to the text

Without this the graph and the vector store never meet. Three linking methods, cheapest first:

1. **`metadata`** — the document declared which entities it touches. Exact, zero guessing.
2. **`id_token`** — the chunk text literally contains the node id (`BID-2024-089`). Exact.
3. **`label`** — the chunk text contains the node's name (`Tidewater Frozen Holdings`).
   Matched on word boundaries, and only for labels distinctive enough to be worth it.

Labels shorter than `MIN_LABEL_LEN` are skipped, because linking on `"Milk"` or `"Active"`
would attach half the corpus to one node and make expansion useless.

In [ ]:
import re as _re

MIN_LABEL_LEN   = 6      # ignore labels shorter than this — too generic to link on
LABEL_BATCH     = 800    # labels per compiled regex (keeps the pattern a sane size)

def link_graph_to_chunks(rebuild=True, verbose=True):
    """Populate rag_node_mentions: node <-> chunk."""
    if rebuild:
        with db() as cur:
            cur.execute(f"TRUNCATE {MENTION_TABLE}")

    with db(dict_rows=True) as cur:
        cur.execute(f"SELECT chunk_id, doc_id, embed_input FROM {CHUNK_TABLE}")
        chunks = cur.fetchall()
        cur.execute(f"SELECT doc_id, metadata FROM {DOC_TABLE}")
        doc_meta = {r["doc_id"]: (r["metadata"] or {}) for r in cur.fetchall()}
        cur.execute(f"SELECT node_id, label FROM {NODE_TABLE}")
        nodes = cur.fetchall()

    if not chunks:
        print("No chunks yet — run Section B.4 first.")
        return 0
    known_ids = {n["node_id"] for n in nodes}
    rows = {}          # (node_id, chunk_id) -> method, first method wins

    def add(nid, cid, did, method):
        if nid in known_ids:
            rows.setdefault((nid, cid), (did, method))

    # ---- 1. declared in the document's metadata -> every chunk of that document ----
    for c in chunks:
        for e in (doc_meta.get(c["doc_id"], {}).get("entities") or []):
            add(str(e), c["chunk_id"], c["doc_id"], "metadata")

    # ---- 2. the node id appears verbatim in the chunk ----
    id_like = [n["node_id"] for n in nodes if _re.match(r"^[A-Z]{2,5}-", n["node_id"])]
    for i in range(0, len(id_like), LABEL_BATCH):
        batch = id_like[i:i + LABEL_BATCH]
        pat = _re.compile(r"(?<![A-Za-z0-9])(" +
                          "|".join(_re.escape(x) for x in batch) + r")(?![A-Za-z0-9-])")
        for c in chunks:
            for m in set(pat.findall(c["embed_input"])):
                add(m, c["chunk_id"], c["doc_id"], "id_token")

    # ---- 3. the node's label appears in the chunk ----
    by_label = {}
    for n in nodes:
        lab = (n["label"] or "").strip()
        if len(lab) >= MIN_LABEL_LEN and lab != n["node_id"]:
            by_label.setdefault(lab.lower(), n["node_id"])
    labels = list(by_label)
    for i in range(0, len(labels), LABEL_BATCH):
        batch = labels[i:i + LABEL_BATCH]
        pat = _re.compile(r"\b(" + "|".join(_re.escape(x) for x in batch) + r")\b",
                          _re.IGNORECASE)
        for c in chunks:
            for m in set(pat.findall(c["embed_input"])):
                add(by_label[m.lower()], c["chunk_id"], c["doc_id"], "label")

    payload = [(nid, did, cid, method) for (nid, cid), (did, method) in rows.items()]
    if payload:
        with db() as cur:
            execute_values(cur, f"""
                INSERT INTO {MENTION_TABLE} (node_id, doc_id, chunk_id, method)
                VALUES %s ON CONFLICT (node_id, chunk_id) DO NOTHING
            """, payload, page_size=1000)

    if verbose:
        by_method = {}
        for _, _, _, m in payload:
            by_method[m] = by_method.get(m, 0) + 1
        print("   by method:", ", ".join(f"{k}={v}" for k, v in sorted(by_method.items())))
    print(f"Linked {len(payload):,} node↔chunk mentions across {len(chunks):,} chunks.")
    return len(payload)


link_graph_to_chunks()

# ---- what did we end up with? ----
with db(dict_rows=True) as cur:
    cur.execute(f"SELECT count(*) AS n FROM {NODE_TABLE}");  _n = cur.fetchone()["n"]
    cur.execute(f"SELECT count(*) AS n FROM {EDGE_TABLE}");  _e = cur.fetchone()["n"]
    cur.execute(f"SELECT count(*) AS n FROM {MENTION_TABLE}"); _m = cur.fetchone()["n"]
    cur.execute(f"""SELECT node_type, count(*) AS n FROM {NODE_TABLE}
                    GROUP BY node_type ORDER BY n DESC LIMIT 12""")
    _types = cur.fetchall()
    cur.execute(f"""SELECT predicate, count(*) AS n FROM {EDGE_TABLE}
                    GROUP BY predicate ORDER BY n DESC LIMIT 12""")
    _preds = cur.fetchall()
print(f"\nGRAPH: {_n:,} nodes | {_e:,} edges | {_m:,} mentions")
print("  top node types:", ", ".join(f"{r['node_type']}={r['n']}" for r in _types))
print("  top predicates:", ", ".join(f"{r['predicate']}={r['n']}" for r in _preds))

## G.6 — Traversal

`expand()` is the heart of it: a recursive CTE that walks outward from a set of seed nodes,
treating edges as undirected (you want the neighbours regardless of which way the arrow
points), and refusing to walk *through* hub nodes.

**The hub guard is the difference between GraphRAG and noise.** In the seed corpus
`ORG-RUSD` touches all 16 documents. Two hops through it reaches everything, and "everything"
is not a retrieval result. Hubs can still be *returned* — they just aren't used as relays.

In [ ]:
EXPAND_SQL = f"""
WITH RECURSIVE deg AS (
    -- degree of every node, both directions, used by the hub guard
    SELECT node_id, count(*)::int AS degree FROM (
        SELECT src_id AS node_id FROM {EDGE_TABLE}
        UNION ALL
        SELECT dst_id FROM {EDGE_TABLE}
    ) u GROUP BY node_id
),
walk(id, hop, path) AS (
    SELECT n.node_id, 0, ARRAY[n.node_id]
    FROM {NODE_TABLE} n
    WHERE n.node_id = ANY(%(seeds)s)
  UNION ALL
    -- Step outward. The CASE flips the edge so we can traverse it in either direction.
    SELECT CASE WHEN e.src_id = w.id THEN e.dst_id ELSE e.src_id END,
           w.hop + 1,
           w.path || (CASE WHEN e.src_id = w.id THEN e.dst_id ELSE e.src_id END)
    FROM walk w
    JOIN deg d ON d.node_id = w.id AND d.degree <= %(hub_max)s   -- never relay through a hub
    JOIN {EDGE_TABLE} e ON (e.src_id = w.id OR e.dst_id = w.id)
    WHERE w.hop < %(hops)s
      AND NOT (CASE WHEN e.src_id = w.id THEN e.dst_id ELSE e.src_id END) = ANY(w.path)
)
SELECT w.id AS node_id, MIN(w.hop) AS hop, n.node_type, n.label,
       COALESCE(d.degree, 0) AS degree
FROM walk w
JOIN {NODE_TABLE} n ON n.node_id = w.id
LEFT JOIN deg d ON d.node_id = w.id
GROUP BY w.id, n.node_type, n.label, d.degree
ORDER BY MIN(w.hop), COALESCE(d.degree, 0) DESC
LIMIT %(limit)s
"""

def expand(seed_ids, hops=None, hub_max=None, limit=None):
    """Walk outward from seed nodes. Returns [{node_id, hop, node_type, label, degree}]."""
    seed_ids = [s for s in dict.fromkeys(seed_ids)]          # de-dup, keep order
    if not seed_ids:
        return []
    with db(dict_rows=True) as cur:
        cur.execute(EXPAND_SQL, {
            "seeds": seed_ids,
            "hops": GRAPH_HOPS if hops is None else hops,
            "hub_max": GRAPH_HUB_DEGREE_MAX if hub_max is None else hub_max,
            "limit": GRAPH_MAX_NODES if limit is None else limit,
        })
        return [dict(r) for r in cur.fetchall()]


def subgraph_facts(node_ids, limit=None):
    """Every edge whose BOTH endpoints are in the given set, rendered with labels."""
    node_ids = list(node_ids)
    if not node_ids:
        return []
    with db(dict_rows=True) as cur:
        cur.execute(f"""
            SELECT s.label AS src, s.node_id AS src_id, e.predicate,
                   t.label AS dst, t.node_id AS dst_id
            FROM {EDGE_TABLE} e
            JOIN {NODE_TABLE} s ON s.node_id = e.src_id
            JOIN {NODE_TABLE} t ON t.node_id = e.dst_id
            WHERE e.src_id = ANY(%s) AND e.dst_id = ANY(%s)
            LIMIT %s
        """, (node_ids, node_ids, GRAPH_MAX_FACTS if limit is None else limit))
        return [dict(r) for r in cur.fetchall()]


def nodes_in_chunks(chunk_ids):
    """Which nodes are mentioned by these chunks — the entry point into the graph."""
    if not chunk_ids:
        return []
    with db(dict_rows=True) as cur:
        cur.execute(f"""
            SELECT m.node_id, n.label, n.node_type, count(*) AS hits
            FROM {MENTION_TABLE} m
            JOIN {NODE_TABLE} n ON n.node_id = m.node_id
            WHERE m.chunk_id = ANY(%s)
            GROUP BY m.node_id, n.label, n.node_type
            ORDER BY hits DESC
        """, (list(chunk_ids),))
        return [dict(r) for r in cur.fetchall()]


def chunks_for_nodes(expanded, exclude_chunk_ids=(), limit=None, per_doc=None):
    """Chunks that talk about these nodes — the way back out of the graph into text.

    `expanded` is the output of expand(), so each node carries its hop distance. Two things
    stop this from degenerating:

      1. HOP WEIGHTING — a chunk about a 1-hop neighbour outranks one about a 3-hop stranger.
         weight = 1/(1+hop)^2, so hop 0 = 1.00, hop 1 = 0.25, hop 2 = 0.11.
      2. PER-DOCUMENT CAP — at most `per_doc` chunks from any one document. Without this a
         single long document that name-drops thirty expanded entities consumes the entire
         budget, and the two-paragraph email holding the other half of the answer is never
         read. That is not a hypothetical: it is exactly what happens without this clause.
    """
    if not expanded:
        return []
    ids     = [n["node_id"] for n in expanded]
    weights = [1.0 / (1 + n.get("hop", 0)) ** 2 for n in expanded]
    with db(dict_rows=True) as cur:
        cur.execute(f"""
            WITH w(node_id, weight) AS (
                SELECT * FROM unnest(%(ids)s::text[], %(weights)s::float8[])
            ),
            scored AS (
                SELECT c.chunk_id, c.doc_id, c.chunk_index, c.content, c.metadata, d.title,
                       count(DISTINCT m.node_id) AS node_hits,
                       sum(w.weight)             AS graph_score
                FROM {MENTION_TABLE} m
                JOIN w              ON w.node_id = m.node_id
                JOIN {CHUNK_TABLE} c ON c.chunk_id = m.chunk_id
                JOIN {DOC_TABLE}   d ON d.doc_id   = c.doc_id
                WHERE NOT (c.chunk_id = ANY(%(exclude)s))
                GROUP BY c.chunk_id, c.doc_id, c.chunk_index, c.content, c.metadata, d.title
            ),
            ranked AS (
                SELECT *, row_number() OVER (PARTITION BY doc_id
                                             ORDER BY graph_score DESC, chunk_id) AS rn
                FROM scored
            )
            SELECT chunk_id, doc_id, chunk_index, content, metadata, title,
                   node_hits, graph_score
            FROM ranked
            WHERE rn <= %(per_doc)s
            ORDER BY graph_score DESC, node_hits DESC
            LIMIT %(limit)s
        """, {"ids": ids, "weights": weights,
              "exclude": list(exclude_chunk_ids) or [-1],
              "per_doc": GRAPH_MAX_CHUNKS_PER_DOC if per_doc is None else per_doc,
              "limit": GRAPH_MAX_EXTRA_CHUNKS if limit is None else limit})
        return [dict(r) for r in cur.fetchall()]


# ---- explorer helpers, for poking at the graph by hand ----
def neighbors(node_id, hops=1):
    """Everything within `hops` of one node. Handy for sanity-checking the build."""
    rows = expand([node_id], hops=hops)
    for r in rows:
        if r["hop"] == 0:
            continue
        print(f"  {r['hop']} hop  {r['node_type']:<16} {r['node_id']:<24} {r['label'][:48]}")
    return rows


def find_node(text, limit=15):
    """Search nodes by label or id — how you find a starting point."""
    with db(dict_rows=True) as cur:
        cur.execute(f"""
            SELECT node_id, node_type, label FROM {NODE_TABLE}
            WHERE label ILIKE %s OR node_id ILIKE %s
            ORDER BY length(label) LIMIT %s
        """, (f"%{text}%", f"%{text}%", limit))
        rows = [dict(r) for r in cur.fetchall()]
    for r in rows:
        print(f"  {r['node_type']:<16} {r['node_id']:<24} {r['label'][:52]}")
    return rows


print("Traversal ready: expand(), neighbors(), find_node(), subgraph_facts().")

## G.7 — `ask_graphrag()`

The full pipeline. Note stage 2: the *graph* decides what else to read, not the embedding
model. That is the whole point — it retrieves passages the question doesn't resemble.

The prompt gets both the passages **and** the subgraph as explicit facts, so the model can say
*"the certificate is required by CTR-2024-0489-TFH"* and have a stated relationship behind it
rather than an inference.

In [ ]:
GRAPH_SYSTEM_RULES = f"""You are a procurement analyst assistant. Answer using ONLY the
numbered SOURCES and the KNOWLEDGE GRAPH FACTS below.

Rules:
1. Cite every fact from a source with its number in square brackets, e.g. [2]. No citation,
   no claim.
2. The GRAPH FACTS are verified relationships from the database. Use them to connect
   entities across sources and to explain HOW things relate. Cite them as [graph].
3. If several entities or values legitimately match, list ALL of them with attribution.
   Never collapse distinct figures into one, and never present an aggregate as an
   individual's value or the reverse.
4. If the question names a specific entity, answer for THAT entity. If the sources only cover
   a different one, say so rather than substituting it.
5. Quote figures exactly as written — same units, scale, currency and precision.
6. When the answer spans several documents, make the chain explicit: what connects to what,
   and where each piece came from.
7. If the sources and facts do not contain the answer, reply exactly: "{REFUSAL}"
8. Be concise. Lead with the answer, then the supporting detail."""


def format_graph_facts(facts) -> str:
    if not facts:
        return "(no relationships found between the retrieved entities)"
    return "\n".join(
        f"  {f['src']} ({f['src_id']}) --[{f['predicate']}]--> {f['dst']} ({f['dst_id']})"
        for f in facts)


def ask_graphrag(question: str, filters: dict = None, hops: int = None,
                 use_reranker: bool = USE_RERANKER, verbose: bool = False) -> dict:
    """Hybrid retrieval, then graph expansion, then grounded generation.

    Stages, each timed:
      1. seed      - normal hybrid search (Section B.5)
      2. link      - which entities do the seed chunks mention?
      3. expand    - walk the graph from those entities
      4. gather    - pull in chunks attached to the neighbours
      5. rerank    - score the union
      6. generate  - answer from passages + subgraph
    """
    timings, t = {}, time.perf_counter

    # ---- 1. seed with ordinary hybrid retrieval ----
    t0 = t()
    seed_hits = retrieve(question, mode="hybrid",
                         top_k=max(GRAPH_SEED_CHUNKS, RERANK_CANDIDATES // 2),
                         filters=filters)
    timings["seed_ms"] = (t() - t0) * 1000
    if not seed_hits:
        return {"answer": REFUSAL, "sources": [], "graph": {"seed_nodes": [], "expanded": [],
                "facts": []}, "timings": timings, "refused": True}

    # ---- 2. which entities did we land on? ----
    t0 = t()
    seed_chunk_ids = [h["chunk_id"] for h in seed_hits[:GRAPH_SEED_CHUNKS]]
    seed_nodes = nodes_in_chunks(seed_chunk_ids)
    timings["link_ms"] = (t() - t0) * 1000

    # ---- 3. walk outward ----
    t0 = t()
    expanded = expand([n["node_id"] for n in seed_nodes], hops=hops)
    timings["expand_ms"] = (t() - t0) * 1000

    # ---- 4. read what the graph pointed at ----
    t0 = t()
    all_seen = {h["chunk_id"] for h in seed_hits}
    extra = chunks_for_nodes(expanded, exclude_chunk_ids=all_seen)
    for e in extra:                       # graph-sourced chunks have no search scores
        e.setdefault("cosine_distance", 1.0)
        e.setdefault("txt_rank", None)
        e["via_graph"] = True
    for h in seed_hits:
        h["via_graph"] = False
    facts = subgraph_facts([n["node_id"] for n in expanded])
    timings["gather_ms"] = (t() - t0) * 1000

    # ---- 5. rerank the union ----
    candidates = seed_hits + extra
    t0 = t()
    hits = rerank(question, candidates) if use_reranker else candidates[:RERANK_TOP_N]
    timings["rerank_ms"] = (t() - t0) * 1000

    # ---- 6. generate ----
    prompt = (f"{GRAPH_SYSTEM_RULES}\n\n"
              f"KNOWLEDGE GRAPH FACTS:\n{format_graph_facts(facts)}\n\n"
              f"SOURCES:\n{format_sources(hits)}\n\n"
              f"QUESTION: {question}")
    if verbose:
        print(prompt[:2500], "\n...\n")

    t0 = t()
    resp = _with_retry(lambda: genai_client.models.generate_content(
        model=CHAT_MODEL, contents=prompt,
        config=GenerateContentConfig(temperature=0, max_output_tokens=1400)))
    timings["generate_ms"] = (t() - t0) * 1000

    answer = (resp.text or "").strip()
    return {
        "answer": answer,
        "sources": [{"n": i, "doc_id": h["doc_id"], "title": h["title"],
                     "content": h["content"], "metadata": h.get("metadata") or {},
                     "cosine_distance": round(h.get("cosine_distance", 1.0), 4),
                     "rerank_score": h.get("rerank_score"),
                     "via_graph": h.get("via_graph", False)}
                    for i, h in enumerate(hits, 1)],
        "graph": {"seed_nodes": seed_nodes, "expanded": expanded, "facts": facts},
        "timings": timings,
        "refused": answer.startswith(REFUSAL),
    }


def show_graph(result: dict):
    """Print a GraphRAG answer, its sources, and the graph reasoning behind it."""
    print(result["answer"])
    g = result.get("graph", {})
    if g.get("seed_nodes"):
        print("\nEntities the question landed on:")
        for n in g["seed_nodes"][:8]:
            print(f"  • {n['node_type']:<14} {n['node_id']:<24} {n['label'][:44]}")
    if g.get("expanded"):
        by_hop = {}
        for n in g["expanded"]:
            by_hop.setdefault(n["hop"], []).append(n)
        print(f"\nGraph expansion ({len(g['expanded'])} nodes):")
        for hop in sorted(by_hop):
            names = ", ".join(n["label"][:26] for n in by_hop[hop][:6])
            more = f" (+{len(by_hop[hop]) - 6} more)" if len(by_hop[hop]) > 6 else ""
            print(f"  {hop} hop: {names}{more}")
    if g.get("facts"):
        print(f"\nRelationships given to the model ({len(g['facts'])}):")
        for f in g["facts"][:8]:
            print(f"  {f['src'][:26]} --[{f['predicate']}]--> {f['dst'][:26]}")
    print("\nSources:")
    for s in result["sources"]:
        tag = " ← pulled in by the graph" if s.get("via_graph") else ""
        print(f"  [{s['n']}] {s['doc_id']} (rerank={s['rerank_score']}){tag}")
    print("  timings:", {k: round(v) for k, v in result["timings"].items()}, "ms")


print("GraphRAG ready — ask_graphrag(\"your question\"), or use the router in Section C.")

## G.8 — See the graph work

Before trusting it, look at it. `find_node()` locates a starting point, `neighbors()` shows
what it connects to. If these look wrong, the retrieval built on them will be wrong too.

In [ ]:
# What does the graph know about Tidewater?
find_node("Tidewater")

In [ ]:
# Everything within 2 hops of the certificate that expires first.
# This is the chain that plain vector search cannot follow.
neighbors("CERT-TW-SQF2-STK", hops=2)

In [ ]:
# The multi-hop question, answered with the graph.
# Watch the "← pulled in by the graph" tags: those are passages the embedding
# model would never have ranked highly, because they don't look like the question.
show_graph(ask_graphrag(
    "When does Tidewater's Stockton SQF certificate expire and what happens if it lapses?"))

In [ ]:
# Compare: same question, plain RAG. Usually finds the expiry date and misses
# the contractual consequence and the renewal audit date.
show(ask_rag(
    "When does Tidewater's Stockton SQF certificate expire and what happens if it lapses?"))

---
# Section C — The router

One cheap Gemini call reads the question, the list of SQL tables, a summary of the indexed
corpus and the size of the knowledge graph, then picks an engine and says why.

| Route | Chosen when the answer is… | Example |
|---|---|---|
| 🗄️ **sql** | computed from columns — a count, total, ranking, filtered list | *"Which supplier has the highest total bid amount?"* |
| 📄 **rag** | a single fact one document states | *"What are Globex's payment terms?"* |
| 🕸️ **graph** | a relationship, dependency, consequence, or a chain across documents | *"What happens if Tidewater's SQF certificate lapses?"* |

**Safety nets.** An engine with nothing behind it is never chosen: no documents → everything
goes to SQL; graph not built → the graph route degrades to RAG. If SQL returns nothing usable,
it retries via RAG. If the graph finds no entity to start from, it retries via RAG. A
malformed router reply fails open to RAG, which refuses cleanly rather than guessing.

**Override:** `ask("...", mode="sql" | "rag" | "graph")` skips the router entirely.

> **Why the router disables thinking.** `gemini-2.5-flash` reasons before answering by
> default, and those thinking tokens come out of `max_output_tokens`. On a small budget the
> allowance is spent before a single character of JSON is emitted, the reply is empty, and
> every question quietly falls back to RAG. The config below sets `thinking_budget=0` and a
> 512-token budget, and reports the real error text rather than just an exception class name.

In [ ]:
ROUTER_PROMPT = """You route a question to ONE of three answering engines over the same
company database.

ENGINE "sql" — writes a PostgreSQL query over these structured tables:
{tables}
Choose it when the answer is a number, count, sum, average, ranking, comparison, date range,
or a filtered list of records — anything computed FROM COLUMNS.

ENGINE "rag" — semantic + keyword search over {n_docs} indexed document(s), e.g.:
{titles}
Choose it when the answer is a SINGLE FACT a document states in prose: a price, a term, a
date, a policy, a clause, "what does X say about Y".

ENGINE "graph" — the same documents, but retrieval follows a knowledge graph of
{n_nodes} entities and {n_edges} relationships (suppliers, bids, RFPs, contracts,
certifications, people, facilities and how they connect).
Choose it when answering requires CONNECTING THINGS rather than looking one thing up:
  - the question mentions two or more entities and asks how they relate
  - it asks what DEPENDS ON, is AFFECTED BY, is REQUIRED BY, or FOLLOWS FROM something
  - it asks about consequences, chains, or knock-on effects ("what happens if X lapses")
  - it asks "everything about X" where X appears across several document types
  - answering it plainly needs facts from documents that would not resemble each other

QUESTION: {question}

Rules:
- COUNT/TOTAL/RANK/COMPARE over records → "sql".
- One fact stated in one document → "rag".
- Relationships, dependencies, consequences, or a chain across documents → "graph".
- Fits no engine (greeting, general knowledge, chit-chat) → "none".
- When torn between "rag" and "graph", prefer "graph" only if a single document is
  genuinely unlikely to hold the whole answer.

Respond with RAW JSON only:
{{"engine": "sql" | "rag" | "graph" | "none", "reason": "<one short sentence>"}}"""


# ---------------------------------------------------------------------------------
# Router model config. gemini-2.5-flash does extended thinking by DEFAULT, and thinking
# tokens are drawn from max_output_tokens. With a small budget the whole allowance is
# consumed by thinking, the response text comes back EMPTY, the JSON parse throws, and
# every question silently falls back to RAG. Two defences: turn thinking off, and give
# the budget real headroom.
try:
    from google.genai.types import ThinkingConfig
    _NO_THINKING = ThinkingConfig(thinking_budget=0)
except ImportError:                    # older google-genai — the token bump alone suffices
    _NO_THINKING = None

_ROUTER_CFG = {"temperature": 0, "max_output_tokens": 512,
               "response_mime_type": "application/json"}
if _NO_THINKING is not None:
    _ROUTER_CFG["thinking_config"] = _NO_THINKING


def _rag_corpus(limit: int = 15):
    """(document count, a few titles) — what the router is told the document side contains."""
    try:
        with db(dict_rows=True) as cur:
            cur.execute(f"SELECT count(*) AS n FROM {DOC_TABLE} WHERE indexed_hash IS NOT NULL")
            n = cur.fetchone()["n"]
            if not n:
                return 0, []
            cur.execute(f"SELECT title FROM {DOC_TABLE} WHERE indexed_hash IS NOT NULL "
                        f"ORDER BY doc_id LIMIT %s", (limit,))
            return n, [r["title"] for r in cur.fetchall()]
    except Exception:
        return 0, []


def _graph_size():
    """(nodes, edges, mentions). All three must be non-zero for the graph route to work:
    a graph with no mentions cannot be entered from a question."""
    try:
        with db(dict_rows=True) as cur:
            cur.execute(f"SELECT count(*) AS n FROM {NODE_TABLE}");    n = cur.fetchone()["n"]
            cur.execute(f"SELECT count(*) AS n FROM {EDGE_TABLE}");    e = cur.fetchone()["n"]
            cur.execute(f"SELECT count(*) AS n FROM {MENTION_TABLE}"); m = cur.fetchone()["n"]
        return n, e, m
    except Exception:
        return 0, 0, 0


def route(question: str) -> dict:
    """Decide which engine answers. Returns {"engine": "sql"|"rag"|"graph"|"none", "reason"}."""
    n_docs, titles = _rag_corpus()
    n_nodes, n_edges, n_mentions = _graph_size()
    graph_ready = bool(n_docs and n_nodes and n_edges and n_mentions)

    # Deterministic short-circuits — never spend a model call on a one-option decision.
    if not KNOWN_TABLES and not n_docs:
        return {"engine": "none", "reason": "No business tables and no indexed documents."}
    if not n_docs:
        return {"engine": "sql", "reason": "No documents indexed — SQL is the only engine available."}
    if not KNOWN_TABLES and not graph_ready:
        return {"engine": "rag", "reason": "No business tables and no graph — RAG only."}

    prompt = ROUTER_PROMPT.format(
        tables=", ".join(sorted(KNOWN_TABLES)) or "(none)",
        n_docs=n_docs,
        titles="\n".join(f"  - {t}" for t in titles),
        n_nodes=n_nodes, n_edges=n_edges,
        question=question,
    )
    try:
        resp = genai_client.models.generate_content(
            model=ROUTER_MODEL, contents=prompt,
            config=GenerateContentConfig(**_ROUTER_CFG))
        if not (resp.text or "").strip():
            # Be explicit instead of dying inside the JSON parser two lines later.
            fr = resp.candidates[0].finish_reason if resp.candidates else "?"
            raise ValueError(f"empty response, finish_reason={fr}")
        data = _parse_json_reply(resp.text)
        engine = str(data.get("engine", "")).lower()
        if engine not in ("sql", "rag", "graph", "none"):
            raise ValueError(f"unexpected engine {engine!r}")
        if engine == "graph" and not graph_ready:
            return {"engine": "rag",
                    "reason": "Graph route chosen but the graph is not built — falling back to RAG."}
        return {"engine": engine, "reason": data.get("reason", "")}
    except Exception as e:
        # Fail OPEN toward RAG: it refuses cleanly when it finds nothing, whereas a bad
        # SQL route burns three generate-validate attempts before giving up.
        # Say WHAT went wrong, not just the exception class — "Router unavailable
        # (JSONDecodeError)" tells you nothing you can act on.
        return {"engine": "rag",
                "reason": f"Router unavailable ({type(e).__name__}: {str(e)[:110]}) "
                          f"— defaulting to RAG."}


def ask(question: str, mode: str = "auto", verbose: bool = True, **kwargs):
    """The one entry point.

    mode="auto" (default) -> the router picks
    mode="sql"            -> ask_sql();      kwargs: execute, max_attempts, on_ambiguity
    mode="rag"            -> ask_rag();      kwargs: filters, use_reranker
    mode="graph"          -> ask_graphrag(); kwargs: filters, use_reranker, hops

    Returns whatever the chosen engine returns: a pandas DataFrame (SQL), or a
    {"answer", "sources", ...} dict (RAG / GraphRAG). show_any() prints any of them.
    """
    if mode not in ("auto", "sql", "rag", "graph"):
        raise ValueError('mode must be "auto", "sql", "rag" or "graph"')

    if mode == "auto":
        decision = route(question)
        engine, reason = decision["engine"], decision["reason"]
        if verbose:
            icon = {"sql": "🗄️", "rag": "📄", "graph": "🕸️", "none": "🚫"}[engine]
            print(f"{icon} routed to {engine.upper()} — {reason}")
        if engine == "none":
            print("   Ask something about your tables, your documents, or how your entities relate,")
            print("   or force an engine with mode=\"sql\" / \"rag\" / \"graph\".")
            return None
    else:
        engine = mode
        if verbose:
            print(f"➡️  forced to {engine.upper()}")

    if engine == "sql":
        sql_kwargs = {k: v for k, v in kwargs.items()
                      if k in ("execute", "max_attempts", "on_ambiguity")}
        result = ask_sql(question, **sql_kwargs)
        # Fallback: SQL found nothing usable but documents exist — try the document side.
        if mode == "auto" and (result is None or (hasattr(result, "empty") and result.empty)):
            n_docs, _ = _rag_corpus()
            if n_docs:
                print("\n↩️  SQL returned nothing usable — retrying via RAG…")
                return show_any(ask_rag(question, **{k: v for k, v in kwargs.items()
                                                     if k in ("filters", "use_reranker")}))
        return result

    if engine == "graph":
        g_kwargs = {k: v for k, v in kwargs.items() if k in ("filters", "use_reranker", "hops")}
        result = ask_graphrag(question, **g_kwargs)
        # Fallback: the graph found no entry point (no entities in the seed chunks).
        if mode == "auto" and not result["graph"]["seed_nodes"]:
            print("\n↩️  No graph entities matched this question — retrying via plain RAG…")
            return show_any(ask_rag(question, **{k: v for k, v in kwargs.items()
                                                 if k in ("filters", "use_reranker")}))
        show_graph(result)
        return result

    rag_kwargs = {k: v for k, v in kwargs.items() if k in ("filters", "use_reranker")}
    return show_any(ask_rag(question, **rag_kwargs))


def show_any(result):
    """Print a result from any engine. Returns it unchanged, so you keep the value."""
    if result is None:
        return None
    if isinstance(result, dict) and "graph" in result:
        show_graph(result)
    elif isinstance(result, dict) and "answer" in result:
        show(result)
    elif isinstance(result, pd.DataFrame):
        display(result)
    return result


print("✅ Router ready — 3 engines.  ask(\"...\")  |  mode=\"sql\" | \"rag\" | \"graph\"")

---

# 🌍 Section H — Global search

Everything above answers questions about **specific things**. Ask *"what happens if
Tidewater's certificate lapses?"* and the graph walks from that entity to its neighbours and
reads the chunks attached to them. That is **local search**, and it is what Sections B and G
do.

Now ask this:

> *"What are the recurring compliance risks across all our suppliers?"*

Every engine you have fails, and not because retrieval is weak. **No chunk contains that
answer.** It exists only as a pattern spread across the whole corpus. Retrieval cannot find
what was never written down — the answer has to be *synthesised*, and synthesis over a whole
corpus is too expensive to do per question.

So it is done once, at index time, in three steps:

```
 INDEX (once)
   1. CLUSTER      group densely-connected parts of the graph into communities
   2. SUMMARISE    one LLM call per community  →  a written report on what it IS
   3. EMBED        vector-index those reports so queries can pre-select

 QUERY (per question)
   4. PRESELECT    the K communities whose reports are closest to the question
   5. MAP          each report scores what it contributes to THIS question
   6. REDUCE       the best-scoring points are combined into one answer
```

## What this section adds beyond the textbook version

| | Naive implementation | Here |
|---|---|---|
| Re-running detection | throws away every summary you paid for | summaries **survive** if a community's membership is unchanged |
| Query cost | maps over **every** community | pre-selects the top *K* by embedding similarity |
| Spending | unbounded | prints an estimate first; `max_calls` caps it |
| Two people run it at once | duplicate LLM spend, interleaved writes | PostgreSQL advisory lock |
| Failures | silent | every query logged to `rag_query_log` with engine, latency and outcome |
| Very broad questions | one flat level | optional level-1 hierarchy over the communities |

## What is *not* touched

`ask_sql()`, `ask_rag()`, `ask_graphrag()`, `route()` and `ROUTER_PROMPT` are unchanged.
Section H wraps `ask()` — it keeps a reference to your existing one and delegates to it for
everything that is not a corpus-wide question. That is why this section must come **after**
the router cell: it needs `ask` to already exist.

## H.1 — Configuration and schema

Three new tables. Nothing existing is altered.

`member_hash` is the important column: a fingerprint of a community's exact membership. When
you re-run clustering after rebuilding the graph, communities whose membership did not change
keep their summary instead of being re-summarised. On a stable corpus a rebuild becomes nearly
free — without it, every rebuild silently re-spends your entire summarisation budget.

In [ ]:
import hashlib as _hashlib, time as _time, re as _re
from concurrent.futures import ThreadPoolExecutor

COMMUNITY_TABLE = "rag_communities"
COMM_MAP_TABLE  = "rag_node_community"
QUERY_LOG_TABLE = "rag_query_log"

# ---- clustering ----
COMMUNITY_MIN_SIZE      = 3    # smaller clusters are not worth an LLM call
COMMUNITY_MAX_NODES     = 60   # entities described in one summary prompt
COMMUNITY_MAX_EDGES     = 80   # relationships shown per summary
COMMUNITY_MAX_CHUNKS    = 4    # representative passages shown per summary
COMMUNITY_EXCLUDE_TYPES = {"value"}   # 'Produce', '4.8/5' — these connect to everything and
                                      # smear every cluster into one blob. Excluded from
                                      # CLUSTERING only; they stay in the graph for local search.

# ---- summarisation (this is where the money goes) ----
SUMMARY_MAX_CALLS   = None   # None = no cap. Set an int while you are evaluating.
SUMMARY_WORKERS     = 4
SUMMARY_MODEL       = CHAT_MODEL

# ---- global query ----
GLOBAL_LEVEL        = 0      # 0 = detailed communities. 1 = the hierarchy from H.4 (broader,
                             # cheaper, less specific). Only set 1 after build_hierarchy().
GLOBAL_PRESELECT_K  = 12     # map over only the K most relevant reports. None = all of them.
                             # THIS is what keeps global search affordable at scale.
GLOBAL_BATCH_SIZE   = 5      # reports per map call
GLOBAL_MIN_RATING   = 3      # skip communities the model rated less important than this
GLOBAL_TOP_POINTS   = 20     # key points carried into the reduce step
GLOBAL_WORKERS      = 4

# Cheap pre-filter: if a question contains none of these, we never spend a model call
# deciding whether it is global, and ask() behaves exactly as it does today.
_GLOBAL_HINTS = _re.compile(
    r"\b(across all|across the|overall|in general|recurring|recurrent|common(?:ly)?|"
    r"themes?|patterns?|trends?|typical(?:ly)?|most (?:common|frequent)|"
    r"what kinds? of|summar(?:y|ise|ize)|main (?:risks?|issues?|problems?|concerns?)|"
    r"generally|as a whole|every supplier|all suppliers|all documents|all contracts)\b",
    _re.IGNORECASE)

GLOBAL_DDL = f"""
CREATE TABLE IF NOT EXISTS {COMMUNITY_TABLE} (
    level             int   NOT NULL DEFAULT 0,
    community_id      int   NOT NULL,
    size              int   NOT NULL,
    node_ids          text[] NOT NULL,
    child_ids         int[],                  -- level >= 1: which communities it contains
    member_hash       text  NOT NULL,         -- fingerprint: lets summaries survive a rebuild
    title             text,
    summary           text,
    rating            real,                   -- the model's own 0-10 importance score
    summary_embedding vector({EMBED_DIM}),     -- powers GLOBAL_PRESELECT_K
    created_at        timestamptz NOT NULL DEFAULT now(),
    PRIMARY KEY (level, community_id)
);

CREATE TABLE IF NOT EXISTS {COMM_MAP_TABLE} (
    node_id      text NOT NULL REFERENCES {NODE_TABLE}(node_id) ON DELETE CASCADE,
    level        int  NOT NULL DEFAULT 0,
    community_id int  NOT NULL,
    PRIMARY KEY (node_id, level)
);

-- Every question, its engine, its latency and whether it worked. The single most useful
-- table you will build: it turns "the answers feel wrong" into a query you can run.
CREATE TABLE IF NOT EXISTS {QUERY_LOG_TABLE} (
    id         bigserial PRIMARY KEY,
    ts         timestamptz NOT NULL DEFAULT now(),
    question   text NOT NULL,
    engine     text NOT NULL,
    ok         boolean NOT NULL,
    latency_ms int,
    detail     jsonb NOT NULL DEFAULT '{{}}'::jsonb
);

CREATE INDEX IF NOT EXISTS {COMM_MAP_TABLE}_c   ON {COMM_MAP_TABLE} (level, community_id);
CREATE INDEX IF NOT EXISTS {COMMUNITY_TABLE}_h  ON {COMMUNITY_TABLE} (member_hash);
CREATE INDEX IF NOT EXISTS {QUERY_LOG_TABLE}_ts ON {QUERY_LOG_TABLE} (ts DESC);
"""

with db() as cur:
    cur.execute(GLOBAL_DDL)
print(f"✅ Section H schema ready: {COMMUNITY_TABLE}, {COMM_MAP_TABLE}, {QUERY_LOG_TABLE}")

## H.2 — Production plumbing

Three small pieces that separate a notebook from a system.

**`_llm()`** — every model call in this section goes through it. It disables thinking where
the model allows it (`thinking_budget = 0` on flash, `128` on `gemini-2.5-pro`, whose floor is
128 and which returns `400 INVALID_ARGUMENT` if you ask for 0), detects empty and truncated
responses, retries once at four times the budget, and **never returns an empty string** — an
empty response is a failure, and callers that treat it as data produce errors far from the
real cause.

**`_exclusive()`** — a PostgreSQL session-level advisory lock. Summarisation is the expensive
step; two people running it at once would pay twice and interleave their writes. The lock
makes the second one fail immediately with a clear message instead.

**`_log_query()`** — writes to `rag_query_log`. Logging failures must never break a query, so
it swallows its own errors.

In [ ]:
from google.genai.types import GenerateContentConfig as _GCC
try:
    from google.genai.types import ThinkingConfig as _TC
except ImportError:
    _TC = None

_THINK_OK = {}

def _llm(prompt, *, model=None, max_tokens=4096, json_out=False, label="call", _grow=True):
    """Call Gemini and return TEXT. Raises rather than returning an empty string."""
    model = model or SUMMARY_MODEL
    cfg = {"temperature": 0, "max_output_tokens": int(max_tokens)}
    if json_out:
        cfg["response_mime_type"] = "application/json"
    ladder = []
    if _TC is not None and _THINK_OK.get(model, True):
        ladder.append({**cfg, "thinking_config":
                       _TC(thinking_budget=128 if "pro" in model.lower() else 0)})
    ladder.append(cfg)

    resp, last = None, None
    for c in ladder:
        try:
            resp = _with_retry(lambda c=c: genai_client.models.generate_content(
                model=model, contents=prompt, config=_GCC(**c)))
            break
        except Exception as e:
            last = e
            if "thinking_config" in c and ("INVALID_ARGUMENT" in str(e) or "400" in str(e)):
                _THINK_OK[model] = False
                print(f"ℹ️  {model} rejects thinking_config — dropping it for this session.")
                continue
            raise
    if resp is None:
        raise last
    fr = str(getattr(resp.candidates[0], "finish_reason", "") or "") if resp.candidates else ""
    text = (resp.text or "").strip()
    if (not text or fr.upper().endswith("MAX_TOKENS")) and _grow:
        return _llm(prompt, model=model, max_tokens=int(max_tokens) * 4,
                    json_out=json_out, label=label, _grow=False)
    if not text:
        raise RuntimeError(f"{label}: {model} produced no text (finish_reason={fr or '?'}).")
    return text


@contextlib.contextmanager
def _exclusive(name):
    """Session-level advisory lock, so two runs cannot summarise the same communities."""
    conn = POOL.getconn()
    got = False
    try:
        with conn.cursor() as cur:
            cur.execute("SELECT pg_try_advisory_lock(hashtext(%s))", (name,))
            got = cur.fetchone()[0]
        conn.commit()
        if not got:
            raise RuntimeError(
                f"Another process is already running '{name}'. Wait for it, or if you are "
                f"sure it died: SELECT pg_advisory_unlock_all() on that connection.")
        yield
    finally:
        if got:
            with conn.cursor() as cur:
                cur.execute("SELECT pg_advisory_unlock(hashtext(%s))", (name,))
            conn.commit()
        POOL.putconn(conn)


def _log_query(question, engine, ok, latency_ms, detail=None):
    """Never let logging break a query."""
    try:
        with db() as cur:
            cur.execute(f"""INSERT INTO {QUERY_LOG_TABLE}
                            (question, engine, ok, latency_ms, detail)
                            VALUES (%s, %s, %s, %s, %s::jsonb)""",
                        (question[:4000], engine, bool(ok), int(latency_ms or 0),
                         json.dumps(detail or {})))
    except Exception:
        pass


def _member_hash(node_ids):
    return _hashlib.sha256("\u0000".join(sorted(node_ids)).encode()).hexdigest()

print("✅ Plumbing ready: _llm(), _exclusive(), _log_query()")

## H.3 — Clustering

Free, deterministic, seconds. Re-run it after every graph rebuild.

Two choices worth understanding.

**`value` nodes are excluded.** Those `"Produce"` / `"4.8/5"` / `"Active"` nodes are connected
to almost everything, so leaving them in produces one giant cluster and nothing useful. They
remain in the graph — local search still traverses them — they are simply not clustered.

**Summaries are preserved by membership hash.** Each cluster is fingerprinted by its exact
member list. On a re-run, any cluster whose fingerprint already has a summary keeps it. If your
graph is stable, re-clustering costs nothing; if half of it changed, you re-summarise half.

`networkx` gives markedly better clusters than the built-in fallback. The fallback is
deterministic label propagation, which is honest but merges communities joined by a single
edge — on a test graph of two triangles linked by one edge, networkx correctly found two
communities and the fallback found one. Install it.

In [ ]:
def build_communities(min_size=None, level=0, verbose=True):
    """Cluster the graph. Preserves summaries for communities that did not change."""
    min_size = COMMUNITY_MIN_SIZE if min_size is None else min_size
    excl = list(COMMUNITY_EXCLUDE_TYPES) or ['\u0000']

    with db(dict_rows=True) as cur:
        cur.execute(f"""SELECT e.src_id, e.dst_id
                        FROM {EDGE_TABLE} e
                        JOIN {NODE_TABLE} s ON s.node_id = e.src_id
                        JOIN {NODE_TABLE} t ON t.node_id = e.dst_id
                        WHERE s.node_type <> ALL(%s) AND t.node_type <> ALL(%s)""",
                    (excl, excl))
        edges = [(r["src_id"], r["dst_id"]) for r in cur.fetchall()]
    if not edges:
        print("No edges to cluster — run Sections G.3 to G.5 first.")
        return 0

    try:
        import networkx as nx
        G = nx.Graph(); G.add_edges_from(edges)
        groups = [sorted(g) for g in nx.community.greedy_modularity_communities(G)]
        how = "networkx greedy modularity"
    except ImportError:
        adj = {}
        for a, b in edges:
            adj.setdefault(a, set()).add(b); adj.setdefault(b, set()).add(a)
        groups = _label_propagation(adj)
        how = "label propagation (pip install networkx for better clusters)"

    groups = [g for g in groups if len(g) >= min_size]
    groups.sort(key=len, reverse=True)
    if not groups:
        print(f"No cluster reached min_size={min_size}. Lower it, or build more edges.")
        return 0

    # ---- carry summaries across, keyed on exact membership ----
    hashes = [_member_hash(g) for g in groups]
    with db(dict_rows=True) as cur:
        cur.execute(f"""SELECT member_hash, title, summary, rating, summary_embedding
                        FROM {COMMUNITY_TABLE}
                        WHERE summary IS NOT NULL AND member_hash = ANY(%s)""", (hashes,))
        kept = {r["member_hash"]: r for r in cur.fetchall()}

    with db() as cur:
        cur.execute(f"DELETE FROM {COMMUNITY_TABLE} WHERE level = %s", (level,))
        cur.execute(f"DELETE FROM {COMM_MAP_TABLE}  WHERE level = %s", (level,))
        execute_values(cur, f"""
            INSERT INTO {COMMUNITY_TABLE}
                (level, community_id, size, node_ids, member_hash,
                 title, summary, rating, summary_embedding)
            VALUES %s
        """, [(level, cid, len(g), g, h,
               (kept.get(h) or {}).get("title"),
               (kept.get(h) or {}).get("summary"),
               (kept.get(h) or {}).get("rating"),
               (kept.get(h) or {}).get("summary_embedding"))
              for cid, (g, h) in enumerate(zip(groups, hashes))], page_size=200)
        execute_values(cur, f"""
            INSERT INTO {COMM_MAP_TABLE} (node_id, level, community_id) VALUES %s
            ON CONFLICT (node_id, level) DO UPDATE SET community_id = EXCLUDED.community_id
        """, [(n, level, cid) for cid, g in enumerate(groups) for n in g], page_size=1000)

    if verbose:
        print(f"{len(groups)} communities via {how}")
        print(f"   {len(kept)} summary(ies) preserved, {len(groups) - len(kept)} to write")
        with db(dict_rows=True) as cur:
            for cid, g in list(enumerate(groups))[:8]:
                cur.execute(f"SELECT label FROM {NODE_TABLE} WHERE node_id = ANY(%s) LIMIT 5",
                            (g[:5],))
                names = ", ".join(r["label"][:22] for r in cur.fetchall())
                print(f"   #{cid:<3} {len(g):>4} nodes   {names}{' …' if len(g) > 5 else ''}")
        print(f"\nNext: summarize_communities()")
    return len(groups)


def _label_propagation(adj, rounds=25):
    """Deterministic fallback when networkx is unavailable.

    Each node adopts the most common label among its neighbours. Sorted iteration and
    a deterministic tie-break mean the result is identical on every run — unlike the
    usual randomised implementation, which would reshuffle your communities (and so
    invalidate every preserved summary) on each rebuild.
    """
    label = {n: n for n in sorted(adj)}
    for _ in range(rounds):
        changed = False
        for n in sorted(adj):
            if not adj[n]:
                continue
            counts = {}
            for m in adj[n]:
                counts[label[m]] = counts.get(label[m], 0) + 1
            best = sorted(counts.items(), key=lambda kv: (-kv[1], str(kv[0])))[0][0]
            if best != label[n]:
                label[n] = best; changed = True
        if not changed:
            break
    groups = {}
    for n, l in label.items():
        groups.setdefault(l, []).append(n)
    return [sorted(v) for v in groups.values()]

print("Clustering ready — build_communities()")

## H.4 — Summaries

**This is the step that costs money**: one LLM call per unsummarised community. It prints an
estimate and stops if you exceed `SUMMARY_MAX_CALLS`.

Each prompt gets three kinds of context, and the third is what makes the difference:
the community's **entities**, the **relationships** between them, and a few **representative
passages** — the chunks that mention the most members. Without passages the model summarises a
list of names and produces something generic; with them it summarises the actual documents and
quotes real figures.

Each finished summary is embedded, which is what makes `GLOBAL_PRESELECT_K` possible.

In [ ]:
COMMUNITY_PROMPT = """You are summarising one cluster of a business knowledge graph.

Write a report on this community: what it is about, who and what is in it, and what the
relationships between them mean for the business.

ENTITIES:
{entities}

RELATIONSHIPS:
{relationships}

REPRESENTATIVE PASSAGES:
{passages}

Return RAW JSON only:
{{"title": "<short specific name for this community, 3-8 words>",
  "summary": "<200-350 words. Lead with what this community IS. Then the important entities
              and what connects them. Then anything notable: risks, dependencies, conflicts,
              obligations, gaps. Use ONLY the information above. Name names and quote figures
              exactly as written.>",
  "rating": <0-10, how important this community is for understanding the business overall>}}"""


def _community_context(node_ids):
    """Entities, relationships and representative passages for one community."""
    ids = list(node_ids)[:COMMUNITY_MAX_NODES]
    with db(dict_rows=True) as cur:
        cur.execute(f"""SELECT node_id, node_type, label FROM {NODE_TABLE}
                        WHERE node_id = ANY(%s)""", (ids,))
        nodes = [dict(r) for r in cur.fetchall()]
        cur.execute(f"""SELECT s.label AS src, e.predicate, t.label AS dst
                        FROM {EDGE_TABLE} e
                        JOIN {NODE_TABLE} s ON s.node_id = e.src_id
                        JOIN {NODE_TABLE} t ON t.node_id = e.dst_id
                        WHERE e.src_id = ANY(%s) AND e.dst_id = ANY(%s)
                        LIMIT %s""", (ids, ids, COMMUNITY_MAX_EDGES))
        rels = [dict(r) for r in cur.fetchall()]
        cur.execute(f"""SELECT c.content, d.title, count(DISTINCT m.node_id) AS hits
                        FROM {MENTION_TABLE} m
                        JOIN {CHUNK_TABLE} c ON c.chunk_id = m.chunk_id
                        JOIN {DOC_TABLE}   d ON d.doc_id   = c.doc_id
                        WHERE m.node_id = ANY(%s)
                        GROUP BY c.chunk_id, c.content, d.title
                        ORDER BY hits DESC, length(c.content) DESC
                        LIMIT %s""", (ids, COMMUNITY_MAX_CHUNKS))
        chunks = [dict(r) for r in cur.fetchall()]

    ent = "\n".join(f"  - {n['label']} ({n['node_type']})" for n in nodes) or "  (none)"
    rel = "\n".join(f"  - {r['src'][:40]} --[{r['predicate']}]--> {r['dst'][:40]}"
                     for r in rels) or "  (none)"
    psg = "\n\n".join(f"  [{c['title']}] {c['content'][:900]}" for c in chunks) or "  (none)"
    return ent, rel, psg


def summarize_communities(level=0, only_missing=True, max_calls=None, dry_run=False,
                          verbose=True):
    """One LLM call per community. Resumable, capped, and safe to run twice."""
    max_calls = SUMMARY_MAX_CALLS if max_calls is None else max_calls
    with db(dict_rows=True) as cur:
        cur.execute(f"""SELECT community_id, node_ids, child_ids, summary
                        FROM {COMMUNITY_TABLE} WHERE level = %s ORDER BY size DESC""", (level,))
        comms = [dict(r) for r in cur.fetchall()]
    todo = [c for c in comms if not (only_missing and c["summary"])]
    if max_calls:
        todo = todo[:int(max_calls)]

    print(f"level {level}: {len(comms)} communities, {len(todo)} to summarise "
          f"→ {len(todo)} LLM call(s) + {len(todo)} embedding(s)")
    if dry_run:
        print("DRY RUN — nothing sent. Call without dry_run=True to proceed.")
        return 0
    if not todo:
        print("Nothing to do.")
        return 0

    def _one(c):
        try:
            if level == 0:
                ent, rel, psg = _community_context(c["node_ids"])
            else:
                ent, rel, psg = _hierarchy_context(c["child_ids"])
            data = _parse_json_reply(_llm(
                COMMUNITY_PROMPT.format(entities=ent, relationships=rel, passages=psg),
                max_tokens=4096, json_out=True, label="community summary"))
            title   = str(data.get("title", ""))[:300]
            summary = str(data.get("summary", ""))
            rating  = float(data.get("rating") or 5)
            vec = embed_texts([f"{title}\n{summary}"], "RETRIEVAL_DOCUMENT")[0]
            return c["community_id"], title, summary, rating, vec
        except Exception as e:
            print(f"  ⚠️  community {c['community_id']}: {type(e).__name__}: {str(e)[:130]}")
            return None

    t0 = _time.perf_counter()
    with _exclusive(f"graphrag_summarize_level_{level}"):
        with ThreadPoolExecutor(max_workers=SUMMARY_WORKERS) as pool:
            out = [r for r in pool.map(_one, todo) if r]
        with db() as cur:
            for cid, title, summary, rating, vec in out:
                cur.execute(f"""UPDATE {COMMUNITY_TABLE}
                                SET title=%s, summary=%s, rating=%s, summary_embedding=%s::vector
                                WHERE level=%s AND community_id=%s""",
                            (title, summary, rating, vec, level, cid))

    if verbose:
        for cid, title, _, rating, _v in sorted(out, key=lambda x: -x[3])[:10]:
            print(f"   #{cid:<3} rating {rating:>4.1f}   {title}")
    print(f"✅ Summarised {len(out)}/{len(todo)} in {_time.perf_counter() - t0:.0f}s")
    with db(dict_rows=True) as cur:
        cur.execute(f"SELECT count(*) AS n FROM {COMMUNITY_TABLE} "
                    f"WHERE level=%s AND summary IS NULL", (level,))
        left = cur.fetchone()["n"]
    if left:
        print(f"   {left} still unsummarised — raise max_calls and run again.")
    return len(out)

print("Summaries ready — summarize_communities(dry_run=True) to see the cost first")

## H.5 — Optional: a second level

Level 0 communities are detailed and specific. For a very broad question — *"what is this
business exposed to?"* — mapping over dozens of them is expensive and the answer drowns in
detail.

Level 1 clusters the **communities themselves**: two communities are linked if any edge joins
a node in one to a node in the other. Each level-1 report is written from its children's
summaries, so it needs no chunk reading and is cheap.

This is optional. Build it if you have more than ~20 level-0 communities, then set
`GLOBAL_LEVEL = 1` for broad questions. With fewer than that, level 0 is better — more
specific, and the cost difference is negligible.

In [ ]:
def build_hierarchy(min_size=2, verbose=True):
    """Cluster the communities themselves into level-1 super-communities."""
    with db(dict_rows=True) as cur:
        cur.execute(f"""SELECT community_id, node_ids FROM {COMMUNITY_TABLE} WHERE level = 0""")
        base = {r["community_id"]: list(r["node_ids"]) for r in cur.fetchall()}
        if len(base) < 2 * min_size:
            print(f"Only {len(base)} level-0 communities — a hierarchy would not help.")
            return 0
        # Two communities are linked when any edge crosses between them.
        cur.execute(f"""
            SELECT a.community_id AS c1, b.community_id AS c2, count(*)::int AS n
            FROM {EDGE_TABLE} e
            JOIN {COMM_MAP_TABLE} a ON a.node_id = e.src_id AND a.level = 0
            JOIN {COMM_MAP_TABLE} b ON b.node_id = e.dst_id AND b.level = 0
            WHERE a.community_id <> b.community_id
            GROUP BY 1, 2
        """)
        links = [(r["c1"], r["c2"]) for r in cur.fetchall()]
    if not links:
        print("No edges cross between communities — nothing to group.")
        return 0

    try:
        import networkx as nx
        G = nx.Graph(); G.add_edges_from(links)
        groups = [sorted(g) for g in nx.community.greedy_modularity_communities(G)]
    except ImportError:
        adj = {}
        for a, b in links:
            adj.setdefault(a, set()).add(b); adj.setdefault(b, set()).add(a)
        groups = _label_propagation(adj)

    groups = [g for g in groups if len(g) >= min_size]
    groups.sort(key=len, reverse=True)
    if not groups:
        print(f"No super-community reached min_size={min_size}.")
        return 0

    rows = []
    for cid, children in enumerate(groups):
        members = sorted({n for c in children for n in base.get(c, [])})
        rows.append((1, cid, len(members), members, list(children), _member_hash(members)))
    hashes = [r[5] for r in rows]
    with db(dict_rows=True) as cur:
        cur.execute(f"""SELECT member_hash, title, summary, rating, summary_embedding
                        FROM {COMMUNITY_TABLE}
                        WHERE level = 1 AND summary IS NOT NULL AND member_hash = ANY(%s)""",
                    (hashes,))
        kept = {r["member_hash"]: r for r in cur.fetchall()}

    with db() as cur:
        cur.execute(f"DELETE FROM {COMMUNITY_TABLE} WHERE level = 1")
        cur.execute(f"DELETE FROM {COMM_MAP_TABLE}  WHERE level = 1")
        execute_values(cur, f"""
            INSERT INTO {COMMUNITY_TABLE}
                (level, community_id, size, node_ids, child_ids, member_hash,
                 title, summary, rating, summary_embedding)
            VALUES %s
        """, [(*r, (kept.get(r[5]) or {}).get("title"),
                   (kept.get(r[5]) or {}).get("summary"),
                   (kept.get(r[5]) or {}).get("rating"),
                   (kept.get(r[5]) or {}).get("summary_embedding")) for r in rows],
           page_size=100)
        execute_values(cur, f"""
            INSERT INTO {COMM_MAP_TABLE} (node_id, level, community_id) VALUES %s
            ON CONFLICT (node_id, level) DO UPDATE SET community_id = EXCLUDED.community_id
        """, [(n, 1, r[1]) for r in rows for n in r[3]], page_size=1000)

    if verbose:
        print(f"{len(groups)} level-1 super-communities "
              f"({len(kept)} summary(ies) preserved)")
        for cid, children in list(enumerate(groups))[:8]:
            print(f"   L1 #{cid:<3} contains level-0 communities {children[:8]}")
        print("\nNext: summarize_communities(level=1)   then set GLOBAL_LEVEL = 1")
    return len(groups)


def _hierarchy_context(child_ids):
    """A level-1 report is written from its children's reports — no chunk reading needed."""
    with db(dict_rows=True) as cur:
        cur.execute(f"""SELECT community_id, title, summary, rating FROM {COMMUNITY_TABLE}
                        WHERE level = 0 AND community_id = ANY(%s) AND summary IS NOT NULL
                        ORDER BY rating DESC NULLS LAST""", (list(child_ids or []),))
        kids = [dict(r) for r in cur.fetchall()]
    ent = "\n".join(f"  - Community {k['community_id']}: {k['title']}" for k in kids) or "  (none)"
    psg = "\n\n".join(f"  [{k['title']}] {(k['summary'] or '')[:1200]}" for k in kids) or "  (none)"
    return ent, "  (see the child reports below)", psg

print("Hierarchy ready — build_hierarchy() (optional)")

## H.6 — Global search

Pre-select, map, reduce.

**Pre-selection is the part most tutorials leave out**, and it is what makes this affordable.
Naively, every question maps over every community summary: 200 communities is 40 LLM calls
*per question*. Instead the question is embedded and compared against the summary embeddings,
and only the `GLOBAL_PRESELECT_K` closest reports are mapped. Cost stops growing with corpus
size.

Set `GLOBAL_PRESELECT_K = None` to map over everything — more thorough, and the right choice
while you have only a handful of communities.

**The map prompt is explicitly told that returning nothing is correct.** Without that,
models manufacture relevance for every report they are shown and the reduce step drowns in
weak points.

In [ ]:
MAP_PROMPT = """You are answering a question using summaries of communities in a knowledge graph.

From the COMMUNITY REPORTS below, extract the points that help answer the QUESTION.

Score each point 0-100 for how directly and importantly it helps answer THIS question.
If a report is irrelevant, produce no point for it. An empty list is a correct and expected
answer when nothing is relevant — do not invent relevance.

Each point must be self-contained: name the entities, quote the figures. The writer of the
final answer will not see these reports, only your points.

Return RAW JSON only:
{{"points": [{{"description": "<specific, self-contained statement>", "score": 85}}]}}

QUESTION: {question}

COMMUNITY REPORTS:
{reports}"""

REDUCE_PROMPT = """You are a business analyst writing a final answer.

Below are key points gathered from across the entire document corpus, each with an importance
score. Synthesise them into one coherent answer to the QUESTION.

Rules:
- Use ONLY these points. Add nothing from outside them.
- Organise by theme, not by score. Lead with what matters most.
- Where points conflict, say so rather than silently picking one.
- Be specific: name names, quote figures exactly as given.
- If the points do not answer the question, reply exactly: "{refusal}"

QUESTION: {question}

KEY POINTS:
{points}"""


def _select_communities(question, level, min_rating, k):
    """The K most relevant reports, by embedding similarity. Falls back to all of them."""
    if k:
        try:
            qv = embed_query(question)
            with db(dict_rows=True) as cur:
                cur.execute(f"""SELECT community_id, title, summary, rating, size,
                                       (summary_embedding <=> %(qv)s::vector) AS dist
                                FROM {COMMUNITY_TABLE}
                                WHERE level = %(lv)s AND summary IS NOT NULL
                                  AND summary_embedding IS NOT NULL
                                  AND COALESCE(rating, 0) >= %(mr)s
                                ORDER BY summary_embedding <=> %(qv)s::vector
                                LIMIT %(k)s""",
                            {"qv": qv, "lv": level, "mr": min_rating, "k": int(k)})
                rows = [dict(r) for r in cur.fetchall()]
            if rows:
                return rows, "preselected"
        except Exception as e:
            print(f"  ⚠️  preselection unavailable ({type(e).__name__}) — using all reports.")
    with db(dict_rows=True) as cur:
        cur.execute(f"""SELECT community_id, title, summary, rating, size
                        FROM {COMMUNITY_TABLE}
                        WHERE level = %s AND summary IS NOT NULL
                          AND COALESCE(rating, 0) >= %s
                        ORDER BY rating DESC NULLS LAST, size DESC""", (level, min_rating))
        return [dict(r) for r in cur.fetchall()], "all"


def global_search(question, level=None, min_rating=None, preselect_k=..., verbose=True):
    """Map-reduce over community summaries. For questions about the corpus as a whole."""
    level      = GLOBAL_LEVEL if level is None else level
    min_rating = GLOBAL_MIN_RATING if min_rating is None else min_rating
    k          = GLOBAL_PRESELECT_K if preselect_k is ... else preselect_k
    timings, t = {}, _time.perf_counter
    started = t()

    comms, how = _select_communities(question, level, min_rating, k)
    if not comms:
        msg = (f"No community summaries at level {level}. Run build_communities() and "
               f"summarize_communities()" + (" and build_hierarchy()" if level else "") + ".")
        _log_query(question, "global", False, (t() - started) * 1000, {"reason": "no summaries"})
        return {"answer": msg, "points": [], "communities_read": 0,
                "timings": timings, "refused": True}

    batches = [comms[i:i + GLOBAL_BATCH_SIZE] for i in range(0, len(comms), GLOBAL_BATCH_SIZE)]
    if verbose:
        print(f"   map: {len(comms)} report(s) ({how}, level {level}) "
              f"in {len(batches)} batch(es)…")

    def _map(batch):
        reports = "\n\n".join(
            f"--- Community {c['community_id']}: {c['title']} (rating {c['rating']}) ---\n"
            f"{c['summary']}" for c in batch)
        try:
            data = _parse_json_reply(_llm(
                MAP_PROMPT.format(question=question, reports=reports),
                max_tokens=4096, json_out=True, label="global map"))
            return [p for p in (data.get("points") or [])
                    if isinstance(p, dict) and p.get("description")]
        except Exception as e:
            print(f"  ⚠️  map batch failed: {type(e).__name__}: {str(e)[:120]}")
            return []

    t0 = t()
    with ThreadPoolExecutor(max_workers=GLOBAL_WORKERS) as pool:
        points = [p for b in pool.map(_map, batches) for p in b]
    timings["map_ms"] = (t() - t0) * 1000

    def _score(p):
        try:
            return float(p.get("score") or 0)
        except (TypeError, ValueError):
            return 0.0

    points.sort(key=_score, reverse=True)
    top = points[:GLOBAL_TOP_POINTS]
    if verbose:
        print(f"   {len(points)} key point(s), keeping top {len(top)}")
    if not top:
        _log_query(question, "global", False, (t() - started) * 1000,
                   {"communities": len(comms), "points": 0})
        return {"answer": REFUSAL, "points": [], "communities_read": len(comms),
                "timings": timings, "refused": True}

    t0 = t()
    answer = _llm(REDUCE_PROMPT.format(
        question=question, refusal=REFUSAL,
        points="\n".join(f"  - ({_score(p):.0f}) {p['description']}" for p in top)),
        max_tokens=4096, label="global reduce").strip()
    timings["reduce_ms"] = (t() - t0) * 1000

    refused = answer.startswith(REFUSAL)
    _log_query(question, "global", not refused, (t() - started) * 1000,
               {"level": level, "selection": how, "communities": len(comms),
                "points_found": len(points), "points_used": len(top)})
    return {"answer": answer, "points": top, "communities_read": len(comms),
            "selection": how, "level": level, "timings": timings, "refused": refused}


def show_global(result):
    print(result["answer"])
    print(f"\nSynthesised from {result.get('communities_read', 0)} community report(s) "
          f"({result.get('selection', '?')}, level {result.get('level', 0)}); "
          f"{len(result.get('points', []))} key points used.")
    for p in result.get("points", [])[:6]:
        try:
            s = f"{float(p.get('score') or 0):.0f}"
        except (TypeError, ValueError):
            s = "?"
        print(f"  ({s}) {p['description'][:110]}")
    print("  timings:", {k: round(v) for k, v in result["timings"].items()}, "ms")
    return result

print("Global search ready — global_search(\"...\")")

## H.7 — Wiring into `ask()`

`ask()` keeps a reference to your existing router-driven version and delegates everything that
is not a corpus-wide question. Your four engines behave exactly as before.

The global check is **two-stage on purpose**. A regex runs first; if the question has no
corpus-wide phrasing at all, no model call happens and there is zero added latency or cost. The
second stage exists because *"what are the payment terms across all three bids?"* matches the
regex but is still a local question — the model settles it.

And every question, whichever engine answers, is now recorded in `rag_query_log`.

In [ ]:
_ask_engines = ask          # your Section C ask() — unchanged


def _is_global(question, verbose=True):
    if not _GLOBAL_HINTS.search(question or ""):
        return False                     # stage 1: free
    try:                                 # stage 2: one cheap call, only when it might matter
        d = _parse_json_reply(_llm(
            'Does this question ask about a document corpus AS A WHOLE (themes, patterns, '
            'recurring issues, "across all…"), or about SPECIFIC named things?\n\n'
            f'QUESTION: {question}\n\n'
            'Return RAW JSON only: {"global": true or false, "reason": "<short>"}',
            max_tokens=512, json_out=True, label="global check"))
        return bool(d.get("global"))
    except Exception as e:
        if verbose:
            print(f"⚠️  global check failed ({type(e).__name__}) — treating as local.")
        return False


def ask(question, mode="auto", verbose=True, **kwargs):
    """The one entry point. mode = "auto" | "sql" | "rag" | "graph" | "global"."""
    started = _time.perf_counter()

    if mode == "global":
        if verbose:
            print("🌍 forced to GLOBAL")
        return show_global(global_search(question, verbose=verbose))

    if mode == "auto" and _is_global(question, verbose=verbose):
        with db(dict_rows=True) as cur:
            cur.execute(f"""SELECT count(*) AS n FROM {COMMUNITY_TABLE}
                            WHERE level = %s AND summary IS NOT NULL""", (GLOBAL_LEVEL,))
            ready = cur.fetchone()["n"]
        if ready:
            if verbose:
                print(f"🌍 routed to GLOBAL — corpus-wide question, {ready} report(s) available.")
            return show_global(global_search(question, verbose=verbose))
        if verbose:
            print("🌍 This looks corpus-wide, but no community summaries exist yet.")
            print("   Run build_communities() then summarize_communities(). "
                  "Falling back to the local engines.")

    try:
        result = _ask_engines(question, mode=mode, verbose=verbose, **kwargs)
        ok = result is not None
    except Exception:
        _log_query(question, mode, False, (_time.perf_counter() - started) * 1000,
                   {"error": "engine raised"})
        raise
    _log_query(question, mode, ok, (_time.perf_counter() - started) * 1000, {})
    return result


# ---------------------------------------------------------------- inspectors
def communities(level=0, top=20):
    with db(dict_rows=True) as cur:
        cur.execute(f"""SELECT community_id, size, rating, title,
                               (summary IS NOT NULL) AS done,
                               (summary_embedding IS NOT NULL) AS embedded
                        FROM {COMMUNITY_TABLE} WHERE level = %s
                        ORDER BY rating DESC NULLS LAST, size DESC LIMIT %s""", (level, top))
        rows = [dict(r) for r in cur.fetchall()]
    for r in rows:
        flags = ("" if r["done"] else "  ⚠️ not summarised") + \
                ("" if r["embedded"] or not r["done"] else "  ⚠️ not embedded")
        print(f"  L{level} #{r['community_id']:<3} size={r['size']:<4} "
              f"rating={r['rating']} {(r['title'] or '(untitled)')[:48]}{flags}")
    return rows


def community_report(cid, level=0):
    """Read one report in full — how you sanity-check the synthesis before trusting it."""
    with db(dict_rows=True) as cur:
        cur.execute(f"SELECT * FROM {COMMUNITY_TABLE} WHERE level=%s AND community_id=%s",
                    (level, cid))
        c = cur.fetchone()
        if not c:
            print(f"No community {cid} at level {level}."); return None
        cur.execute(f"""SELECT node_type, label FROM {NODE_TABLE}
                        WHERE node_id = ANY(%s) ORDER BY node_type LIMIT 30""",
                    (list(c["node_ids"]),))
        members = cur.fetchall()
    print(f"L{level} COMMUNITY #{cid} — {c['title']}   "
          f"(size {c['size']}, rating {c['rating']})\n")
    print(c["summary"] or "(not summarised)")
    print("\nMembers:")
    for m in members:
        print(f"  {m['node_type']:<16} {m['label'][:56]}")
    return dict(c)


def query_log(limit=25):
    """What has been asked, which engine answered, how long it took, did it work."""
    with db(dict_rows=True) as cur:
        cur.execute(f"""SELECT ts, engine, ok, latency_ms, left(question, 70) AS question
                        FROM {QUERY_LOG_TABLE} ORDER BY ts DESC LIMIT %s""", (limit,))
        rows = [dict(r) for r in cur.fetchall()]
    return pd.DataFrame(rows) if rows else print("No queries logged yet.")


def graphrag_health():
    """One call that tells you whether Section H is ready and where it is not."""
    with db(dict_rows=True) as cur:
        for label, sql in [
            ("nodes",        f"SELECT count(*) AS n FROM {NODE_TABLE}"),
            ("edges",        f"SELECT count(*) AS n FROM {EDGE_TABLE}"),
            ("mentions",     f"SELECT count(*) AS n FROM {MENTION_TABLE}"),
            ("communities",  f"SELECT count(*) AS n FROM {COMMUNITY_TABLE} WHERE level=0"),
            ("summarised",   f"SELECT count(*) AS n FROM {COMMUNITY_TABLE} "
                             f"WHERE level=0 AND summary IS NOT NULL"),
            ("embedded",     f"SELECT count(*) AS n FROM {COMMUNITY_TABLE} "
                             f"WHERE level=0 AND summary_embedding IS NOT NULL"),
            ("L1 groups",    f"SELECT count(*) AS n FROM {COMMUNITY_TABLE} WHERE level=1"),
            ("queries",      f"SELECT count(*) AS n FROM {QUERY_LOG_TABLE}"),
        ]:
            cur.execute(sql)
            print(f"  {label:<14} {cur.fetchone()['n']:>7,}")
        cur.execute(f"SELECT count(*) AS n FROM {COMMUNITY_TABLE} "
                    f"WHERE level=0 AND summary IS NULL")
        missing = cur.fetchone()["n"]
    if missing:
        print(f"\n  ⚠️  {missing} community(ies) unsummarised — global search will miss them.")
    else:
        print("\n  ✅ global search is ready")


print("\n✅ Section H wired in.")
print("   ask(\"...\")                 corpus-wide questions now route to GLOBAL")
print("   ask(\"...\", mode=\"global\")  force it")
print("   communities() · community_report(0) · query_log() · graphrag_health()")

## H.8 — Build order

Run once, in this order. Step 1 is free; step 2 is where the money goes.

Re-run **1** after any graph rebuild — it is free and it preserves summaries for unchanged
communities. Re-run **2** afterwards to fill in only what changed.

In [ ]:
# 1. Cluster the graph — free, seconds, deterministic.
# build_communities()

# 2. See what summarising would cost, THEN pay for it.
# summarize_communities(dry_run=True)
# summarize_communities()

# 3. Optional: a broader second level, only worth it above ~20 communities.
# build_hierarchy()
# summarize_communities(level=1)
# GLOBAL_LEVEL = 1        # then broad questions use the shorter, cheaper reports

graphrag_health()

In [ ]:
# LOCAL — unchanged, routed by your existing router.
ask("What happens to the Valleverde contract if their cold-chain certification lapses?")

In [ ]:
# GLOBAL — the question that had no answer before this section existed.
ask("What are the recurring compliance and delivery risks across all suppliers?")

In [ ]:
# Sanity-check the synthesis before you trust it: read a report in full,
# and check what has been asked so far.
# community_report(0)
# communities()
# query_log()

## H.9 — Operating it

**Cost.** One LLM call per community at build time, plus one embedding. Queries cost
`ceil(GLOBAL_PRESELECT_K / GLOBAL_BATCH_SIZE) + 1` calls — with the defaults, four. Without
pre-selection that grows with your corpus; with it, it does not.

**When to rebuild.** Documents changed → re-run ingest, then `build_relational_graph()` /
`build_metadata_graph()` / `link_graph_to_chunks()`, then `build_communities()`, then
`summarize_communities()`. Only communities whose membership actually changed are re-summarised.

**How to tell it is working.** `graphrag_health()` shows whether every community is summarised
and embedded. `query_log()` shows which engine answered what, and how long it took — that table
is how "the answers feel wrong" becomes a question you can actually investigate.

**Where it goes wrong, and what it means:**

| Symptom | Cause | Fix |
|---|---|---|
| One giant community | hub nodes still being clustered | add their `node_type` to `COMMUNITY_EXCLUDE_TYPES` |
| Every community rated 8–10 | the model has no contrast to judge against | normal below ~10 communities; ignore the rating and set `GLOBAL_MIN_RATING = 0` |
| Global answers are vague | summaries were written from labels, not text | check `MENTION_TABLE` is populated — no mentions means no passages in the prompt |
| Global search returns the refusal | pre-selection picked the wrong reports | raise `GLOBAL_PRESELECT_K`, or set it to `None` |
| Summaries vanish after a rebuild | clustering changed membership | expected; only unchanged communities are preserved by design |
| `RuntimeError: Another process is already running` | the advisory lock | someone else is summarising — wait for them |

**Before production.** Move the database password out of Section 0.2 into an environment
variable and rotate it. Give the application a read-only role. And keep `rag_query_log` — it is
the cheapest observability you will ever add, and the questions it captures become the
evaluation set that tells you whether any future change actually helped.

## C.1 — Try it

The first two should route to **SQL**, the third to **RAG**, the fourth should be refused.
Replace them with your own questions once you have seen the routing work.

In [ ]:
ask("What is the name of the supplier having the highest total amount, and for which category?")

In [ ]:
ask("How many RFPs are there in total?")

In [ ]:
# Document-shaped question -> RAG. This one has exactly one right answer in the
# Section B.3b corpus, so a wrong retrieval is immediately obvious.
ask("Why did Acme Foods lose the produce award even though its bid was cheaper?")

In [ ]:
# Exact-token lookup — the case pure vector search loses and the keyword leg wins.
ask("What is the MSC Chain of Custody certificate number for Harbor Point Seafood, and when does it expire?")

In [ ]:
# The trap: 4.8/5 is ONE supplier's rating, 8.5/10 is the district-wide index.
# A careless system collapses them into a single number. This should keep them apart.
ask("What was the customer satisfaction level in the Q1 supplier performance review?")

In [ ]:
# Cross-document question — the answer is assembled from the contract, the compliance
# register and the email thread, all of which must be retrieved and attributed.
ask("When does Tidewater's Stockton SQF certificate expire and what happens if it lapses?")

In [ ]:
# Relationship question -> should route to GRAPH.
ask("Which supplier is connected to Harbor Point Seafood, and what does that mean for the frozen contract?")

In [ ]:
# Consequence / dependency question -> should route to GRAPH.
ask("What depends on CERT-TW-SQF2-STK and what happens if it expires?")

In [ ]:
# Neither engine can answer this -> refused, not invented.
ask("What is the capital of France?")

In [ ]:
# Forcing an engine, and passing engine-specific options through:
# ask("What is the total bids amount?", mode="sql", on_ambiguity="both")
# ask("What are the delivery requirements?", mode="rag", filters={"category": "Dairy"})
# ask("Which customer spent the most in total?", mode="sql", execute=False)  # build SQL, don't run
print("See the commented examples above for per-engine options.")

---
# Section D — Optional: testing, evaluation, benchmarking

**Everything below is commented out.** None of it is needed to answer questions — it is how
you *prove* the system works and *justify* the configuration to someone else. Uncomment a
cell when you want to run it.

| | What it gives you |
|---|---|
| **D.1** | Guardrail self-test — watch dangerous SQL get rejected |
| **D.2** | Retrieval quality — recall@k, nDCG@k, per-stage latency, hybrid vs vector vs text |
| **D.3** | Answer quality — LLM-as-judge on groundedness / correctness / completeness |
| **D.4** | Config sweep — 3 chunkers × models × dimensions, the evidence behind your settings |

## D.1 — Guardrail self-test (SQL engine)

Uncomment and run. Every line should say **REJECTED**. If any says PASSED, stop and fix it
before pointing this at anything you care about.

In [ ]:
# ── OPTIONAL: uncomment the whole cell to run the guardrail self-test ──
# _bad = [
#     ("DELETE FROM " + (KNOWN_TABLES[0] if KNOWN_TABLES else "x"), "a write statement"),
#     ("SELECT 1; DROP TABLE users",                                "hidden second statement"),
#     ("SELECT * FROM table_that_does_not_exist_xyz",               "nonexistent table"),
#     ("SELECT 2+2",                                                "no table from your DB"),
#     ("UPDATE t SET a=1",                                          "an UPDATE"),
# ]
# if KNOWN_TABLES:
#     t0 = KNOWN_TABLES[0]
#     _bad.append((f"SELECT z.made_up_column_xyz FROM {t0} z", "invented column on a real table"))
#
# print("Guardrail self-test:")
# all_ok = True
# for q, why in _bad:
#     ok, reason = validate_sql(q)
#     verdict = "❌ PASSED (BUG!)" if ok else "✅ REJECTED"
#     if ok:
#         all_ok = False
#     print(f"  {verdict}  [{why}]  →  {reason if reason else q}")
# print("\n" + ("✅ All guardrails working." if all_ok else "❌ Some guardrail failed — do NOT use, re-check steps."))

## D.2 — Retrieval evaluation (RAG engine)

`GOLD` is your labelled question set: which documents legitimately answer each question, and
the literal string that must appear in a retrieved chunk. `must_contain` is what catches
*right document, wrong chunk* — the failure mode that recall@k alone hides.

Edit `GOLD` to reflect **your** corpus before trusting the numbers.

In [ ]:
# ── OPTIONAL: the labelled evaluation set, written against the Section B.3b corpus. ──
# # doc_ids      : every document that legitimately answers the question.
# # must_contain : the literal string that must appear in a retrieved chunk for the answer to
# #                be derivable — this is what catches "right document, wrong chunk".
# GOLD = [
#     {"q": "What did Valle Verde quote for romaine lettuce?",
#      "doc_ids": ["bid-2024-089-valleverde"], "must_contain": ["3.20"]},
#     {"q": "What was Valle Verde's customer satisfaction rating in the Q1 FY2025 review?",
#      "doc_ids": ["perf-review-q1-fy2025"], "must_contain": ["4.8"]},
#     {"q": "What was the overall district satisfaction index for Q1 FY2025?",
#      "doc_ids": ["perf-review-q1-fy2025"], "must_contain": ["8.5"]},
#     {"q": "What were Globex Dairy's payment terms?",
#      "doc_ids": ["bid-2024-104-globex"], "must_contain": ["Net-45"]},
#     {"q": "Which suppliers offered an early payment discount?",
#      "doc_ids": ["bid-2024-104-globex", "bid-2024-117-tidewater",
#                  "bid-2024-118-northlake", "bid-2024-091-acme"],
#      "must_contain": ["early payment discount"]},
#     {"q": "What are the evaluation criteria weightings for RFP-2024-0412?",
#      "doc_ids": ["rfp-2024-0412-produce"], "must_contain": ["40%"]},
#     {"q": "Why did Acme Foods lose the produce award despite bidding lower?",
#      "doc_ids": ["award-memo-rfp-0412"], "must_contain": ["14.2"]},
#     {"q": "Which supplier had the lowest on-time delivery rate in Q1 FY2025?",
#      "doc_ids": ["perf-review-q1-fy2025"], "must_contain": ["92.0%"]},
#     {"q": "When does Tidewater's Stockton SQF certificate expire?",
#      "doc_ids": ["cert-compliance-register-2024", "supplier-profile-tidewater",
#                  "contract-ctr-2024-0489-tfh"], "must_contain": ["2025-04-30"]},
#     {"q": "Who owns Harbor Point Seafood?",
#      "doc_ids": ["supplier-profile-tidewater", "bid-2024-117-tidewater"],
#      "must_contain": ["Tidewater"]},
#     {"q": "What is the MSC Chain of Custody certificate number for Harbor Point Seafood?",
#      "doc_ids": ["cert-compliance-register-2024", "supplier-profile-tidewater",
#                  "bid-2024-117-tidewater"], "must_contain": ["MSC-C-58817"]},
#     {"q": "What liquidated damages apply if on-time delivery falls below 92% under CTR-2024-0412-VV?",
#      "doc_ids": ["contract-ctr-2024-0412-vv"], "must_contain": ["2%"]},
#     {"q": "What did Addendum No. 1 change about the delivery window?",
#      "doc_ids": ["addendum-rfp-0412-001"], "must_contain": ["4:30 AM"]},
#     {"q": "What was the total value of Tidewater's bid?",
#      "doc_ids": ["bid-2024-117-tidewater"], "must_contain": ["2,984,220"]},
#     {"q": "What is the price of pollock fillet SKU HP-SEA-7702?",
#      "doc_ids": ["bid-2024-117-tidewater"], "must_contain": ["0.8140"]},
#     # Unanswerable — the system must refuse, not invent.
#     {"q": "What is Valle Verde's parental leave policy?",
#      "doc_ids": [], "must_contain": [], "unanswerable": True},
# ]
#
# print(f"{len(GOLD)} evaluation questions "
#       f"({sum(1 for g in GOLD if g.get('unanswerable'))} unanswerable).")

In [ ]:
# ── OPTIONAL: recall@k / nDCG@k / latency, per retrieval mode ──
# def _ndcg(ranked_docs, gold_docs, k):
#     """Binary-relevance nDCG@k - rewards ranking a correct doc first, not merely including it."""
#     dcg = sum(1.0 / math.log2(i + 2) for i, d in enumerate(ranked_docs[:k]) if d in gold_docs)
#     ideal = sum(1.0 / math.log2(i + 2) for i in range(min(len(gold_docs), k)))
#     return dcg / ideal if ideal else 0.0
#
# def evaluate(mode: str = "hybrid", use_reranker: bool = False, k: int = RERANK_TOP_N):
#     """Retrieval-quality metrics with per-stage latency.
#
#     Uses the UNCACHED embedder so embed_ms reflects a real cold request. Every config is
#     scored on the same final list size k, so reranked and non-reranked runs compare fairly.
#     """
#     answerable = [g for g in GOLD if not g.get("unanswerable")]
#     recall = mrr = ndcg = support = 0.0
#     t_embed = t_search = t_rerank = 0.0
#
#     for g in answerable:
#         t0 = time.perf_counter()
#         qv = embed_query(g["q"])
#         t_embed += (time.perf_counter() - t0) * 1000
#
#         t0 = time.perf_counter()
#         hits = retrieve(g["q"], qv=qv, mode=mode,
#                         top_k=RERANK_CANDIDATES if use_reranker else k)
#         t_search += (time.perf_counter() - t0) * 1000
#
#         if use_reranker:
#             t0 = time.perf_counter()
#             hits = rerank(g["q"], hits, top_n=k)
#             t_rerank += (time.perf_counter() - t0) * 1000
#         hits = hits[:k]
#
#         ranked = [h["doc_id"] for h in hits]
#         gold_docs = set(g["doc_ids"])
#         if gold_docs & set(ranked):
#             recall += 1
#             first = next(i for i, d in enumerate(ranked) if d in gold_docs)
#             mrr += 1.0 / (first + 1)
#         ndcg += _ndcg(ranked, gold_docs, k)
#         # answer_support: is the answer string actually present in what we retrieved?
#         blob = " ".join(h["content"] for h in hits).lower()
#         if all(s.lower() in blob for s in g["must_contain"]):
#             support += 1
#
#     n = len(answerable)
#     return {
#         "mode": mode + (" + rerank" if use_reranker else ""),
#         "recall": recall / n, "mrr": mrr / n, "ndcg": ndcg / n,
#         "support": support / n,
#         "embed_ms": t_embed / n, "search_ms": t_search / n,
#         "rerank_ms": (t_rerank / n) if use_reranker else 0.0,
#     }
#
# rows = []
# for mode, rr in [("text", False), ("vector", False), ("hybrid", False), ("hybrid", True)]:
#     r = evaluate(mode, rr)
#     rows.append(r)
#     print(f"  done: {r['mode']}")
#
# k = RERANK_TOP_N
# hdr = (f"{'configuration':<20}{'recall@' + str(k):>10}{'mrr':>8}{'ndcg@' + str(k):>9}"
#        f"{'support':>9}{'embed ms':>10}{'search ms':>11}{'rerank ms':>11}")
# print("\n" + hdr)
# print("-" * len(hdr))
# for r in rows:
#     print(f"{r['mode']:<20}{r['recall']:>10.2f}{r['mrr']:>8.2f}{r['ndcg']:>9.2f}"
#           f"{r['support']:>9.2f}{r['embed_ms']:>10.0f}{r['search_ms']:>11.1f}"
#           f"{r['rerank_ms']:>11.0f}")
#
# print("\n`search ms` is now its own column - a few milliseconds of Postgres work that v1.1 was")
# print("reporting as ~3200 ms because it timed the embedding API call alongside it.")
# print("On six documents most configs will tie; the spread appears at real corpus size, and the")
# print("exact-ID and multi-entity questions are where hybrid and reranking already separate.")

## D.3 — Answer-quality evaluation (LLM-as-judge)

Retrieval metrics say the right passage was *found*. This says the answer was actually
*right, grounded and complete*. Strict on purpose.

In [ ]:
# ── OPTIONAL: LLM-as-judge scoring ──
# JUDGE_PROMPT = """You are evaluating a retrieval-augmented answer. Be strict.
#
# QUESTION: {question}
# EXPECTED FACT(S) a correct answer must contain: {expected}
#
# SOURCES THAT WERE RETRIEVED:
# {sources}
#
# ANSWER GIVEN:
# {answer}
#
# Score 0-5 on each axis:
# - groundedness: every claim is supported by the sources (5 = fully grounded, 0 = fabricated)
# - correctness: the answer contains the expected fact(s), stated accurately
# - completeness: if several entities or values legitimately match, all are covered and attributed
#
# Return ONLY JSON: {{"groundedness": int, "correctness": int, "completeness": int, "why": "one sentence"}}"""
#
# def judge_answers(mode: str = "hybrid", use_reranker: bool = True):
#     scored, refusals_ok, refusals_total = [], 0, 0
#
#     for g in GOLD:
#         res = ask(g["q"], mode=mode, use_reranker=use_reranker)
#
#         if g.get("unanswerable"):
#             refusals_total += 1
#             ok = res["refused"]
#             refusals_ok += ok
#             print(f"{'PASS' if ok else 'FAIL'}  (refusal) {g['q'][:60]}")
#             continue
#
#         src = format_sources(res["sources"]) or "(none)"
#         try:
#             jr = _with_retry(lambda: genai_client.models.generate_content(
#                 model=CHAT_MODEL,
#                 contents=JUDGE_PROMPT.format(question=g["q"],
#                                              expected=", ".join(g["must_contain"]),
#                                              sources=src, answer=res["answer"]),
#                 config=GenerateContentConfig(temperature=0,
#                                              response_mime_type="application/json")))
#             verdict = json.loads(jr.text)
#         except Exception as e:
#             print(f"  judge failed on {g['q'][:40]}: {type(e).__name__}: {e}")
#             continue
#
#         # Deterministic backstop: does the expected literal actually appear in the answer?
#         verdict["exact"] = all(s.lower() in res["answer"].lower() for s in g["must_contain"])
#         scored.append(verdict)
#         print(f"g={verdict['groundedness']} c={verdict['correctness']} "
#               f"comp={verdict['completeness']} exact={verdict['exact']}  {g['q'][:55]}")
#
#     if scored:
#         avg = lambda key: sum(s[key] for s in scored) / len(scored)
#         print(f"\nGroundedness {avg('groundedness'):.2f}/5 | "
#               f"Correctness {avg('correctness'):.2f}/5 | "
#               f"Completeness {avg('completeness'):.2f}/5")
#         print(f"Expected fact literally present: "
#               f"{sum(s['exact'] for s in scored)}/{len(scored)}")
#     if refusals_total:
#         print(f"Correct refusals: {refusals_ok}/{refusals_total}")
#
# judge_answers()

## D.4 — Configuration sweep

Reproduces the three chunkers verbatim and scores every (chunker × model × dimension)
combination on the same gold set, each in its own table. Slow and it costs real embedding
calls — but it is what turns *"we chose 350-token semantic chunks"* into a defensible claim.

Run the three cells in order.

In [ ]:
# ── OPTIONAL: the three candidate chunkers, kept for a fair comparison ──
# # =====================================================================================
# # v1.1's three chunkers, reproduced verbatim so the comparison is fair rather than rigged.
# # Keeping them here means you can show a client the exact methods that were considered.
# # =====================================================================================
#
# def chunk_fixed(text, size=400, overlap=60):
#     """v1.1's 'fixed_400': slide a fixed-width window over raw CHARACTERS.
#     Simple and predictable, but happily cuts mid-word and mid-number."""
#     text = text.strip()
#     step = max(1, size - overlap)          # how far the window advances each time
#     return [text[i:i + size] for i in range(0, len(text), step) if text[i:i + size].strip()]
#
# def chunk_sentence(text, size=500):
#     """v1.1's 'sentence': pack whole sentences up to a CHARACTER budget.
#     Respects sentence boundaries, but a character budget does not map cleanly to the
#     model's token limit, so chunk sizes vary unpredictably in token terms."""
#     sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", text.strip()) if s.strip()]
#     chunks, buf = [], ""
#     for s in sents:
#         if buf and len(buf) + len(s) > size:
#             chunks.append(buf); buf = s    # flush the full chunk, start a new one
#         else:
#             buf = f"{buf} {s}".strip()     # room left - keep appending
#     if buf:
#         chunks.append(buf)
#     return chunks
#
# def chunk_tokens(text, size=256, overlap=40):
#     """v1.1's 'token_256': slide a window over TOKENS.
#     Guarantees the model's token limit is respected, but cuts mid-sentence."""
#     toks = _enc.encode(text)
#     step = max(1, size - overlap)
#     return [_enc.decode(toks[i:i + size]) for i in range(0, len(toks), step)]
#
# # The candidate chunkers. Each is a function: text -> list of text pieces.
# CHUNKERS = {
#     "fixed_400":     lambda t: chunk_fixed(t, 400, 60),      # v1.1
#     "sentence_500":  lambda t: chunk_sentence(t, 500),       # v1.1
#     "token_256":     lambda t: chunk_tokens(t, 256, 40),     # v1.1
#     "recursive_350": lambda t: split_text(t, 350, 60),       # v2 default (Section B.2)
#     "recursive_600": lambda t: split_text(t, 600, 100),      # v2, larger chunks
# }
#
# # The configurations to compare. Trim this list to control cost - every entry re-embeds
# # the whole corpus once.
# #   model / dim -> which embedder, at what vector size
# #   chunker     -> a key from CHUNKERS above
# #   header      -> True = embed the contextual header (Section B.2); False = the ablation
# SWEEP_CONFIGS = [
#     # --- vary the CHUNKER, holding the model fixed ("why this chunking method?") ---
#     {"name": "gem001-1536 / fixed_400",     "model": "gemini-embedding-001", "dim": 1536, "chunker": "fixed_400",     "header": True},
#     {"name": "gem001-1536 / sentence_500",  "model": "gemini-embedding-001", "dim": 1536, "chunker": "sentence_500",  "header": True},
#     {"name": "gem001-1536 / token_256",     "model": "gemini-embedding-001", "dim": 1536, "chunker": "token_256",     "header": True},
#     {"name": "gem001-1536 / recursive_350", "model": "gemini-embedding-001", "dim": 1536, "chunker": "recursive_350", "header": True},
#     {"name": "gem001-1536 / recursive_600", "model": "gemini-embedding-001", "dim": 1536, "chunker": "recursive_600", "header": True},
#
#     # --- vary the MODEL, holding the chunker fixed ("why not 004 / 005?") ---
#     {"name": "text-emb-004 / recursive_350", "model": "text-embedding-004",  "dim": 768,  "chunker": "recursive_350", "header": True},
#     {"name": "text-emb-005 / recursive_350", "model": "text-embedding-005",  "dim": 768,  "chunker": "recursive_350", "header": True},
#
#     # --- vary the DIMENSION ("can we halve storage?") ---
#     {"name": "gem001-3072 / recursive_350", "model": "gemini-embedding-001", "dim": 3072, "chunker": "recursive_350", "header": True},
#     {"name": "gem001-768  / recursive_350", "model": "gemini-embedding-001", "dim": 768,  "chunker": "recursive_350", "header": True},
#
#     # --- ABLATION: identical to the v2 default, header removed ("does the header help?") ---
#     {"name": "gem001-1536 / recursive_350 / NO HEADER", "model": "gemini-embedding-001", "dim": 1536,
#      "chunker": "recursive_350", "header": False},
# ]
#
# print(f"{len(SWEEP_CONFIGS)} configurations defined, {len(CHUNKERS)} chunkers available.")
# print("Each config re-embeds the whole corpus once - trim SWEEP_CONFIGS to control cost.")

In [ ]:
# ── OPTIONAL: sweep machinery ──
# # =====================================================================================
# # The sweep machinery. Each config gets its own table because vectors from different
# # models/dimensions are not comparable and cannot share a column.
# # =====================================================================================
#
# _sweep_qcache = {}   # (model, dim, question) -> vector. Avoids re-embedding the same
#                      # question once per config, which would multiply the API bill.
#
# def _sweep_embed_query(cfg, question):
#     key = (cfg["model"], cfg["dim"], question)
#     if key not in _sweep_qcache:
#         _sweep_qcache[key] = embed_texts([question], "RETRIEVAL_QUERY",
#                                          model=cfg["model"], dim=cfg["dim"])[0]
#     return _sweep_qcache[key]
#
# def _sweep_table_name(cfg):
#     """Turn a config name into a legal, unique Postgres table name."""
#     slug = re.sub(r"[^a-z0-9]+", "_", cfg["name"].lower()).strip("_")
#     return f"rag_sweep_{slug}"[:60]        # Postgres identifiers max out at 63 characters
#
# def sweep_ingest(cfg):
#     """Chunk + embed the whole corpus under ONE configuration, into its own table."""
#     table = _sweep_table_name(cfg)
#     with db() as cur:
#         cur.execute(f"DROP TABLE IF EXISTS {table}")     # safe: these are throwaway tables
#         cur.execute(f"""
#             CREATE TABLE {table} (
#                 chunk_id  bigserial PRIMARY KEY,
#                 doc_id    text NOT NULL,
#                 content   text NOT NULL,
#                 embedding vector({cfg['dim']}) NOT NULL
#             )
#         """)
#
#     with db(dict_rows=True) as cur:
#         cur.execute(f"SELECT doc_id, title, content, metadata FROM {DOC_TABLE} ORDER BY doc_id")
#         docs = cur.fetchall()
#
#     chunk_fn = CHUNKERS[cfg["chunker"]]
#     rows = []                       # (doc_id, clean_text, text_we_embed)
#     for d in docs:
#         header = contextual_header(d["doc_id"], d["title"], d["metadata"]) if cfg["header"] else ""
#         for piece in chunk_fn(d["content"]):
#             rows.append((d["doc_id"], piece, f"{header}\n\n{piece}" if header else piece))
#
#     vectors = embed_texts([r[2] for r in rows], "RETRIEVAL_DOCUMENT",
#                           model=cfg["model"], dim=cfg["dim"])
#
#     with db() as cur:
#         execute_values(cur,
#                        f"INSERT INTO {table} (doc_id, content, embedding) VALUES %s",
#                        [(r[0], r[1], str(v)) for r, v in zip(rows, vectors)],
#                        template="(%s, %s, %s::vector)", page_size=200)
#         # HNSW cannot index beyond 2000 dims - the 3072 config falls back to an exact scan.
#         # That is fine for a sweep (we are measuring retrieval QUALITY, not speed) but it is
#         # exactly the constraint that makes 3072 impractical to actually deploy.
#         if cfg["dim"] <= 2000:
#             cur.execute(f"CREATE INDEX ON {table} USING hnsw (embedding vector_cosine_ops)")
#         cur.execute(f"ANALYZE {table}")
#     return table, len(rows)
#
# def sweep_eval(cfg, table, k=RERANK_TOP_N):
#     """Score one configuration on the GOLD questions using dense retrieval only.
#
#     Vector-only on purpose: the keyword leg is identical for every config (it does not use
#     embeddings), so including it would dilute exactly the difference we are trying to measure.
#     """
#     answerable = [g for g in GOLD if not g.get("unanswerable")]
#     recall = mrr = support = 0.0
#     for g in answerable:
#         qv = _sweep_embed_query(cfg, g["q"])
#         with db(dict_rows=True) as cur:
#             cur.execute(f"""
#                 SELECT doc_id, content, embedding <=> %s::vector AS distance
#                 FROM {table} ORDER BY distance LIMIT %s
#             """, (str(list(qv)), k))
#             hits = [dict(r) for r in cur.fetchall()]
#
#         ranked = [h["doc_id"] for h in hits]
#         gold_docs = set(g["doc_ids"])
#         if gold_docs & set(ranked):
#             recall += 1
#             mrr += 1.0 / (next(i for i, d in enumerate(ranked) if d in gold_docs) + 1)
#         blob = " ".join(h["content"] for h in hits).lower()
#         if all(s.lower() in blob for s in g["must_contain"]):
#             support += 1            # was the answer TEXT actually retrieved, not just the doc?
#
#     n = len(answerable)
#     return {"recall": recall / n, "mrr": mrr / n, "support": support / n, "n": n}
#
# def wilson_ci(p, n, z=1.96):
#     """95% confidence interval for a proportion (Wilson score interval).
#
#     Plain-English version: 'the true score is somewhere in this range'. Wilson is used
#     instead of the textbook formula because it stays sensible for small n and for scores
#     near 0 or 1 - exactly our situation. Returns (low, high).
#     """
#     if n == 0:
#         return (0.0, 0.0)
#     denom = 1 + z**2 / n
#     centre = (p + z**2 / (2 * n)) / denom
#     margin = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
#     return (max(0.0, centre - margin), min(1.0, centre + margin))

In [ ]:
# ── OPTIONAL: run the sweep (set CONFIGS_TO_RUN = SWEEP_CONFIGS[:4] for a cheap trial) ──
# # =====================================================================================
# # Run the sweep. Set CONFIGS_TO_RUN = SWEEP_CONFIGS[:4] first if you want a cheap trial.
# # =====================================================================================
# CONFIGS_TO_RUN = SWEEP_CONFIGS
#
# sweep_results, sweep_tables = [], []
# for cfg in CONFIGS_TO_RUN:
#     try:
#         t0 = time.perf_counter()
#         table, n_chunks = sweep_ingest(cfg)
#         ingest_s = time.perf_counter() - t0
#         sweep_tables.append(table)
#
#         scores = sweep_eval(cfg, table)
#         # pgvector stores a vector as 4 bytes per dimension plus ~8 bytes of row overhead.
#         mb = n_chunks * (4 * cfg["dim"] + 8) / 1024 / 1024
#         sweep_results.append({**cfg, **scores, "chunks": n_chunks,
#                               "ingest_s": ingest_s, "vector_mb": mb, "ok": True})
#         print(f"  ok   {cfg['name']:<42} {n_chunks:>4} chunks  {ingest_s:>5.1f}s")
#     except Exception as e:
#         # A config can legitimately fail - a model not enabled in your project, a dimension
#         # the model does not support. Record it and keep going rather than losing the run.
#         sweep_results.append({**cfg, "ok": False, "error": f"{type(e).__name__}: {e}"})
#         print(f"  FAIL {cfg['name']:<42} {str(e)[:60]}")
#
# # ---------------------------------- report ----------------------------------
# ok = [r for r in sweep_results if r["ok"]]
# ok.sort(key=lambda r: (r["support"], r["mrr"], r["recall"]), reverse=True)
#
# hdr = (f"{'configuration':<44}{'recall':>8}{'mrr':>7}{'support':>9}"
#        f"{'95% CI (support)':>20}{'chunks':>8}{'vec MB':>9}")
# print("\n" + hdr)
# print("-" * len(hdr))
# for r in ok:
#     lo, hi = wilson_ci(r["support"], r["n"])
#     print(f"{r['name']:<44}{r['recall']:>8.2f}{r['mrr']:>7.2f}{r['support']:>9.2f}"
#           f"{f'[{lo:.2f}, {hi:.2f}]':>20}{r['chunks']:>8}{r['vector_mb']:>9.2f}")
#
# for r in sweep_results:
#     if not r["ok"]:
#         print(f"  skipped: {r['name']} -> {r['error'][:90]}")
#
# # ------------------------- the honesty check on your own result -------------------------
# if ok:
#     best = ok[0]
#     b_lo, b_hi = wilson_ci(best["support"], best["n"])
#     # Anything whose interval overlaps the winner's cannot be called worse on this evidence.
#     tied = [r for r in ok[1:] if wilson_ci(r["support"], r["n"])[1] >= b_lo]
#     print(f"\nHighest score: {best['name']} (answer-support {best['support']:.2f})")
#     print(f"Statistically indistinguishable from it on {best['n']} questions: {len(tied)} config(s)")
#     if tied:
#         for r in tied[:5]:
#             print(f"    - {r['name']}")
#         print(f"\n  With only {best['n']} questions the confidence interval is ~"
#               f"±{(b_hi - b_lo) / 2 * 100:.0f} points, so this sweep CANNOT separate those.")
#         print("  Report it as directional. To actually rank them, grow GOLD to 50-100 questions.")
#     else:
#         print("  This config beats every other tested config beyond the confidence interval.")
#
# # ------------------------------- clean up the temp tables -------------------------------
# # These really are disposable - unlike v1.1's sweep, which dropped the winning table.
# # Comment this out if you want to inspect a sweep table by hand afterwards.
# with db() as cur:
#     for t in sweep_tables:
#         cur.execute(f"DROP TABLE IF EXISTS {t}")
# print(f"\nDropped {len(sweep_tables)} temporary sweep table(s). "
#       f"Production tables ({DOC_TABLE}, {CHUNK_TABLE}) untouched.")

---
# Section E — Troubleshooting & production notes

### Routing
| Symptom | Fix |
|---|---|
| Everything routes to SQL | nothing is ingested yet — check `SELECT count(*) FROM rag_documents WHERE indexed_hash IS NOT NULL` |
| Everything routes to RAG | `KNOWN_TABLES` is empty — re-run A.1 |
| A question keeps going to the wrong engine | force it with `mode=`, then add the distinction to `ROUTER_PROMPT` |
| Router feels slow | it is one `gemini-2.5-flash` call; drop it by calling `ask_sql` / `ask_rag` directly |

### SQL engine
| Symptom | Fix |
|---|---|
| `could not connect to server` | host/port/credentials in 0.2, or Postgres not reachable |
| `403 PERMISSION_DENIED` from Vertex | authenticate, and enable the Vertex AI API on the project |
| `404 model not found` | change `SQL_MODEL` / `LOCATION` |
| Uses the wrong table, ignores an `_old` table | write a note in **A.3**, then re-run A.3 → A.7 |
| Gate refuses a valid question | rephrase with your real table names, or relax the gate prompt in A.4 |
| Correct SQL, 0 rows | table is empty or the filter matched nothing — see A.1's report |

### RAG engine
| Symptom | Fix |
|---|---|
| `type "vector" does not exist` | pgvector not installed on the server: `CREATE EXTENSION vector` needs superuser once |
| Always refuses | nothing ingested (B.4), or `MAX_COSINE_DISTANCE` too tight |
| Retrieves the right doc, wrong chunk | raise `CHUNK_OVERLAP_TOKENS`, or add the distinguishing field to `HEADER_FIELDS` |
| Exact IDs / SKUs not found | raise `W_TEXT` — that is the keyword leg's job |
| Answers are vague | lower `RERANK_TOP_N`; irrelevant passages in the prompt make answers worse |
| Changed `EMBED_MODEL` or `EMBED_DIM` | you **must** run `ingest(force=True)` — old vectors are meaningless now |

### Before this goes anywhere near production
- **Credentials** — move the password out of 0.2 into `os.environ["PGPASSWORD"]` or Secret Manager, and rotate the current one.
- **Least privilege** — give the app a Postgres role with `SELECT` only. The validators are belt; this is braces.
- **Logging** — record every (question, route, SQL/sources, outcome) tuple. The good ones become few-shot examples in A.5 and the router prompt in C.
- **Re-ingest on change** — wire `ingest()` to whatever updates your documents; the content hash makes it cheap to run often.
- **Serving** — when you outgrow the notebook, `ask()` is already the whole API surface. Wrap it in FastAPI/Streamlit and keep everything below it unchanged.

### Teardown (destroys the vector index only — never touches your business tables)
```python
# with db() as cur:
#     cur.execute(f"DROP TABLE IF EXISTS {CHUNK_TABLE}, {DOC_TABLE} CASCADE")
```